# CUBIC MARS — PS3 Root-Cause & Severity Intelligence

## SageMaker edition — leakage-safe, temporally validated, and device-specific

This notebook produces an honest assessment for the three operating populations:

| Population | What this notebook delivers | Honest limitation |
|---|---|---|
| **TVM** | Temporal severity and derived-component root-cause models, action ranking, driver analysis, drift, and cross-validation | The root-cause target is the current *derived component* label, not yet a confirmed 9-class ServiceNow RCA label. |
| **Gate** | The same model/evaluation framework, with small-sample safeguards and a report-only fallback | A model is not promoted when there are too few events or class support is weak. |
| **Validator** | A dedicated OOS-proxy discovery pack: severity mix, repeat-failure rate, device priorities, trends, and hotspots | No PS3 classifier is trained when the availability-event incident feed contains zero Validator rows. |

### What is materially stronger than the prior notebook

- Separates **train / validation / final test** chronologically, with embargo windows. The test set is never used to select a model.
- Blocks incident-time ServiceNow fields and raw fault text from pre-incident modelling. A feature time-contract and single-feature audit are saved for every head.
- Replaces full-history facility frequency with **strictly causal**, prior-day device and facility recurrence features.
- Uses a **validation-selected bake-off** (baseline, Logistic Regression, Random Forest, HistGradientBoosting, and optional XGBoost / LightGBM / CatBoost); all test results are diagnostic only after selection is frozen.
- Writes train, validation, and test accuracy/F1/balanced-accuracy/AUC/PR-AUC/calibration metrics, plus overfit and promotion gates.
- Compares a purged walk-forward CV with an unseen-device group stress test. These are complementary checks, not substitutes for the final temporal test.
- Adds discoveries: severity × component lift, Pareto concentration, hotspots, recurrence, precursor profiles, prediction errors, confidence, anomaly indicators, and a Validator OOS proxy pack.

### Important interpretation rule

`failure_level_label` and `derived_component_type` describe incidents already observed. The model may only use data available before `AE_START_DTM`. It is therefore an **early triage / precursor classifier**, not a claim that it predicts a future fault from post-incident ticket fields. Leave `ALLOW_INCIDENT_TIME_METADATA=False` in production.


## 0. SageMaker setup

Run this cell once in the SageMaker kernel, then restart the kernel if `%pip` requests it. The optional libraries are guarded in code: the baseline scikit-learn path still runs if an optional estimator is unavailable.

```bash
%pip install -q "numpy<2" pandas scikit-learn matplotlib pyarrow s3fs joblib boto3 xgboost lightgbm catboost shap optuna
```

Before a real run, export the Gold and optional Silver Delta tables to the S3 parquet locations in the configuration. SageMaker reads S3 parquet directly; it does not require Spark.


## 1. Configuration — edit only this section for a normal run


In [ ]:
import os
from pathlib import Path

CONFIG = {
    # ------------------------------------------------------------------ run identity
    "CITY_ID": "CHI",
    "DEVICE_SCOPE": ["TVM", "GATE", "VALIDATOR"],
    "OUTPUT_ROOT": "PS3_outputs_pavan_enhanced",  # root/{tvm,gates,validators,cross_device}
    "RANDOM_STATE": 42,

    # ------------------------------------------------------------------ Gold source
    # SageMaker normally uses parquet. "auto" uses Spark only when a Spark session is already active.
    "DATA_SOURCE": "auto",  # auto | parquet | databricks_spark
    "GOLD_TABLE": "mars_dev.gold.device_ps3_incident",
    "GOLD_PARQUET": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/gold/device_ps3_incident",
    "S3_REGION": "us-east-1",

    # ------------------------------------------------------------------ optional Silver enrichments
    "SILVER_PREFIX": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver",
    "ENRICH_SOURCES": True,
    "ENRICH_USAGE": True,
    "ENRICH_STATION": True,
    "ENRICH_METRIC": True,
    "ENRICH_UPTIME": True,
    "ENRICH_ASOF_TOLERANCE_DAYS": 10,
    "USAGE_LIFECYCLE_TABLE": "mars_dev.silver.usage_lifecycle_daily",
    "STATION_NETWORK_TABLE": "mars_dev.silver.station_network_daily",
    "METRIC_DAILY_TABLE": "mars_dev.silver.metric_daily",
    "UPTIME_INTERVALS_TABLE": "mars_dev.silver.device_uptime_intervals",
    "VALIDATOR_PROXY_TABLE": "mars_dev.silver.device_failures",
    "INCIDENT_ROOT_CAUSE_TABLE": "mars_dev.silver.incident_root_cause",

    # ------------------------------------------------------------------ targets and honest scope
    "SEVERITY_TARGET": "failure_level_label",
    "ROOTCAUSE_TARGET": "derived_component_type",
    "SEVERITY_COLLAPSE": {
        "PURCHASE_CARD": "MAJOR",
        "PURCHASE_PRODUCT": "MAJOR",
        "NONPAYMENT": "MAJOR",
        "ALL_PURCHASE": "CRITICAL",
        "ALL_FUNCTIONS": "CRITICAL",
        "BUS_READER": "CRITICAL",
        "BUS_READER_ASSEMBLY": "CRITICAL",
    },

    # ------------------------------------------------------------------ feature-time contract
    # Keep False for a real pre-incident / early-triage model. Ticket metadata may only be
    # enabled deliberately for a separately labelled post-incident diagnostic experiment.
    "ALLOW_INCIDENT_TIME_METADATA": False,
    "SAFE_CATEGORICAL_CANDIDATES": [
        "device_model", "device_model_category", "manufacturer", "firmware_major",
        "component_family", "facility_type",
    ],
    "DROP_SUSPICIOUS_FEATURES": False,
    "SUSPICIOUS_SINGLE_FEATURE_F1": 0.95,

    # ------------------------------------------------------------------ validation design
    "MIN_TRAIN_DATE": "2024-01-01",
    "TRAIN_FRACTION": 0.60,
    "VALIDATION_FRACTION": 0.20,
    "TEST_FRACTION": 0.20,
    "EMBARGO_DAYS": 2,
    "MIN_ROWS_TO_MODEL": {"TVM": 250, "GATE": 200, "VALIDATOR": 250},
    "MIN_TRAIN_CLASS_ROWS": 30,
    "MIN_EVAL_ROWS": 20,
    "WALK_FORWARD_SPLITS": 4,
    "GROUP_STRESS_SPLITS": 3,
    "GROUP_STRESS_TEST_FRACTION": 0.20,

    # ------------------------------------------------------------------ model and tuning controls
    "RF_TREES": 400,
    "RUN_XGBOOST": True,
    "RUN_LIGHTGBM": True,
    "RUN_CATBOOST": True,
    "RUN_OPTUNA": True,
    "N_OPTUNA_TRIALS": 20,
    "TUNE_CV_SPLITS": 3,
    "TUNE_MAX_ROWS": 60000,

    # ------------------------------------------------------------------ promotion guardrails
    "SEVERITY_F1_FLOOR": 0.55,
    "ROOTCAUSE_F1_FLOOR": 0.45,
    "MIN_VALIDATION_DELTA_VS_BASELINE": 0.02,
    "MAX_TRAIN_VALIDATION_F1_GAP": 0.20,
    "MAX_TEST_VALIDATION_F1_DROP": 0.15,
    "MAX_UNSEEN_TEST_LABEL_RATE": 0.05,

    # --- 2026-07-26: severity target + overfitting/calibration controls -------
    # Train severity on the binary MAJOR/CRITICAL collapse the dashboard actually
    # consumes, instead of the 4-5 class failure_level_label. Two balanced classes
    # raise macro-F1 structurally and remove the collapse-mapping bug class.
    "SEVERITY_BINARY_TARGET": True,
    # Regularisation. The 26-Jul run failed 3 of 4 heads on the train-validation
    # gap guardrail (0.330 / 0.453 / 0.449) with RandomForest at train F1 = 1.000.
    "REG_RF_MAX_DEPTH": 12,
    "REG_RF_MIN_LEAF": 5,
    "REG_HGB_MAX_ITER": 250,
    "REG_HGB_LEAVES": 15,
    "REG_HGB_MIN_LEAF": 20,
    "REG_HGB_L2": 1.0,
    "REG_XGB_MAX_DEPTH": 4,
    "REG_XGB_MIN_CHILD_WEIGHT": 5,
    "REG_XGB_LAMBDA": 2.0,
    "REG_LGB_LEAVES": 15,
    "REG_LGB_MIN_CHILD": 30,
    "REG_LGB_LAMBDA": 2.0,
    "REG_CAT_DEPTH": 4,
    "REG_CAT_L2": 6.0,
    # Probability calibration -- GATE/root_cause failed only on gap + test ECE
    # 0.192 vs the 0.15 guardrail, so this is the head closest to promotion.
    "CALIBRATE_PROBABILITIES": True,
    "CALIBRATION_METHOD": "isotonic",   # "isotonic" | "sigmoid"
    "CALIBRATION_FOLDS": 3,
    "MAX_TEST_ECE": 0.15,

    # ------------------------------------------------------------------ safe local dry run
    # Set True before first use to prove the notebook and outputs work without S3.
    "SMOKE_TEST_SYNTHETIC": False,

    # ------------------------------------------------------------------ RDS dashboard publishing (disabled by default)
    "RDS_DASHBOARD_BUNDLE_ENABLED": True,
    "RDS_SCHEMA": "mars_dashboard",
    "RDS_TABLE_PREFIX": "ps3",
    "RDS_RUN_ID": None,  # optional immutable override; otherwise generated from the run timestamp
    "RDS_REQUIRED_QUALITY_SCORE": 97.0,
    "RDS_BLOCK_ON_INPUT_QUALITY": True,
    "RDS_WRITE_ENABLED": False,  # opt in only after DDL/readiness review
    "RDS_CONNECTION_ENV": "PS3_RDS_DATABASE_URL",
    "RDS_WRITE_CHUNK_SIZE": 5000,

    # ------------------------------------------------------------------ reporting granularity and reliability discovery
    "OUTPUT_DEVICE_AND_SERIAL_LEVEL": True,
    "UNCLASSIFIED_ROOT_CAUSE_LABEL": "UNCLASSIFIED_NO_ROOT_CAUSE_PREDICTION",
    "COMPONENT_PRIOR_STRENGTH": 20,
    "COMPONENT_RECURRENCE_WINDOW_DAYS": 30,
    "COMPONENT_RECENT_WINDOW_DAYS": 90,
    "COMPONENT_MIN_EVIDENCE_EVENTS": 5,
    "COMPONENT_PRIORITY_TOP_N": 20,
    "ANOMALY_MIN_REFERENCE_ROWS": 20,
    "COMPONENT_PRIORITY_WEIGHTS": {
        "incident_volume": 0.25,
        "criticality": 0.35,
        "recurrence": 0.20,
        "recent_activity": 0.10,
        "recency": 0.10,
    },
}

assert abs(CONFIG["TRAIN_FRACTION"] + CONFIG["VALIDATION_FRACTION"] + CONFIG["TEST_FRACTION"] - 1.0) < 1e-9
assert abs(sum(CONFIG["COMPONENT_PRIORITY_WEIGHTS"].values()) - 1.0) < 1e-9
RS = CONFIG["RANDOM_STATE"]


## 2. Imports, optional packages, run logging, and output helpers


In [ ]:
import json
import math
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

# SageMaker and hosted kernels can have a read-only home directory. Set this before importing matplotlib.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, cohen_kappa_score, confusion_matrix,
    f1_score, log_loss, matthews_corrcoef, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight


try:
    from sklearn.calibration import CalibratedClassifierCV
except Exception:  # pragma: no cover
    CalibratedClassifierCV = None


def optional_import(name):
    try:
        module = __import__(name)
        print(f"[optional] {name} {getattr(module, '__version__', 'available')}")
        return module
    except Exception as exc:
        print(f"[optional] {name} unavailable ({type(exc).__name__}); its path will be skipped")
        return None


xgb = optional_import("xgboost")
lgb = optional_import("lightgbm")
catb = optional_import("catboost")
optuna = optional_import("optuna")
shap = optional_import("shap")

PASTEL = ["#377eb8", "#4daf4a", "#ff7f00", "#984ea3", "#e41a1c", "#a65628", "#f781bf", "#999999"]
ENRICHMENT_AUDIT = []
SYNTHETIC_SOURCES = {}


def make_output_dir():
    root = Path(CONFIG["OUTPUT_ROOT"]).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    return root


def write_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, default=str)


def write_csv(frame, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=index)


def safe_name(value):
    return "".join(ch if ch.isalnum() or ch in "_-" else "_" for ch in str(value))[:80]


def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")


def plot_or_close(path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.close()


## 3. Data loading, canonical schema handling, and data-quality contract


In [ ]:
COLUMN_ALIASES = {
    "availability_event_id": ["availability_event_id", "ae_id", "event_id", "availability_id"],
    "device_id": ["device_id", "deviceid", "ncs_device_id", "u_ncs_device_id"],
    "mars_device_category": ["mars_device_category", "device_category", "device_type", "device_type_name"],
    "AE_START_DTM": ["ae_start_dtm", "availability_start_dtm", "event_start_dtm", "start_dtm", "opened_at"],
    "transit_day": ["transit_day", "service_day", "event_day", "ledger_date"],
    "failure_level_label": ["failure_level_label", "failure_level_name", "severity_label"],
    "derived_component_type": ["derived_component_type", "root_cause_component", "component_type"],
    "matched_serial_nbr": ["matched_serial_nbr", "serial_number", "device_serial_number", "serial_nbr"],
    "FACILITY_ID": ["facility_id", "station_id", "stop_point_id"],
    "FACILITY_NAME": ["facility_name", "station_name", "stop_point_name"],
}


def _find_column(columns, candidates):
    lookup = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        hit = lookup.get(str(candidate).lower())
        if hit is not None:
            return hit
    return None


def canonicalise_incident_columns(frame):
    """Rename known aliases only; unknown source columns are deliberately never auto-features."""
    df = frame.copy()
    rename = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        if canonical in df.columns:
            continue
        source = _find_column(df.columns, [canonical] + aliases)
        if source is not None and source not in rename:
            rename[source] = canonical
    if rename:
        log(f"schema aliases applied: {rename}")
        df = df.rename(columns=rename)
    return df, rename


def normalise_category(value):
    text = str(value).strip().upper()
    if "TVM" in text or "VENDING" in text:
        return "TVM"
    if "GATE" in text or "TURNSTILE" in text:
        return "GATE"
    if any(token in text for token in ("VALIDATOR", "VALID", "BMV", "READER")):
        return "VALIDATOR"
    return text


def active_spark():
    spark = globals().get("spark")
    if spark is not None:
        return spark
    try:
        from pyspark.sql import SparkSession
        return SparkSession.getActiveSession()
    except Exception:
        return None


def load_table(table_name, parquet_path, required=True, synthetic_key=None):
    """Spark when explicitly available; otherwise direct S3/Parquet read for SageMaker."""
    if synthetic_key and synthetic_key in SYNTHETIC_SOURCES:
        return SYNTHETIC_SOURCES[synthetic_key].copy()
    source = CONFIG["DATA_SOURCE"]
    spark_session = active_spark()
    errors = []
    if source in ("auto", "databricks_spark") and spark_session is not None:
        try:
            log(f"loading Spark table: {table_name}")
            return spark_session.table(table_name).toPandas()
        except Exception as exc:
            errors.append(f"Spark: {type(exc).__name__}: {str(exc)[:180]}")
            if source == "databricks_spark":
                raise RuntimeError(errors[-1]) from exc
    if source in ("auto", "parquet") and parquet_path:
        try:
            storage_options = {}
            if str(parquet_path).startswith("s3://"):
                storage_options = {"client_kwargs": {"region_name": CONFIG["S3_REGION"]}}
            log(f"loading parquet: {parquet_path}")
            return pd.read_parquet(parquet_path, storage_options=storage_options or None)
        except Exception as exc:
            errors.append(f"Parquet: {type(exc).__name__}: {str(exc)[:180]}")
    message = " | ".join(errors) or "No compatible loader was available"
    if required:
        raise RuntimeError(
            f"Could not load {table_name}. {message}. "
            "For SageMaker, verify the S3 export, the execution role, and pyarrow+s3fs."
        )
    log(f"optional source skipped: {table_name} — {message}")
    return None


def coerce_decimal_objects(df):
    from decimal import Decimal
    out = df.copy()
    for column in out.columns:
        if out[column].dtype == object:
            values = out[column].dropna()
            if len(values) and isinstance(values.iloc[0], Decimal):
                out[column] = pd.to_numeric(out[column], errors="coerce")
    return out


def severity_bucket(series):
    """Map known severity codes without assuming unknown values are MAJOR."""
    mapping = {str(k).upper(): v for k, v in CONFIG["SEVERITY_COLLAPSE"].items()}
    critical_numbers = {"4", "5", "16"}

    def one(value):
        text = str(value).strip().upper()
        if text in mapping:
            return mapping[text]
        if text in critical_numbers or "CRIT" in text or "ALL_FUNCTION" in text:
            return "CRITICAL"
        if text in {"", "NAN", "NONE", "NULL", "UNKNOWN"}:
            return "UNKNOWN"
        # 2026-07-26 FIX: an unmapped code must NOT default to MAJOR. That default
        # is the same failure mode that drove pct_critical_pred to 0.0 fleet-wide
        # in the 19-Jul run -- it silently reclassifies unknown severities as
        # non-critical. Surface them instead.
        return "UNKNOWN"

    return series.map(one)


def validate_and_prepare_incidents(frame):
    df, renamed = canonicalise_incident_columns(coerce_decimal_objects(frame))
    report = {"input_rows": int(len(df)), "aliases_applied": renamed, "issues": []}
    required = ["device_id", "mars_device_category", "AE_START_DTM"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"Gold schema missing required columns: {missing}. Available: {list(df.columns)[:40]}")

    df["AE_START_DTM"] = pd.to_datetime(df["AE_START_DTM"], errors="coerce", utc=False)
    invalid_time = int(df["AE_START_DTM"].isna().sum())
    if invalid_time:
        report["issues"].append(f"Dropped {invalid_time:,} rows with invalid AE_START_DTM")
        df = df[df["AE_START_DTM"].notna()].copy()
    if "transit_day" not in df.columns:
        df["transit_day"] = df["AE_START_DTM"].dt.normalize()
        report["issues"].append("Derived transit_day from AE_START_DTM")
    else:
        df["transit_day"] = pd.to_datetime(df["transit_day"], errors="coerce").fillna(df["AE_START_DTM"].dt.normalize())

    df["mars_device_category"] = df["mars_device_category"].map(normalise_category)
    df["device_id"] = df["device_id"].astype("string").str.strip()
    missing_device = int(df["device_id"].isna().sum() + (df["device_id"] == "").sum())
    if missing_device:
        report["issues"].append(f"Dropped {missing_device:,} rows with missing device_id")
        df = df[df["device_id"].notna() & (df["device_id"] != "")].copy()

    if "availability_event_id" not in df.columns:
        df["availability_event_id"] = [f"generated-{i}" for i in range(len(df))]
        report["issues"].append("availability_event_id absent; generated stable row identifiers")
    df["availability_event_id"] = df["availability_event_id"].astype("string")

    duplicates = int(df["availability_event_id"].duplicated(keep=False).sum())
    report["duplicate_event_rows_before_dedup"] = duplicates
    if duplicates:
        check_columns = [c for c in (CONFIG["SEVERITY_TARGET"], CONFIG["ROOTCAUSE_TARGET"]) if c in df.columns]
        conflicts = 0
        if check_columns:
            conflicts = int(
                df.groupby("availability_event_id")[check_columns]
                .nunique(dropna=False)
                .max(axis=1)
                .gt(1)
                .sum()
            )
        report["duplicate_label_conflicts"] = conflicts
        report["issues"].append(
            f"Deduplicated {duplicates:,} duplicated availability-event rows; label-conflict IDs={conflicts:,}"
        )
        df = df.sort_values(["availability_event_id", "AE_START_DTM"], kind="mergesort").drop_duplicates(
            "availability_event_id", keep="first"
        )

    df = df[df["transit_day"].astype(str) >= CONFIG["MIN_TRAIN_DATE"]].copy()
    df = df.sort_values(["AE_START_DTM", "availability_event_id"], kind="mergesort").reset_index(drop=True)
    df["_row_id"] = np.arange(len(df), dtype=int)
    report.update(
        {
            "output_rows": int(len(df)),
            "n_devices": int(df["device_id"].nunique()),
            "categories": df["mars_device_category"].value_counts().to_dict(),
            "date_start": str(df["AE_START_DTM"].min()),
            "date_end": str(df["AE_START_DTM"].max()),
        }
    )
    report["data_quality_assessment"] = assess_incident_data_quality(df)
    if not report["data_quality_assessment"].get("quality_gate_pass", False):
        report["issues"].append(
            f"Prepared incident data-quality score {report['data_quality_assessment'].get('quality_score')} is below "
            f"the RDS readiness threshold {report['data_quality_assessment'].get('quality_gate_threshold')}"
        )
    return df, report


def probe_servicenow_conformance(output_root):
    path = f"{CONFIG['SILVER_PREFIX']}/incident_root_cause"
    result = {"available": False, "table": CONFIG["INCIDENT_ROOT_CAUSE_TABLE"]}
    try:
        source = load_table(CONFIG["INCIDENT_ROOT_CAUSE_TABLE"], path, required=False, synthetic_key="incident_root_cause")
        if source is None or source.empty:
            write_json(result, Path(output_root) / "servicenow_conformance_probe.json")
            return result
        result["available"] = True
        result["rows"] = int(len(source))
        for col in ("from_svn_stage", "from_cta_sn_mirror"):
            if col in source.columns:
                result[f"{col}_true_rate"] = float(pd.to_numeric(source[col], errors="coerce").fillna(0).mean())
        if "availability_event_id" in source.columns:
            result["duplicate_availability_event_id"] = int(source["availability_event_id"].duplicated().sum())
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {str(exc)[:240]}"
    write_json(result, Path(output_root) / "servicenow_conformance_probe.json")
    return result


## 4. Causal feature engineering and leakage-safe Silver as-of joins


In [ ]:
BASE_NUMERIC_FEATURES = [
    "events_24h_prior", "oos_onsets_24h", "bhu_events_24h", "chu_events_24h",
    "printer_events_24h", "gate_mech_events_24h", "csc_reader_events_24h",
    "comms_events_24h", "oos_onsets_7d_prior", "events_7d_prior",
    "component_age_days",
]
INCIDENT_TIME_METADATA = {
    "sn_event_code_id", "sn_event_code_name", "sn_priority", "sn_category",
    "sn_chargeable_level", "from_svn_stage", "from_cta_sn_mirror", "incident_duration_min",
}
HARD_LEAK_COLUMNS = {
    "AE_FAULT_DESCRIPTION", "AE_SYMPTOM", "AE_PROBLEM", "AE_RESOLUTION", "combined_text",
    "affected_component", "matched_component", "root_cause_category", "derived_component_type",
    "failure_level", "failure_level_label", "desc_printer_flag", "desc_card_reader_flag",
    "desc_bill_handler_flag", "desc_coin_flag", "desc_comms_flag", "desc_timeout_flag",
    "desc_replacement_flag", "AE_FAULT_STATE", "wot_state", "request_type", "is_chargeable",
    "is_device_fault", "kpi_rule_id", "kpi_category_name",
}


def add_temporal_features(df):
    out = df.copy()
    out["_ae_date"] = pd.to_datetime(out["AE_START_DTM"], errors="coerce").dt.normalize()
    out["ae_hour"] = pd.to_datetime(out["AE_START_DTM"], errors="coerce").dt.hour.fillna(12).astype(float)
    out["ae_dow"] = pd.to_datetime(out["AE_START_DTM"], errors="coerce").dt.dayofweek.fillna(0).astype(float)
    out["ae_month"] = pd.to_datetime(out["AE_START_DTM"], errors="coerce").dt.month.fillna(1).astype(float)
    return out


def _causal_history_table(df, entity_col, prefix):
    """Daily incident recurrence features using events strictly before the current day."""
    if entity_col not in df.columns:
        return None
    daily = (
        df[[entity_col, "_ae_date"]]
        .dropna()
        .groupby([entity_col, "_ae_date"], dropna=False)
        .size()
        .rename("events_today")
        .reset_index()
        .sort_values([entity_col, "_ae_date"], kind="mergesort")
    )
    outputs = []
    for _, group in daily.groupby(entity_col, sort=False, dropna=False):
        g = group.copy().sort_values("_ae_date", kind="mergesort")
        dates = g["_ae_date"].to_numpy(dtype="datetime64[ns]")
        counts = g["events_today"].to_numpy(dtype=float)
        cumulative = np.r_[0.0, np.cumsum(counts)]
        idx = np.arange(len(g))
        start_30 = np.searchsorted(dates, dates - np.timedelta64(30, "D"), side="left")
        start_7 = np.searchsorted(dates, dates - np.timedelta64(7, "D"), side="left")
        g[f"{prefix}_prior_incidents"] = cumulative[idx]
        g[f"{prefix}_prior_incidents_30d"] = cumulative[idx] - cumulative[start_30]
        g[f"{prefix}_prior_incidents_7d"] = cumulative[idx] - cumulative[start_7]
        outputs.append(g.drop(columns="events_today"))
    return pd.concat(outputs, ignore_index=True) if outputs else None


def add_causal_recurrence_features(df):
    out = df.copy()
    for entity, prefix in (("device_id", "hist_device"), ("FACILITY_ID", "hist_facility")):
        hist = _causal_history_table(out, entity, prefix)
        if hist is not None:
            out = out.merge(hist, on=[entity, "_ae_date"], how="left", validate="m:1")
    return out


def _source_column(frame, candidates):
    return _find_column(frame.columns, candidates)


def strict_asof_join(left, right, left_key, right_key, right_date, value_columns, prefix, by_columns=None):
    """Attach a source row no newer than the prior day; preserve one output row per incident."""
    by_columns = by_columns or []
    required_left = ["_row_id", left_key, "_ae_date"] + by_columns
    if any(col not in left.columns for col in required_left):
        raise KeyError(f"Left join keys unavailable: {required_left}")
    required_right = [right_key, right_date] + value_columns + by_columns
    if any(col not in right.columns for col in required_right):
        raise KeyError(f"Right join keys unavailable: {required_right}")

    l = left[required_left].copy()
    l["_asof_key"] = pd.to_datetime(l["_ae_date"], errors="coerce")
    r = right[required_right].copy().rename(columns={right_key: left_key, right_date: "_source_date"})
    r["_asof_key"] = pd.to_datetime(r["_source_date"], errors="coerce") + pd.Timedelta(days=1)
    # merge_asof requires identical `by` dtypes. Normalise identifiers explicitly rather than
    # depending on pandas' nullable-string representation in the source parquet.
    for key in [left_key] + by_columns:
        l[key] = l[key].astype("string").astype(object)
        r[key] = r[key].astype("string").astype(object)
    l = l.dropna(subset=["_asof_key", left_key]).sort_values("_asof_key", kind="mergesort")
    r = r.dropna(subset=["_asof_key", left_key]).sort_values("_asof_key", kind="mergesort")

    right_unique = [left_key, "_asof_key"] + by_columns
    duplicate_source_rows = int(r.duplicated(right_unique, keep=False).sum())
    if duplicate_source_rows:
        # A deterministic guard is preferable to a hidden row multiplication. The audit makes this visible.
        r = r.drop_duplicates(right_unique, keep="last")
    rename_values = {column: f"{prefix}{column}" for column in value_columns}
    r = r.rename(columns=rename_values)
    merged = pd.merge_asof(
        l,
        r[[left_key, "_asof_key"] + by_columns + list(rename_values.values())],
        on="_asof_key",
        by=[left_key] + by_columns,
        direction="backward",
        tolerance=pd.Timedelta(days=CONFIG["ENRICH_ASOF_TOLERANCE_DAYS"]),
    )
    matched = merged[["_row_id"] + list(rename_values.values())]
    result = left.merge(matched, on="_row_id", how="left", validate="1:1")
    coverage = float(matched[list(rename_values.values())].notna().any(axis=1).mean()) if len(matched) else 0.0
    return result, {"matched_coverage": coverage, "duplicate_source_rows": duplicate_source_rows, "n_columns": len(value_columns)}


def enrich_source(df, name, enabled, table, parquet_path, left_key, source_key_candidates,
                  feature_candidates, prefix, by_mapping=None, synthetic_key=None):
    record = {"source": name, "enabled": bool(enabled), "status": "skipped"}
    if not CONFIG["ENRICH_SOURCES"] or not enabled:
        ENRICHMENT_AUDIT.append(record)
        return df
    source = load_table(table, parquet_path, required=False, synthetic_key=synthetic_key)
    if source is None or source.empty:
        record["reason"] = "source unavailable or empty"
        ENRICHMENT_AUDIT.append(record)
        return df
    src = coerce_decimal_objects(source)
    source_key = _source_column(src, source_key_candidates)
    source_date = _source_column(src, ["transit_day", "ledger_date", "event_date", "date"])
    if source_key is None or source_date is None:
        record["reason"] = f"missing source key/date; found columns={list(src.columns)[:30]}"
        ENRICHMENT_AUDIT.append(record)
        return df
    src = src.rename(columns={source_key: left_key, source_date: "_source_day"})
    by_columns = []
    if by_mapping:
        for left_col, source_candidates in by_mapping.items():
            source_col = _source_column(src, source_candidates)
            if source_col is None or left_col not in df.columns:
                record["reason"] = f"missing grouping key for {left_col}"
                ENRICHMENT_AUDIT.append(record)
                return df
            if source_col != left_col:
                src = src.rename(columns={source_col: left_col})
            by_columns.append(left_col)
    present = [c for c in feature_candidates if c in src.columns]
    if not present:
        record["reason"] = "none of the expected enrichment features are present"
        ENRICHMENT_AUDIT.append(record)
        return df
    try:
        result, stats = strict_asof_join(
            df, src, left_key, left_key, "_source_day", present, prefix, by_columns=by_columns
        )
        record.update({"status": "joined", "features": present, **stats})
        ENRICHMENT_AUDIT.append(record)
        log(f"enrichment joined: {name} | features={len(present)} | coverage={stats['matched_coverage']:.1%}")
        return result
    except Exception as exc:
        record.update({"status": "failed", "reason": f"{type(exc).__name__}: {str(exc)[:220]}"})
        ENRICHMENT_AUDIT.append(record)
        log(f"enrichment failed safely: {name} — {record['reason']}")
        return df


def enrich_all(df):
    global ENRICHMENT_AUDIT
    ENRICHMENT_AUDIT = []
    out = df.copy()
    out = enrich_source(
        out, "usage_lifecycle_daily", CONFIG["ENRICH_USAGE"], CONFIG["USAGE_LIFECYCLE_TABLE"],
        f"{CONFIG['SILVER_PREFIX']}/usage_lifecycle_daily", "device_id", ["DEVICE_ID", "device_id"],
        ["days_since_last_failure", "days_since_last_maintenance", "failure_count_30d",
         "cumulative_failure_count", "cumulative_outage_min", "days_in_service"],
        "enr_usage_", synthetic_key="usage_lifecycle_daily",
    )
    out = enrich_source(
        out, "station_network_daily", CONFIG["ENRICH_STATION"], CONFIG["STATION_NETWORK_TABLE"],
        f"{CONFIG['SILVER_PREFIX']}/station_network_daily", "FACILITY_ID", ["FACILITY_ID", "facility_id"],
        ["devices_failed", "total_failure_events", "avg_downtime_minutes", "is_coordinated_failure", "is_major_station_event"],
        "enr_station_", by_mapping={"mars_device_category": ["mars_device_category", "device_category"]},
        synthetic_key="station_network_daily",
    )
    out = enrich_source(
        out, "metric_daily", CONFIG["ENRICH_METRIC"], CONFIG["METRIC_DAILY_TABLE"],
        f"{CONFIG['SILVER_PREFIX']}/metric_daily", "device_id", ["DEVICE_ID", "device_id"],
        ["m401_avg_txn_time_ms", "m401_p95_txn_time_ms", "m401_p99_txn_time_ms", "m401_slow_tap_pct",
         "m401_rolling_7d_avg_ms", "m401_z_score_vs_28d", "volume_drop_flag", "comms_total_count", "comms_event_flag"],
        "enr_metric_", synthetic_key="metric_daily",
    )
    out = enrich_source(
        out, "device_uptime_intervals", CONFIG["ENRICH_UPTIME"], CONFIG["UPTIME_INTERVALS_TABLE"],
        f"{CONFIG['SILVER_PREFIX']}/device_uptime_intervals", "device_id", ["DEVICE_ID", "device_id"],
        ["uptime_pct", "downtime_hours", "hours_since_last_hb", "is_silent_device", "distinct_message_types", "total_msg_count"],
        "enr_uptime_", synthetic_key="device_uptime_intervals",
    )
    return out


## 5. Feature contract, target preparation, and temporal split design


In [ ]:
def feature_availability(column):
    if column in {"ae_hour", "ae_dow", "ae_month"}:
        return "known_at_prediction_time"
    if column.startswith("hist_"):
        return "strictly_prior_incident_history"
    if column.startswith("enr_"):
        return "strict_prior_day_asof_enrichment"
    if column in BASE_NUMERIC_FEATURES:
        return "upstream_preincident_rolling_window"
    if column in CONFIG["SAFE_CATEGORICAL_CANDIDATES"]:
        return "static_or_slowly_changing_asset_attribute"
    return "unknown"


def build_feature_spec(df, target_col):
    blocked = set(HARD_LEAK_COLUMNS) | {CONFIG["SEVERITY_TARGET"], CONFIG["ROOTCAUSE_TARGET"], target_col}
    if not CONFIG["ALLOW_INCIDENT_TIME_METADATA"]:
        blocked |= INCIDENT_TIME_METADATA
    numeric = [c for c in BASE_NUMERIC_FEATURES if c in df.columns]
    numeric += [c for c in ("ae_hour", "ae_dow", "ae_month") if c in df.columns]
    numeric += [c for c in df.columns if c.startswith("hist_") or c.startswith("enr_")]
    numeric = [c for c in dict.fromkeys(numeric) if c not in blocked and pd.api.types.is_numeric_dtype(df[c])]
    categorical = [
        c for c in CONFIG["SAFE_CATEGORICAL_CANDIDATES"]
        if c in df.columns and c not in blocked
    ]
    selected = numeric + categorical
    rows = []
    for column in selected:
        rows.append({
            "feature": column,
            "dtype": str(df[column].dtype),
            "feature_type": "numeric" if column in numeric else "categorical",
            "availability_contract": feature_availability(column),
            "missing_rate": float(df[column].isna().mean()),
            "n_unique": int(df[column].nunique(dropna=True)),
            "selection_status": "selected",
        })
    present_blocked = sorted(c for c in blocked if c in df.columns)
    for column in present_blocked:
        rows.append({
            "feature": column,
            "dtype": str(df[column].dtype),
            "feature_type": "blocked",
            "availability_contract": "post_incident_or_label_leakage",
            "missing_rate": float(df[column].isna().mean()),
            "n_unique": int(df[column].nunique(dropna=True)),
            "selection_status": "blocked",
        })
    return numeric, categorical, pd.DataFrame(rows).sort_values(["selection_status", "feature"])


def make_temporal_splits(df):
    out = df.copy().sort_values(["_ae_date", "_row_id"], kind="mergesort")
    unique_dates = pd.Series(out["_ae_date"].dropna().unique()).sort_values().reset_index(drop=True)
    if len(unique_dates) < 12:
        raise ValueError(f"Only {len(unique_dates)} unique incident days; temporal train/validation/test split is not credible")
    train_end_pos = max(1, int(len(unique_dates) * CONFIG["TRAIN_FRACTION"]) - 1)
    validation_end_pos = max(train_end_pos + 1, int(len(unique_dates) * (CONFIG["TRAIN_FRACTION"] + CONFIG["VALIDATION_FRACTION"])) - 1)
    validation_end_pos = min(validation_end_pos, len(unique_dates) - 2)
    train_end = unique_dates.iloc[train_end_pos]
    validation_end = unique_dates.iloc[validation_end_pos]
    embargo = pd.Timedelta(days=CONFIG["EMBARGO_DAYS"])
    out["_split"] = np.select(
        [
            out["_ae_date"] <= train_end,
            (out["_ae_date"] > train_end + embargo) & (out["_ae_date"] <= validation_end),
            out["_ae_date"] > validation_end + embargo,
        ],
        ["train", "validation", "test"],
        default="embargo",
    )
    summary = (
        out.groupby("_split")
        .agg(rows=("_row_id", "size"), date_start=("_ae_date", "min"), date_end=("_ae_date", "max"))
        .reset_index()
    )
    meta = {
        "train_end": str(train_end),
        "validation_end": str(validation_end),
        "embargo_days": CONFIG["EMBARGO_DAYS"],
        "rows": summary.to_dict(orient="records"),
    }
    required = {"train", "validation", "test"}
    if not required.issubset(set(out["_split"])):
        raise ValueError(f"Temporal split lacks a required partition: {meta}")
    return out, meta


def clean_target(series):
    value = series.astype("string").str.strip()
    return value.mask(value.isna() | value.str.upper().isin(["", "NAN", "NONE", "NULL", "UNKNOWN"]))


def prepare_head_target(df, target_col):
    if target_col not in df.columns:
        return None, {"reason": f"target column {target_col!r} is absent"}
    data = df.copy()
    data["_target_raw"] = clean_target(data[target_col])
    data = data[data["_target_raw"].notna()].copy()
    # 2026-07-26: severity trains on the binary MAJOR/CRITICAL collapse the
    # dashboard consumes, not the 4-5 class failure_level_label. macro-F1 averages
    # over classes, so the skewed multiclass target was capping the score
    # structurally (TVM 0.478 vs a 0.55 floor). UNKNOWN rows are dropped rather
    # than folded into MAJOR.
    if CONFIG.get("SEVERITY_BINARY_TARGET", False) and target_col == CONFIG["SEVERITY_TARGET"]:
        collapsed = severity_bucket(data["_target_raw"])
        keep = collapsed.isin(["MAJOR", "CRITICAL"])
        dropped = int((~keep).sum())
        if dropped:
            log(f"severity target: dropped {dropped} row(s) whose code did not map to MAJOR/CRITICAL")
        data = data.loc[keep].copy()
        data["_target_raw"] = collapsed.loc[keep]
    train = data[data["_split"] == "train"]
    if train.empty:
        return None, {"reason": "no labelled target rows in the training partition"}
    counts = train["_target_raw"].value_counts()
    rare = counts[counts < CONFIG["MIN_TRAIN_CLASS_ROWS"]].index.tolist()
    data["_target"] = data["_target_raw"].where(~data["_target_raw"].isin(rare), "OTHER")
    train_labels = sorted(data.loc[data["_split"] == "train", "_target"].unique().tolist())
    data["_label_supported"] = data["_target"].isin(train_labels)
    unknown_by_split = (
        data.assign(_unsupported=~data["_label_supported"])
        .groupby("_split")["_unsupported"]
        .agg(["sum", "mean"])
        .rename(columns={"sum": "unseen_label_rows", "mean": "unseen_label_rate"})
        .to_dict(orient="index")
    )
    meta = {
        "target": target_col,
        "raw_rows": int(len(data)),
        "rare_train_classes_collapsed_to_OTHER": rare,
        "train_classes": train_labels,
        "unseen_labels_by_split": unknown_by_split,
    }
    if len(train_labels) < 2:
        return None, {**meta, "reason": "fewer than two target classes remain in training"}
    return data, meta


## 6. Leakage audit, preprocessing, metrics, and model factories


In [ ]:
def _single_feature_validation_f1(train, validation, feature):
    """A diagnostic, not automatic proof of leakage. It uses validation, never the test partition."""
    train = train[[feature, "_y"]].dropna(subset=["_y"]).copy()
    validation = validation[[feature, "_y"]].dropna(subset=["_y"]).copy()
    validation = validation[validation["_y"].isin(train["_y"].unique())]
    if len(train) < 30 or len(validation) < 20 or train["_y"].nunique() < 2:
        return np.nan
    try:
        if pd.api.types.is_numeric_dtype(train[feature]):
            median = pd.to_numeric(train[feature], errors="coerce").median()
            x_train = pd.to_numeric(train[feature], errors="coerce").fillna(median).to_numpy().reshape(-1, 1)
            x_valid = pd.to_numeric(validation[feature], errors="coerce").fillna(median).to_numpy().reshape(-1, 1)
        else:
            lookup = {value: idx for idx, value in enumerate(train[feature].astype("string").fillna("<MISSING>").unique())}
            x_train = train[feature].astype("string").fillna("<MISSING>").map(lookup).fillna(-1).to_numpy().reshape(-1, 1)
            x_valid = validation[feature].astype("string").fillna("<MISSING>").map(lookup).fillna(-1).to_numpy().reshape(-1, 1)
        model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=max(10, len(train) // 50), random_state=RS)
        model.fit(x_train, train["_y"])
        pred = model.predict(x_valid)
        return float(f1_score(validation["_y"], pred, average="macro", zero_division=0))
    except Exception:
        return np.nan


def run_feature_audit(train, validation, features, feature_manifest):
    audit = feature_manifest.copy()
    scores = {}
    for feature in features:
        scores[feature] = _single_feature_validation_f1(train, validation, feature)
    audit["single_feature_validation_macro_f1"] = audit["feature"].map(scores)
    audit["suspicion"] = np.where(
        audit["single_feature_validation_macro_f1"] >= CONFIG["SUSPICIOUS_SINGLE_FEATURE_F1"],
        "review", "none"
    )
    suspicious = audit.loc[
        (audit["selection_status"] == "selected") & (audit["suspicion"] == "review"), "feature"
    ].tolist()
    return audit, suspicious


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False)
    except TypeError:  # older SageMaker scikit-learn kernels
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse=False)


def make_preprocessor(numeric_features, categorical_features):
    transformers = []
    if numeric_features:
        transformers.append((
            "numeric",
            Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
            numeric_features,
        ))
    if categorical_features:
        transformers.append((
            "categorical",
            Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", make_one_hot_encoder())]),
            categorical_features,
        ))
    if not transformers:
        raise ValueError("No usable numeric or categorical features remain after the feature contract")
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.0, verbose_feature_names_out=False)


def model_builders(n_classes):
    builders = {
        "BaselineMostFrequent": lambda: DummyClassifier(strategy="most_frequent"),
        "LogisticRegression": lambda: LogisticRegression(max_iter=3000, C=1.0, solver="lbfgs", n_jobs=None),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=CONFIG["RF_TREES"],
            max_depth=CONFIG["REG_RF_MAX_DEPTH"],
            min_samples_leaf=CONFIG["REG_RF_MIN_LEAF"],
            max_features="sqrt", random_state=RS, n_jobs=-1,
        ),
        "HistGradientBoosting": lambda: HistGradientBoostingClassifier(
            max_iter=CONFIG["REG_HGB_MAX_ITER"], learning_rate=0.05,
            max_leaf_nodes=CONFIG["REG_HGB_LEAVES"],
            min_samples_leaf=CONFIG["REG_HGB_MIN_LEAF"],
            l2_regularization=CONFIG["REG_HGB_L2"],
            early_stopping=True, validation_fraction=0.15, n_iter_no_change=20,
            random_state=RS,
        ),
    }
    if xgb is not None and CONFIG["RUN_XGBOOST"]:
        builders["XGBoost"] = lambda: xgb.XGBClassifier(
            n_estimators=400, max_depth=CONFIG["REG_XGB_MAX_DEPTH"], learning_rate=0.05,
            subsample=0.80, colsample_bytree=0.70,
            min_child_weight=CONFIG["REG_XGB_MIN_CHILD_WEIGHT"],
            reg_lambda=CONFIG["REG_XGB_LAMBDA"],
            objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss", tree_method="hist",
            random_state=RS, n_jobs=-1, verbosity=0,
        )
    if lgb is not None and CONFIG["RUN_LIGHTGBM"]:
        builders["LightGBM"] = lambda: lgb.LGBMClassifier(
            n_estimators=400, num_leaves=CONFIG["REG_LGB_LEAVES"], learning_rate=0.05,
            subsample=0.80, colsample_bytree=0.70,
            min_child_samples=CONFIG["REG_LGB_MIN_CHILD"],
            reg_lambda=CONFIG["REG_LGB_LAMBDA"],
            objective="multiclass", num_class=n_classes, random_state=RS, n_jobs=-1, verbose=-1,
        )
    if catb is not None and CONFIG["RUN_CATBOOST"]:
        builders["CatBoost"] = lambda: catb.CatBoostClassifier(
            iterations=400, depth=CONFIG["REG_CAT_DEPTH"], learning_rate=0.05,
            l2_leaf_reg=CONFIG["REG_CAT_L2"], loss_function="MultiClass",
            random_seed=RS, verbose=False, allow_writing_files=False,
        )
    return builders


def maybe_calibrate(model, transformed, y):
    """Wrap a fitted estimator in probability calibration when CONFIG asks for it.

    GATE/root_cause (26-Jul) cleared its macro-F1 floor and failed the promotion
    gate only on train-validation gap + test ECE 0.192 vs the 0.15 guardrail, so
    calibration is the difference between a usable head and a rejected one.

    Skipped rather than raised when a class has fewer rows than folds -- a small
    minority class must not take the whole head down.
    """
    if not CONFIG.get("CALIBRATE_PROBABILITIES", False) or CalibratedClassifierCV is None:
        return model
    folds = int(CONFIG.get("CALIBRATION_FOLDS", 3))
    labels, counts = np.unique(np.asarray(y), return_counts=True)
    if len(labels) < 2 or counts.min() < folds:
        return model
    try:
        calibrated = CalibratedClassifierCV(
            model, method=CONFIG.get("CALIBRATION_METHOD", "isotonic"), cv=folds
        )
        calibrated.fit(transformed, y)
        return calibrated
    except Exception as exc:                                   # noqa: BLE001
        print(f"[calibration] skipped ({type(exc).__name__}: {str(exc)[:120]})")
        return model


def fit_model(factory, X, y, numeric_features, categorical_features):
    preprocessor = make_preprocessor(numeric_features, categorical_features)
    transformed = preprocessor.fit_transform(X)
    model = factory()
    if isinstance(model, DummyClassifier):
        model.fit(transformed, y)
    else:
        weights = compute_sample_weight(class_weight="balanced", y=y)
        try:
            model.fit(transformed, y, sample_weight=weights)
        except TypeError:
            model.fit(transformed, y)
        model = maybe_calibrate(model, transformed, y)
    return model, preprocessor


def aligned_probabilities(model, transformed, n_classes):
    raw = np.asarray(model.predict_proba(transformed))
    result = np.zeros((len(transformed), n_classes), dtype=float)
    classes = np.asarray(getattr(model, "classes_", np.arange(raw.shape[1])), dtype=int)
    for position, label in enumerate(classes):
        if 0 <= label < n_classes:
            result[:, label] = raw[:, position]
    row_sum = result.sum(axis=1, keepdims=True)
    result = np.divide(result, row_sum, out=np.full_like(result, 1.0 / n_classes), where=row_sum > 0)
    return result


def expected_calibration_error(y_true, probabilities, n_bins=10):
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    correct = (np.asarray(y_true) == predicted).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for idx in range(n_bins):
        mask = (confidence >= bins[idx]) & (confidence < bins[idx + 1] if idx < n_bins - 1 else confidence <= bins[idx + 1])
        if mask.any():
            ece += abs(correct[mask].mean() - confidence[mask].mean()) * mask.mean()
    return float(ece)


def metric_bundle(y_true, probabilities, all_labels):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predicted = probabilities.argmax(axis=1)
    observed = np.unique(y_true)
    metrics = {
        "rows": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, predicted)),
        "f1_macro": float(f1_score(y_true, predicted, labels=observed, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, predicted, labels=observed, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)),
        "precision_macro": float(precision_score(y_true, predicted, labels=observed, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, predicted, labels=observed, average="macro", zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y_true, predicted)),
        "mcc": float(matthews_corrcoef(y_true, predicted)),
        "ece": expected_calibration_error(y_true, probabilities),
        "multiclass_brier": float(np.mean(np.sum((np.eye(len(all_labels))[y_true] - probabilities) ** 2, axis=1))),
    }
    try:
        metrics["log_loss"] = float(log_loss(y_true, probabilities, labels=all_labels))
    except Exception:
        metrics["log_loss"] = np.nan
    aucs, aps = [], []
    for label in all_labels:
        target = (y_true == label).astype(int)
        if target.min() == target.max():
            continue
        try:
            aucs.append(roc_auc_score(target, probabilities[:, label]))
            aps.append(average_precision_score(target, probabilities[:, label]))
        except Exception:
            continue
    metrics["roc_auc_macro_ovr"] = float(np.mean(aucs)) if aucs else np.nan
    metrics["pr_auc_macro_ovr"] = float(np.mean(aps)) if aps else np.nan
    return metrics


def probability_entropy(probabilities):
    clipped = np.clip(np.asarray(probabilities), 1e-12, 1.0)
    return -np.sum(clipped * np.log(clipped), axis=1)


## 7. Cross-validation and Optuna tuning — both use training history only


In [ ]:
def purged_walk_forward_indices(frame, n_splits):
    dates = pd.Series(frame["_ae_date"].dropna().unique()).sort_values().reset_index(drop=True)
    if len(dates) < max(8, n_splits + 3):
        return []
    width = max(1, len(dates) // (n_splits + 1))
    embargo = pd.Timedelta(days=CONFIG["EMBARGO_DAYS"])
    splits = []
    for fold in range(n_splits):
        validation_start_pos = (fold + 1) * width
        if validation_start_pos >= len(dates) - 1:
            break
        validation_end_pos = min(len(dates) - 1, validation_start_pos + width - 1)
        validation_start = dates.iloc[validation_start_pos]
        validation_end = dates.iloc[validation_end_pos]
        train_idx = np.flatnonzero(frame["_ae_date"].to_numpy() <= (validation_start - embargo))
        validation_idx = np.flatnonzero(
            (frame["_ae_date"].to_numpy() >= validation_start) & (frame["_ae_date"].to_numpy() <= validation_end)
        )
        if len(train_idx) and len(validation_idx):
            splits.append((fold + 1, train_idx, validation_idx, validation_start, validation_end))
    return splits


def run_robustness_cv(frame, factory, numeric_features, categorical_features, n_classes, out_path):
    rows = []
    # 1) Time-respecting, embargoed walk-forward validation.
    for fold, train_idx, valid_idx, start, end in purged_walk_forward_indices(frame, CONFIG["WALK_FORWARD_SPLITS"]):
        train = frame.iloc[train_idx]
        valid = frame.iloc[valid_idx]
        if train["_y"].nunique() < 2 or not set(valid["_y"]).issubset(set(train["_y"])):
            continue
        try:
            model, prep = fit_model(factory, train[numeric_features + categorical_features], train["_y"], numeric_features, categorical_features)
            probability = aligned_probabilities(model, prep.transform(valid[numeric_features + categorical_features]), n_classes)
            metrics = metric_bundle(valid["_y"], probability, np.arange(n_classes))
            rows.append({"scheme": "purged_walk_forward", "fold": fold, "date_start": str(start), "date_end": str(end), **metrics})
        except Exception as exc:
            rows.append({"scheme": "purged_walk_forward", "fold": fold, "error": f"{type(exc).__name__}: {str(exc)[:160]}"})

    # 2) Unseen-device stress test. It is intentionally labelled as a stress test because it is not time ordered.
    groups = frame["device_id"].astype(str)
    if groups.nunique() >= 4:
        splitter = GroupShuffleSplit(
            n_splits=CONFIG["GROUP_STRESS_SPLITS"], test_size=CONFIG["GROUP_STRESS_TEST_FRACTION"], random_state=RS
        )
        for fold, (train_idx, valid_idx) in enumerate(splitter.split(frame, frame["_y"], groups=groups), 1):
            train = frame.iloc[train_idx]
            valid = frame.iloc[valid_idx]
            if train["_y"].nunique() < 2 or not set(valid["_y"]).issubset(set(train["_y"])):
                continue
            try:
                model, prep = fit_model(factory, train[numeric_features + categorical_features], train["_y"], numeric_features, categorical_features)
                probability = aligned_probabilities(model, prep.transform(valid[numeric_features + categorical_features]), n_classes)
                metrics = metric_bundle(valid["_y"], probability, np.arange(n_classes))
                rows.append({"scheme": "unseen_device_stress", "fold": fold, "date_start": None, "date_end": None, **metrics})
            except Exception as exc:
                rows.append({"scheme": "unseen_device_stress", "fold": fold, "error": f"{type(exc).__name__}: {str(exc)[:160]}"})
    cv = pd.DataFrame(rows)
    write_csv(cv, out_path)
    if not cv.empty and "f1_macro" in cv.columns:
        summary = (
            cv.groupby("scheme")[["f1_macro", "accuracy", "balanced_accuracy", "ece"]]
            .agg(["mean", "std", "min", "max"])
            .round(4)
        )
        summary.to_csv(Path(out_path).with_name(Path(out_path).stem + "_summary.csv"))
        return cv, summary
    return cv, pd.DataFrame()


def tune_xgboost(train, numeric_features, categorical_features, n_classes):
    """Optional inner walk-forward tuning. It never looks at the main validation or final test partitions."""
    if not (CONFIG["RUN_OPTUNA"] and xgb is not None and optuna is not None):
        return None
    tune = train.sort_values("_ae_date", kind="mergesort").copy()
    if CONFIG["TUNE_MAX_ROWS"] and len(tune) > CONFIG["TUNE_MAX_ROWS"]:
        tune = tune.tail(CONFIG["TUNE_MAX_ROWS"]).copy()
    splits = purged_walk_forward_indices(tune, CONFIG["TUNE_CV_SPLITS"])
    if len(splits) < 2:
        return None
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 8.0),
        }
        scores = []
        for _, tr_idx, va_idx, _, _ in splits:
            tr, va = tune.iloc[tr_idx], tune.iloc[va_idx]
            if tr["_y"].nunique() < 2 or not set(va["_y"]).issubset(set(tr["_y"])):
                continue
            factory = lambda p=params: xgb.XGBClassifier(
                objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss", tree_method="hist",
                random_state=RS, n_jobs=-1, verbosity=0, **p,
            )
            model, prep = fit_model(factory, tr[numeric_features + categorical_features], tr["_y"], numeric_features, categorical_features)
            proba = aligned_probabilities(model, prep.transform(va[numeric_features + categorical_features]), n_classes)
            scores.append(metric_bundle(va["_y"], proba, np.arange(n_classes))["f1_macro"])
        return float(np.mean(scores)) if scores else 0.0

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RS))
    study.optimize(objective, n_trials=CONFIG["N_OPTUNA_TRIALS"], show_progress_bar=False)
    return {"best_params": study.best_params, "inner_walk_forward_f1": float(study.best_value), "trials": int(len(study.trials))}


## 8. Insight and diagnostic visualisation helpers


In [ ]:
def plot_confusion_matrix(y_true, probabilities, labels, title, path):
    predicted = probabilities.argmax(axis=1)
    cm = confusion_matrix(y_true, predicted, labels=np.arange(len(labels)))
    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.85), max(5, len(labels) * 0.7)))
    image = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if norm[i, j] > 0.55 else "#222")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
    plot_or_close(path)


def plot_roc_pr(y_true, probabilities, labels, title_prefix, output_dir):
    y_true = np.asarray(y_true)
    all_labels = np.arange(len(labels))
    support = pd.Series(y_true).value_counts()
    ranked = support.index.tolist()[: min(8, len(labels))]
    # ROC
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    for pos, label in enumerate(ranked):
        binary = (y_true == label).astype(int)
        if binary.min() == binary.max():
            continue
        fpr, tpr, _ = roc_curve(binary, probabilities[:, label])
        auc = roc_auc_score(binary, probabilities[:, label])
        ax.plot(fpr, tpr, label=f"{labels[label][:24]} (AUC {auc:.2f})", color=PASTEL[pos % len(PASTEL)])
    ax.plot([0, 1], [0, 1], "--", color="#aaa", lw=1)
    ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title=f"{title_prefix} — ROC (test, OVR)")
    ax.legend(fontsize=7, loc="lower right")
    plot_or_close(Path(output_dir) / "roc_ovr_test.png")
    # PR
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    for pos, label in enumerate(ranked):
        binary = (y_true == label).astype(int)
        if binary.min() == binary.max():
            continue
        precision, recall, _ = precision_recall_curve(binary, probabilities[:, label])
        ap = average_precision_score(binary, probabilities[:, label])
        ax.plot(recall, precision, label=f"{labels[label][:24]} (AP {ap:.2f})", color=PASTEL[pos % len(PASTEL)])
    ax.set(xlabel="Recall", ylabel="Precision", title=f"{title_prefix} — PR (test, OVR)")
    ax.legend(fontsize=7, loc="lower left")
    plot_or_close(Path(output_dir) / "pr_ovr_test.png")


def plot_reliability(y_true, probabilities, title, path):
    confidence = probabilities.max(axis=1)
    correct = (probabilities.argmax(axis=1) == np.asarray(y_true)).astype(float)
    bins = np.linspace(0, 1, 11)
    x_values, y_values, counts = [], [], []
    for i in range(len(bins) - 1):
        mask = (confidence >= bins[i]) & (confidence < bins[i + 1] if i < len(bins) - 2 else confidence <= bins[i + 1])
        if mask.sum() >= 5:
            x_values.append(float(confidence[mask].mean()))
            y_values.append(float(correct[mask].mean()))
            counts.append(int(mask.sum()))
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    ax.plot([0, 1], [0, 1], "--", color="#999", label="Perfect calibration")
    ax.plot(x_values, y_values, "o-", color="#377eb8", label="Model")
    ax.set(xlabel="Mean predicted confidence", ylabel="Observed accuracy", title=title)
    ax.legend(fontsize=8)
    plot_or_close(path)
    return pd.DataFrame({"mean_confidence": x_values, "observed_accuracy": y_values, "rows": counts})


def plot_metric_bakeoff(leaderboard, title, path):
    if leaderboard.empty:
        return
    ranked = leaderboard.sort_values("validation_f1_macro", ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3.5, 0.5 * len(ranked))))
    ax.barh(ranked["model"], ranked["train_f1_macro"], color="#9ecae1", label="Train")
    ax.barh(ranked["model"], ranked["validation_f1_macro"], color="#3182bd", alpha=0.9, label="Validation")
    if "test_f1_macro" in ranked:
        ax.scatter(ranked["test_f1_macro"], np.arange(len(ranked)), marker="D", color="#e6550d", label="Test (diagnostic)", zorder=3)
    ax.set(xlabel="Macro-F1", title=title, xlim=(0, 1))
    ax.legend(fontsize=8)
    plot_or_close(path)


def plot_heatmap(matrix, title, path, fmt=".2f"):
    if matrix.empty:
        return
    fig, ax = plt.subplots(figsize=(max(6, 0.75 * matrix.shape[1]), max(4.5, 0.5 * matrix.shape[0])))
    image = ax.imshow(matrix.values.astype(float), cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(matrix.shape[1])); ax.set_yticks(range(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticklabels(matrix.index, fontsize=8)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, format(float(matrix.iloc[i, j]), fmt), ha="center", va="center", fontsize=7)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
    plot_or_close(path)


def safe_permutation_importance(model, prep, validation, features, y, numeric_features, categorical_features, output_dir, title):
    if len(validation) < 30:
        return pd.DataFrame()
    try:
        transformed = prep.transform(validation[features])
        result = permutation_importance(
            model, transformed, y, n_repeats=8, random_state=RS,
            scoring=lambda estimator, X, yy: f1_score(yy, estimator.predict(X), average="macro", zero_division=0),
        )
        names = list(prep.get_feature_names_out())
        values = pd.DataFrame({"transformed_feature": names, "importance_mean": result.importances_mean, "importance_std": result.importances_std})
        def original(name):
            if name in numeric_features:
                return name
            for column in categorical_features:
                if name == column or name.startswith(f"{column}_"):
                    return column
            return name
        values["feature"] = values["transformed_feature"].map(original)
        grouped = values.groupby("feature", as_index=False).agg(
            importance_mean=("importance_mean", "sum"), importance_std=("importance_std", "mean")
        ).sort_values("importance_mean", ascending=False)
        write_csv(grouped, Path(output_dir) / "permutation_importance_validation.csv")
        if not grouped.empty:
            ax = grouped.head(20).sort_values("importance_mean").plot(
                kind="barh", x="feature", y="importance_mean", legend=False, color="#4daf4a", figsize=(7, 6)
            )
            ax.set_title(title); ax.set_xlabel("Validation macro-F1 decrease after permutation")
            plot_or_close(Path(output_dir) / "permutation_importance_validation.png")
        return grouped
    except Exception as exc:
        log(f"permutation importance skipped: {type(exc).__name__}: {str(exc)[:160]}")
        return pd.DataFrame()


def optional_shap_summary(model, prep, validation, output_dir, title):
    """SHAP is optional and uses validation only; permutation importance remains the portable default."""
    status = {"available": bool(shap is not None), "written": False}
    if shap is None or len(validation) < 30:
        write_json(status, Path(output_dir) / "shap_status.json")
        return status
    model_name = type(model).__name__
    if not any(token in model_name for token in ("Forest", "XGB", "LGBM", "Boost")):
        status["reason"] = f"{model_name} is not a supported tree explainer target"
        write_json(status, Path(output_dir) / "shap_status.json")
        return status
    try:
        transformed = prep.transform(validation)
        names = list(prep.get_feature_names_out())
        sample = pd.DataFrame(transformed, columns=names).sample(min(1000, len(validation)), random_state=RS)
        explainer = shap.TreeExplainer(model)
        values = explainer.shap_values(sample)
        if isinstance(values, list):
            mean_abs = np.mean([np.abs(v) for v in values], axis=0).mean(axis=0)
        else:
            array = np.asarray(values)
            if array.ndim == 3:
                mean_abs = np.abs(array).mean(axis=(0, 2)) if array.shape[1] == len(names) else np.abs(array).mean(axis=(0, 1))
            else:
                mean_abs = np.abs(array).mean(axis=0)
        importance = pd.Series(mean_abs, index=names).sort_values(ascending=False)
        write_csv(importance.rename("mean_abs_shap").reset_index().rename(columns={"index": "feature"}), Path(output_dir) / "shap_importance_validation.csv")
        ax = importance.head(20).sort_values().plot(kind="barh", color="#984ea3", figsize=(7, 6))
        ax.set_title(title)
        plot_or_close(Path(output_dir) / "shap_importance_validation.png")
        status["written"] = True
    except Exception as exc:
        status["reason"] = f"{type(exc).__name__}: {str(exc)[:180]}"
    write_json(status, Path(output_dir) / "shap_status.json")
    return status


## 9. Rich discovery pack — coverage, Pareto, severity/component lift, hotspots, and precursors


In [ ]:
def write_cross_device_overview(df, output_root):
    out = Path(output_root) / "cross_device"
    out.mkdir(parents=True, exist_ok=True)
    coverage = (
        df.groupby("mars_device_category")
        .agg(
            incidents=("_row_id", "size"), devices=("device_id", "nunique"),
            first_incident=("AE_START_DTM", "min"), last_incident=("AE_START_DTM", "max"),
        )
        .reset_index()
    )
    for target in (CONFIG["SEVERITY_TARGET"], CONFIG["ROOTCAUSE_TARGET"]):
        if target in df.columns:
            counts = df.groupby("mars_device_category")[target].nunique(dropna=True).rename(f"{target}_classes")
            coverage = coverage.merge(counts, left_on="mars_device_category", right_index=True, how="left")
    write_csv(coverage, out / "device_type_coverage.csv")
    if CONFIG["SEVERITY_TARGET"] in df.columns:
        mix = pd.crosstab(df["mars_device_category"], severity_bucket(df[CONFIG["SEVERITY_TARGET"]]))
        write_csv(mix.reset_index(), out / "severity_bucket_by_type.csv")
        ax = mix.plot(kind="bar", stacked=True, color=["#4daf4a", "#ff7f00", "#999999"], figsize=(8, 4.5))
        ax.set_title("Severity bucket mix by device type"); ax.set_xlabel("")
        plot_or_close(out / "severity_bucket_by_type.png")
    overview_dates = df["_ae_date"] if "_ae_date" in df.columns else pd.to_datetime(df["AE_START_DTM"], errors="coerce").dt.normalize()
    monthly = df.assign(month=overview_dates.dt.to_period("M").astype(str)).groupby(["month", "mars_device_category"]).size().unstack(fill_value=0)
    write_csv(monthly.reset_index(), out / "monthly_incident_volume.csv")
    if not monthly.empty:
        ax = monthly.plot(figsize=(10, 4.5), marker="o", ms=2)
        ax.set_title("Incident volume over time by device type"); ax.set_xlabel("")
        plt.xticks(rotation=45, ha="right")
        plot_or_close(out / "monthly_incident_volume.png")





def write_cross_device_component_portfolio(output_root):
    """Unify only comparable, dashboard-ready component outputs; missing labels stay explicit."""
    root = Path(output_root)
    out = root / "cross_device"
    out.mkdir(parents=True, exist_ok=True)
    paths = {
        "TVM": root / "tvm" / "component_intelligence",
        "GATE": root / "gates" / "component_intelligence",
        "VALIDATOR": root / "validators" / "validator_oos_proxy" / "component_intelligence",
    }
    portfolios, device_components, prediction_quality = [], [], []
    unavailable = {}
    for category, base in paths.items():
        portfolio_path = base / "component_reliability_portfolio.csv"
        device_path = base / "device_component_reliability_ranking.csv"
        quality_path = base / "component_prediction_reliability.csv"
        status_path = base / "reliability_intelligence_summary.json"
        if portfolio_path.exists():
            frame = pd.read_csv(portfolio_path)
            if not frame.empty:
                frame["device_category"] = frame["device_category"].fillna(category) if "device_category" in frame.columns else category
                portfolios.append(frame)
        if device_path.exists():
            frame = pd.read_csv(device_path)
            if not frame.empty:
                frame["device_category"] = frame["device_category"].fillna(category) if "device_category" in frame.columns else category
                device_components.append(frame)
        if quality_path.exists():
            frame = pd.read_csv(quality_path)
            if not frame.empty:
                frame["device_category"] = frame["device_category"].fillna(category) if "device_category" in frame.columns else category
                prediction_quality.append(frame)
        if status_path.exists():
            try:
                unavailable[category] = json.loads(status_path.read_text()).get("status")
            except Exception:
                unavailable[category] = "status_unreadable"
        else:
            unavailable[category] = "not_available"

    result = {"status": "empty", "portfolio_rows": 0, "device_component_rows": 0, "prediction_quality_rows": 0, "category_status": unavailable}
    if portfolios:
        portfolio = pd.concat(portfolios, ignore_index=True, sort=False)
        portfolio["cross_device_priority_scope"] = "Same transparent historical triage formula within each device category; compare evidence columns as well as rank."
        portfolio = portfolio.sort_values(["reliability_priority_score", "historical_incidents"], ascending=False, kind="mergesort")
        write_csv(portfolio, out / "cross_device_component_reliability_portfolio.csv")
        mix = portfolio.pivot_table(index="component", columns="device_category", values="historical_incidents", aggfunc="sum", fill_value=0)
        write_csv(mix.reset_index(), out / "cross_device_component_incident_matrix.csv")
        totals = mix.sum(axis=0).replace(0, np.nan)
        mix_share = mix.div(totals, axis=1)
        write_csv(mix_share.reset_index(), out / "cross_device_component_mix_by_type.csv")
        if not mix_share.empty:
            plot_heatmap(mix_share, "Cross-device component incident mix by device type", out / "cross_device_component_mix_by_type.png")
        top = portfolio.head(25).sort_values("reliability_priority_score")
        if not top.empty:
            labels = (top["device_category"].astype(str) + " | " + top["component"].astype(str)).tolist()
            figure, axis = plt.subplots(figsize=(12, max(5, 0.32 * len(top) + 1)))
            axis.barh(labels, top["reliability_priority_score"], color="#984ea3")
            axis.set_title("Cross-device component reliability triage priority")
            axis.set_xlabel("Evidence-adjusted historical priority score (0–100)")
            plot_or_close(out / "cross_device_component_priority.png")
        result.update({"status": "available", "portfolio_rows": int(len(portfolio)), "components": int(portfolio["component"].nunique())})
    if device_components:
        device_component = pd.concat(device_components, ignore_index=True, sort=False)
        device_component = device_component.sort_values(["reliability_priority_score", "historical_incidents"], ascending=False, kind="mergesort")
        write_csv(device_component, out / "cross_device_device_component_reliability_ranking.csv")
        result["device_component_rows"] = int(len(device_component))
    if prediction_quality:
        quality = pd.concat(prediction_quality, ignore_index=True, sort=False)
        write_csv(quality, out / "cross_device_component_prediction_reliability.csv")
        result["prediction_quality_rows"] = int(len(quality))
    write_json(result, out / "cross_device_component_intelligence_summary.json")
    return result


def _read_operational_frame(path, category, category_column):
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    if frame.empty:
        return None
    if category_column not in frame.columns:
        frame[category_column] = category
    else:
        frame[category_column] = frame[category_column].fillna(category)
    return frame


def write_cross_device_operational_portfolio(output_root):
    """Combine category test feeds and component intelligence without assuming every head is available."""
    root = Path(output_root)
    out = root / "cross_device"
    out.mkdir(parents=True, exist_ok=True)
    category_paths = {"TVM": "tvm", "GATE": "gates", "VALIDATOR": "validators"}
    serial_frames, device_frames, incident_frames = [], [], []
    for category, folder in category_paths.items():
        category_out = root / folder
        serial = _read_operational_frame(category_out / "test_device_serial_action_ranking.csv", category, "device_category")
        device = _read_operational_frame(category_out / "test_device_action_ranking.csv", category, "device_category")
        incident = _read_operational_frame(category_out / "test_incident_predictions.csv", category, "mars_device_category")
        if serial is not None:
            serial_frames.append(serial)
        if device is not None:
            device_frames.append(device)
        if incident is not None:
            incident["device_category"] = incident["mars_device_category"].fillna(category)
            incident_frames.append(incident)
    result = {"status": "empty", "device_serial_rows": 0, "device_rows": 0, "incident_rows": 0}
    if serial_frames:
        serial = pd.concat(serial_frames, ignore_index=True, sort=False)
        serial = serial.sort_values(["priority_score", "test_incidents"], ascending=False, na_position="last")
        write_csv(serial, out / "cross_device_device_serial_action_ranking.csv")
        priority_mix = pd.crosstab(serial["device_category"], serial["highest_action_priority"]).reindex(columns=["monitor", "plan", "urgent"], fill_value=0)
        write_csv(priority_mix.reset_index(), out / "cross_device_action_priority_by_type.csv")
        if not priority_mix.empty:
            axis = priority_mix.plot(kind="bar", stacked=True, color=["#4daf4a", "#ff7f00", "#e41a1c"], figsize=(9, 4.8))
            axis.set_title("Cross-device action priority by device type")
            axis.set_xlabel(""); axis.set_ylabel("Device / serial pairs")
            plot_or_close(out / "cross_device_action_priority_by_type.png")
        top = serial.head(25).sort_values("priority_score")
        if not top.empty:
            labels = (top["device_category"].astype(str) + " | " + top["device_id"].astype(str) + " | " + top["serial_number"].astype(str)).tolist()
            figure, axis = plt.subplots(figsize=(12, max(5, 0.33 * len(top) + 1)))
            axis.barh(labels, top["priority_score"], color="#984ea3")
            axis.set_title("Cross-device top priority device / serial pairs (frozen test)")
            axis.set_xlabel("Priority score")
            plot_or_close(out / "cross_device_top_device_serial_priority.png")
        result.update({"status": "available", "device_serial_rows": int(len(serial)), "top_priority_score": float(serial["priority_score"].max()) if len(serial) else None})
    if device_frames:
        devices = pd.concat(device_frames, ignore_index=True, sort=False).sort_values("priority_score", ascending=False, na_position="last")
        write_csv(devices, out / "cross_device_device_action_ranking.csv")
        result["device_rows"] = int(len(devices))
    if incident_frames:
        incidents = pd.concat(incident_frames, ignore_index=True, sort=False)
        write_csv(incidents, out / "cross_device_test_incident_predictions.csv")
        result["incident_rows"] = int(len(incidents))
        quality_rows = []
        for category, group in incidents.groupby("device_category", dropna=False):
            row = {"device_category": category, "test_incident_predictions": int(len(group)), "devices": int(group["device_id"].nunique())}
            for head in ("severity", "root_cause"):
                correct = f"{head}_correct"
                if correct in group.columns:
                    numeric = pd.to_numeric(group[correct], errors="coerce")
                    row[f"{head}_evaluable_predictions"] = int(numeric.notna().sum())
                    row[f"{head}_accuracy_on_evaluable_predictions"] = float(numeric.mean()) if numeric.notna().any() else np.nan
            quality_rows.append(row)
        write_csv(pd.DataFrame(quality_rows), out / "cross_device_test_prediction_quality.csv")
    result["component_intelligence"] = write_cross_device_component_portfolio(output_root)
    write_json(result, out / "cross_device_operational_status.json")
    return result

def category_discovery_pack(df, category, output_dir):
    out = Path(output_dir) / "discoveries"
    out.mkdir(parents=True, exist_ok=True)
    summary = {"device_category": category, "incidents": int(len(df)), "devices": int(df["device_id"].nunique())}
    severity_col, component_col = CONFIG["SEVERITY_TARGET"], CONFIG["ROOTCAUSE_TARGET"]
    if severity_col in df.columns:
        df = df.copy()
        df["_severity_bucket"] = severity_bucket(df[severity_col])
        sev = df[severity_col].astype("string").value_counts(dropna=False).rename_axis("severity").reset_index(name="incidents")
        sev["share"] = sev["incidents"] / max(1, sev["incidents"].sum())
        write_csv(sev, out / "severity_distribution.csv")
        summary["critical_share"] = float((df["_severity_bucket"] == "CRITICAL").mean())
    if component_col in df.columns:
        pareto = df[component_col].astype("string").value_counts(dropna=False).rename_axis("component").reset_index(name="incidents")
        pareto["share"] = pareto["incidents"] / max(1, pareto["incidents"].sum())
        pareto["cumulative_share"] = pareto["share"].cumsum()
        write_csv(pareto, out / "root_cause_pareto.csv")
        summary["top_component"] = str(pareto.iloc[0]["component"]) if len(pareto) else None
        ax = pareto.head(15).sort_values("incidents").plot(kind="barh", x="component", y="incidents", legend=False, color="#377eb8", figsize=(7, 5.5))
        ax.set_title(f"{category}: derived component Pareto")
        plot_or_close(out / "root_cause_pareto.png")
    if severity_col in df.columns and component_col in df.columns:
        counts = pd.crosstab(df[component_col].astype("string"), df["_severity_bucket"])
        conditional = counts.div(counts.sum(axis=1), axis=0)
        base = counts.sum(axis=0) / counts.values.sum()
        lift = conditional.div(base, axis=1).replace([np.inf, -np.inf], np.nan)
        write_csv(counts.reset_index(), out / "component_by_severity_counts.csv")
        write_csv(conditional.reset_index(), out / "component_by_severity_conditional_rate.csv")
        write_csv(lift.reset_index(), out / "component_by_severity_lift.csv")
        plot_heatmap(lift.fillna(0), f"{category}: severity lift by derived component", out / "component_severity_lift.png")
    if severity_col in df.columns:
        key = df.groupby("device_id").agg(
            incidents=("_row_id", "size"),
            critical_share=("_severity_bucket", lambda s: float((s == "CRITICAL").mean())),
            last_incident=("AE_START_DTM", "max"),
        ).sort_values(["critical_share", "incidents"], ascending=False)
        write_csv(key.reset_index(), out / "historical_device_hotspots.csv")
        if "FACILITY_ID" in df.columns:
            station = df.groupby("FACILITY_ID").agg(
                incidents=("_row_id", "size"),
                devices=("device_id", "nunique"),
                critical_share=("_severity_bucket", lambda s: float((s == "CRITICAL").mean())),
            ).sort_values(["critical_share", "incidents"], ascending=False)
            write_csv(station.reset_index(), out / "historical_facility_hotspots.csv")
    precursor_candidates = [c for c in BASE_NUMERIC_FEATURES + ["hist_device_prior_incidents_30d", "hist_facility_prior_incidents_30d"] if c in df.columns]
    if severity_col in df.columns and precursor_candidates:
        profile = df.groupby("_severity_bucket")[precursor_candidates].median(numeric_only=True)
        write_csv(profile.reset_index(), out / "severity_precursor_medians.csv")
        plot_heatmap(profile, f"{category}: median pre-incident precursor profile by severity", out / "severity_precursor_profile.png")
    if component_col in df.columns and precursor_candidates:
        profile = df.groupby(component_col)[precursor_candidates].median(numeric_only=True)
        write_csv(profile.reset_index(), out / "component_precursor_medians.csv")
    missing = pd.DataFrame({"column": df.columns, "missing_rate": [float(df[c].isna().mean()) for c in df.columns]})
    write_csv(missing.sort_values("missing_rate", ascending=False), out / "column_completeness.csv")
    write_json(summary, out / "discovery_summary.json")
    return summary

# -----------------------------------------------------------------------------
# Device × component reliability intelligence
# -----------------------------------------------------------------------------
# These outputs are historical reliability discoveries.  They intentionally do
# not turn incident counts into failure *rates* unless an exposure denominator
# is actually present.  The priority score is a transparent triage aid, not a
# probability that a device will fail tomorrow.

COMPONENT_UNKNOWN = "UNCLASSIFIED_COMPONENT"


def _clean_identifier(values, fallback):
    values = pd.Series(values, copy=False).astype("string").str.strip()
    missing = values.isna() | values.fillna("").str.lower().isin({"", "nan", "none", "<na>", "null"})
    return values.mask(missing, fallback).astype("string")


def normalise_component_label(values):
    """Keep source component names, while making missing labels explicit and groupable."""
    return _clean_identifier(values, COMPONENT_UNKNOWN).str.upper()


def _positive_percentile_rank(values):
    """Percentile rank only when a metric contains positive evidence; otherwise return zero."""
    series = pd.to_numeric(values, errors="coerce").fillna(0).clip(lower=0)
    if series.empty or float(series.max()) <= 0:
        return pd.Series(0.0, index=series.index)
    if series.nunique(dropna=True) <= 1:
        return pd.Series(0.5, index=series.index)
    return series.rank(method="average", pct=True).astype(float)


def _recency_percentile_rank(days_since):
    series = pd.to_numeric(days_since, errors="coerce")
    usable = series.dropna()
    if usable.empty:
        return pd.Series(0.0, index=series.index)
    if usable.nunique() <= 1:
        return pd.Series(0.5, index=series.index)
    ranks = 1.0 - usable.rank(method="average", pct=True)
    return ranks.reindex(series.index).fillna(0.0)


def wilson_lower_bound(successes, trials, z=1.96):
    """Conservative 95% lower confidence bound for a binomial share.

    This prevents a one-of-one critical incident from ranking above a component
    with a sustained, well-observed critical burden.
    """
    successes = pd.to_numeric(successes, errors="coerce").fillna(0).clip(lower=0)
    trials = pd.to_numeric(trials, errors="coerce").fillna(0).clip(lower=0)
    safe_trials = trials.where(trials > 0, np.nan)
    proportion = successes / safe_trials
    denominator = 1.0 + (z ** 2) / safe_trials
    centre = proportion + (z ** 2) / (2.0 * safe_trials)
    adjustment = z * np.sqrt((proportion * (1.0 - proportion) + (z ** 2) / (4.0 * safe_trials)) / safe_trials)
    return ((centre - adjustment) / denominator).clip(lower=0.0, upper=1.0)


def _component_evidence_tier(table):
    incidents = pd.to_numeric(table["historical_incidents"], errors="coerce").fillna(0)
    severity = pd.to_numeric(table["known_severity_events"], errors="coerce").fillna(0)
    return np.select(
        [(incidents >= 10) & (severity >= 5), incidents >= CONFIG["COMPONENT_MIN_EVIDENCE_EVENTS"]],
        ["strong", "moderate"],
        default="limited",
    )


def _priority_rationale(row):
    parts = []
    if pd.notna(row.get("critical_rate_wilson_lower_95")) and row["critical_rate_wilson_lower_95"] > 0:
        parts.append(f"conservative critical-share lower bound {row['critical_rate_wilson_lower_95']:.0%}")
    if pd.notna(row.get("repeat_within_window_rate")) and row["repeat_within_window_rate"] > 0:
        parts.append(f"{row['repeat_within_window_rate']:.0%} repeat intervals within {CONFIG['COMPONENT_RECURRENCE_WINDOW_DAYS']}d")
    if pd.notna(row.get("recent_incidents")) and row["recent_incidents"] > 0:
        parts.append(f"{int(row['recent_incidents'])} incidents in the latest {CONFIG['COMPONENT_RECENT_WINDOW_DAYS']}d window")
    if not parts:
        parts.append("incident volume and recency only; no observed severity or recurrence signal")
    return "; ".join(parts)


def _aggregate_component_level(history, group_columns, analysis_end, global_critical_rate):
    aggregations = {
        "historical_incidents": ("_component_event_id", "size"),
        "known_severity_events": ("_known_severity", "sum"),
        "critical_incidents": ("_critical", "sum"),
        "first_incident": ("_event_dtm", "min"),
        "last_incident": ("_event_dtm", "max"),
        "repeat_intervals_within_window": ("_repeat_within_window", "sum"),
        "comparable_repeat_intervals": ("_has_prior_interval", "sum"),
        "median_days_between_incidents": ("_days_since_prior", "median"),
        "recent_incidents": ("_recent_event", "sum"),
        "prior_window_incidents": ("_prior_window_event", "sum"),
    }
    if "_device_id" not in group_columns:
        aggregations["affected_devices"] = ("_device_id", "nunique")
    if "_serial_number" not in group_columns:
        aggregations["affected_serial_numbers"] = ("_serial_number", lambda s: int(s.loc[s.ne(UNKNOWN_SERIAL)].nunique()))
    if "_facility_id" in history.columns:
        if "_facility_id" not in group_columns:
            aggregations["affected_facilities"] = ("_facility_id", lambda s: int(s.loc[s.ne("")].nunique()))
        aggregations["dominant_facility_id"] = ("_facility_id", stable_mode)
    if "_facility_name" in history.columns:
        aggregations["dominant_facility_name"] = ("_facility_name", stable_mode)
    if "_serial_status" in history.columns and "_serial_number" not in group_columns:
        aggregations["serial_number_coverage"] = ("_serial_status", lambda s: float((s == "matched").mean()))

    table = history.groupby(group_columns, dropna=False).agg(**aggregations).reset_index()
    table["critical_share_observed"] = table["critical_incidents"] / table["known_severity_events"].replace(0, np.nan)
    prior_strength = float(CONFIG["COMPONENT_PRIOR_STRENGTH"])
    table["critical_rate_empirical_bayes"] = np.where(
        table["known_severity_events"].gt(0),
        (table["critical_incidents"] + global_critical_rate * prior_strength) /
        (table["known_severity_events"] + prior_strength),
        np.nan,
    )
    table["critical_rate_wilson_lower_95"] = wilson_lower_bound(
        table["critical_incidents"], table["known_severity_events"]
    )
    table["repeat_within_window_rate"] = (
        table["repeat_intervals_within_window"] / table["comparable_repeat_intervals"].replace(0, np.nan)
    )
    table["recent_vs_prior_event_ratio"] = table["recent_incidents"] / table["prior_window_incidents"].clip(lower=1)
    table["emerging_recent_pattern"] = (
        table["recent_incidents"].ge(CONFIG["COMPONENT_MIN_EVIDENCE_EVENTS"])
        & table["prior_window_incidents"].eq(0)
    )
    if pd.notna(analysis_end):
        table["days_since_last_incident"] = (analysis_end - pd.to_datetime(table["last_incident"], errors="coerce")).dt.total_seconds() / 86400.0
    else:
        table["days_since_last_incident"] = np.nan

    weights = CONFIG["COMPONENT_PRIORITY_WEIGHTS"]
    table["incident_volume_percentile"] = _positive_percentile_rank(np.log1p(table["historical_incidents"]))
    table["criticality_percentile"] = _positive_percentile_rank(table["critical_rate_wilson_lower_95"])
    table["recurrence_percentile"] = _positive_percentile_rank(table["repeat_within_window_rate"])
    table["recent_activity_percentile"] = _positive_percentile_rank(table["recent_incidents"])
    table["recency_percentile"] = _recency_percentile_rank(table["days_since_last_incident"])
    raw_score = (
        weights["incident_volume"] * table["incident_volume_percentile"]
        + weights["criticality"] * table["criticality_percentile"]
        + weights["recurrence"] * table["recurrence_percentile"]
        + weights["recent_activity"] * table["recent_activity_percentile"]
        + weights["recency"] * table["recency_percentile"]
    )
    evidence_weight = np.clip(
        np.log1p(table["historical_incidents"]) / np.log1p(max(10, CONFIG["COMPONENT_MIN_EVIDENCE_EVENTS"] * 2)),
        0.0,
        1.0,
    )
    table["evidence_weight"] = evidence_weight
    table["reliability_priority_score"] = 100.0 * raw_score * (0.55 + 0.45 * evidence_weight)
    table["reliability_priority_band"] = pd.cut(
        table["reliability_priority_score"],
        bins=[-0.01, 35, 60, 100.01],
        labels=["monitor", "plan", "urgent"],
        include_lowest=True,
    ).astype("string")
    table["evidence_tier"] = _component_evidence_tier(table)
    table["priority_rationale"] = table.apply(_priority_rationale, axis=1)
    table["priority_score_scope"] = "historical triage score; not a future failure probability"
    rename = {"_device_id": "device_id", "_serial_number": "serial_number", "_component": "component"}
    return table.rename(columns=rename)


def _component_concentration(history):
    by_device = history.groupby(["_component", "_device_id"], dropna=False).size().rename("incidents").reset_index()
    if by_device.empty:
        return pd.DataFrame(columns=["component", "device_incident_hhi", "top_device_id", "top_device_incident_share", "device_concentration_pattern"])
    by_device["component_incidents"] = by_device.groupby("_component")["incidents"].transform("sum")
    by_device["device_incident_share"] = by_device["incidents"] / by_device["component_incidents"].replace(0, np.nan)
    hhi = by_device.groupby("_component")["device_incident_share"].apply(lambda x: float((x ** 2).sum())).rename("device_incident_hhi")
    leaders = by_device.sort_values(["_component", "incidents", "_device_id"], ascending=[True, False, True], kind="mergesort").drop_duplicates("_component")
    out = hhi.reset_index().merge(
        leaders[["_component", "_device_id", "device_incident_share"]], on="_component", how="left"
    ).rename(columns={"_component": "component", "_device_id": "top_device_id", "device_incident_share": "top_device_incident_share"})
    out["device_concentration_pattern"] = np.select(
        [out["device_incident_hhi"].ge(0.50), out["device_incident_hhi"].ge(0.20)],
        ["highly_concentrated", "moderately_concentrated"],
        default="distributed",
    )
    return out


def _write_component_preincident_signatures(history, portfolio, out, category):
    candidates = [
        col for col in history.columns
        if (
            col in BASE_NUMERIC_FEATURES or col.startswith("hist_") or col.startswith("enr_")
        ) and pd.api.types.is_numeric_dtype(history[col]) and history[col].notna().sum() >= 5
    ]
    if not candidates:
        write_csv(pd.DataFrame(columns=["component", "feature", "events_with_feature", "coverage", "component_median", "global_median", "median_delta", "robust_median_z"]), out / "component_preincident_signal_lift.csv")
        return pd.DataFrame()
    rows = []
    for feature in candidates:
        population = pd.to_numeric(history[feature], errors="coerce").dropna()
        global_median = float(population.median()) if not population.empty else np.nan
        mad = float(np.median(np.abs(population - global_median))) if not population.empty else np.nan
        mad_scale = 1.4826 * mad if pd.notna(mad) else np.nan
        for component, group in history.groupby("_component", dropna=False):
            values = pd.to_numeric(group[feature], errors="coerce").dropna()
            median = float(values.median()) if not values.empty else np.nan
            robust_z = (median - global_median) / mad_scale if pd.notna(median) and pd.notna(mad_scale) and mad_scale > 0 else np.nan
            rows.append({
                "component": component,
                "feature": feature,
                "events_with_feature": int(len(values)),
                "coverage": float(len(values) / max(1, len(group))),
                "component_median": median,
                "global_median": global_median,
                "median_delta": median - global_median if pd.notna(median) and pd.notna(global_median) else np.nan,
                "robust_median_z": robust_z,
                "interpretation": "pre-incident median contrast only; descriptive, not causal",
            })
    signals = pd.DataFrame(rows)
    signals["abs_robust_median_z"] = signals["robust_median_z"].abs()
    signals = signals.sort_values(["abs_robust_median_z", "events_with_feature"], ascending=[False, False], na_position="last")
    write_csv(signals, out / "component_preincident_signal_lift.csv")
    if not signals.empty:
        top_components = portfolio.sort_values("reliability_priority_score", ascending=False).head(12)["component"].astype(str).tolist()
        top_features = signals.groupby("feature")["abs_robust_median_z"].max().sort_values(ascending=False).head(12).index.tolist()
        matrix = signals[signals["component"].astype(str).isin(top_components) & signals["feature"].isin(top_features)].pivot(
            index="component", columns="feature", values="robust_median_z"
        )
        if not matrix.empty:
            plot_heatmap(matrix.fillna(0), f"{category}: robust pre-incident signal contrast by component", out / "component_preincident_signal_lift.png")
    return signals


def _write_component_temporal_stability(history, out, category):
    columns = ["component", "train_incidents", "test_incidents", "train_share", "test_share", "share_delta", "psi_contribution"]
    if "_split" not in history.columns or not {"train", "test"}.issubset(set(history["_split"].dropna().astype(str))):
        write_csv(pd.DataFrame(columns=columns), out / "component_temporal_mix_shift.csv")
        status = {"status": "not_available", "reason": "train/test temporal partitions were not both available"}
        write_json(status, out / "component_temporal_mix_shift.json")
        return status
    train = history.loc[history["_split"].astype(str) == "train", "_component"].value_counts()
    test = history.loc[history["_split"].astype(str) == "test", "_component"].value_counts()
    components = sorted(set(train.index.astype(str)) | set(test.index.astype(str)))
    shift = pd.DataFrame({"component": components})
    shift["train_incidents"] = shift["component"].map(train).fillna(0).astype(int)
    shift["test_incidents"] = shift["component"].map(test).fillna(0).astype(int)
    epsilon = 1e-6
    n_components = max(1, len(shift))
    shift["train_share"] = (shift["train_incidents"] + epsilon) / (shift["train_incidents"].sum() + epsilon * n_components)
    shift["test_share"] = (shift["test_incidents"] + epsilon) / (shift["test_incidents"].sum() + epsilon * n_components)
    shift["share_delta"] = shift["test_share"] - shift["train_share"]
    shift["psi_contribution"] = (shift["test_share"] - shift["train_share"]) * np.log(shift["test_share"] / shift["train_share"])
    shift = shift.sort_values("psi_contribution", ascending=False)
    write_csv(shift, out / "component_temporal_mix_shift.csv")
    psi = float(shift["psi_contribution"].sum())
    status = "low" if psi < 0.10 else "moderate" if psi < 0.25 else "high"
    summary = {
        "status": "available",
        "population_stability_index": psi,
        "heuristic_shift_band": status,
        "interpretation": "Component mix shift between training history and frozen test. It is a transferability monitor, not a root cause.",
    }
    write_json(summary, out / "component_temporal_mix_shift.json")
    top = shift.head(15).sort_values("share_delta")
    if not top.empty:
        figure, axis = plt.subplots(figsize=(10, max(4.5, 0.32 * len(top) + 1)))
        colors = np.where(top["share_delta"].ge(0), "#e41a1c", "#377eb8")
        axis.barh(top["component"].astype(str), top["share_delta"], color=colors)
        axis.axvline(0, color="#555555", linewidth=0.8)
        axis.set_title(f"{category}: component mix shift, frozen test vs training")
        axis.set_xlabel("Test share minus training share")
        plot_or_close(out / "component_temporal_mix_shift.png")
    return summary


def _write_component_visuals(portfolio, device_component, monthly, out, category):
    if portfolio.empty:
        return
    color_map = {"monitor": "#4daf4a", "plan": "#ff7f00", "urgent": "#e41a1c"}
    top_components = portfolio.sort_values("reliability_priority_score", ascending=False).head(CONFIG["COMPONENT_PRIORITY_TOP_N"]).sort_values("reliability_priority_score")
    figure, axis = plt.subplots(figsize=(11, max(4.5, 0.34 * len(top_components) + 1)))
    axis.barh(top_components["component"].astype(str), top_components["reliability_priority_score"], color=[color_map.get(v, "#777777") for v in top_components["reliability_priority_band"]])
    axis.set_title(f"{category}: component reliability triage priority")
    axis.set_xlabel("Evidence-adjusted historical priority score (0–100)")
    plot_or_close(out / "component_reliability_priority.png")

    figure, axis = plt.subplots(figsize=(10, 6.5))
    size = 30 + 16 * np.sqrt(portfolio["historical_incidents"].clip(lower=0))
    axis.scatter(
        portfolio["repeat_within_window_rate"].fillna(0),
        portfolio["critical_rate_wilson_lower_95"].fillna(0),
        s=size, alpha=0.65, color="#984ea3",
    )
    for _, row in portfolio.sort_values("reliability_priority_score", ascending=False).head(12).iterrows():
        axis.annotate(str(row["component"]), (row["repeat_within_window_rate"] if pd.notna(row["repeat_within_window_rate"]) else 0, row["critical_rate_wilson_lower_95"] if pd.notna(row["critical_rate_wilson_lower_95"]) else 0), fontsize=8)
    axis.set_title(f"{category}: component recurrence vs conservative critical share")
    axis.set_xlabel(f"Repeat interval share within {CONFIG['COMPONENT_RECURRENCE_WINDOW_DAYS']} days")
    axis.set_ylabel("95% lower bound of observed critical share")
    plot_or_close(out / "component_recurrence_criticality_matrix.png")

    top_device_component = device_component.sort_values("reliability_priority_score", ascending=False).head(CONFIG["COMPONENT_PRIORITY_TOP_N"]).sort_values("reliability_priority_score")
    if not top_device_component.empty:
        labels = (top_device_component["device_id"].astype(str) + " | " + top_device_component["component"].astype(str)).tolist()
        figure, axis = plt.subplots(figsize=(12, max(4.5, 0.33 * len(top_device_component) + 1)))
        axis.barh(labels, top_device_component["reliability_priority_score"], color=[color_map.get(v, "#777777") for v in top_device_component["reliability_priority_band"]])
        axis.set_title(f"{category}: priority device × component pairs")
        axis.set_xlabel("Evidence-adjusted historical priority score (0–100)")
        plot_or_close(out / "top_device_component_priority.png")

    if not device_component.empty:
        top_devices = device_component.groupby("device_id")["reliability_priority_score"].max().sort_values(ascending=False).head(20).index
        top_component_names = portfolio.sort_values("reliability_priority_score", ascending=False).head(12)["component"].tolist()
        matrix = device_component[
            device_component["device_id"].isin(top_devices) & device_component["component"].isin(top_component_names)
        ].pivot_table(index="device_id", columns="component", values="reliability_priority_score", aggfunc="max", fill_value=0)
        if not matrix.empty:
            plot_heatmap(matrix, f"{category}: device × component triage priority", out / "device_component_priority_heatmap.png")

    if monthly is not None and not monthly.empty:
        selected = portfolio.sort_values("reliability_priority_score", ascending=False).head(15)["component"].tolist()
        matrix = monthly[monthly["component"].isin(selected)].pivot(index="component", columns="month", values="incidents").fillna(0)
        if not matrix.empty:
            plot_heatmap(matrix, f"{category}: monthly incident volume by priority component", out / "component_monthly_incident_heatmap.png")

    # A compact, dashboard-ready overview without hiding the detailed charts above.
    figure, axes = plt.subplots(2, 2, figsize=(15, 10))
    portfolio_top = portfolio.sort_values("historical_incidents", ascending=False).head(12).sort_values("historical_incidents")
    axes[0, 0].barh(portfolio_top["component"].astype(str), portfolio_top["historical_incidents"], color="#377eb8")
    axes[0, 0].set_title("Historical component incident Pareto")
    axes[0, 0].set_xlabel("Incidents")
    axes[0, 1].scatter(portfolio["repeat_within_window_rate"].fillna(0), portfolio["critical_rate_wilson_lower_95"].fillna(0), s=size, color="#984ea3", alpha=0.65)
    axes[0, 1].set_title("Recurrence vs conservative critical share")
    axes[0, 1].set_xlabel("Repeat interval share")
    axes[0, 1].set_ylabel("Critical-share lower bound")
    if monthly is not None and not monthly.empty:
        monthly_top = monthly[monthly["component"].isin(portfolio.sort_values("reliability_priority_score", ascending=False).head(5)["component"])].pivot(index="month", columns="component", values="incidents").fillna(0)
        if not monthly_top.empty:
            monthly_top.plot(ax=axes[1, 0], marker="o", ms=2)
            axes[1, 0].set_title("Top priority component incident trend")
            axes[1, 0].tick_params(axis="x", rotation=45)
    else:
        axes[1, 0].set_axis_off()
    if not top_device_component.empty:
        axes[1, 1].barh((top_device_component["device_id"].astype(str) + " | " + top_device_component["component"].astype(str)).head(12)[::-1], top_device_component["reliability_priority_score"].head(12)[::-1], color="#ff7f00")
        axes[1, 1].set_title("Top device × component triage pairs")
        axes[1, 1].set_xlabel("Priority score")
    else:
        axes[1, 1].set_axis_off()
    plot_or_close(out / "component_reliability_dashboard.png")


def write_component_reliability_intelligence(
    data,
    category,
    category_out,
    *,
    date_col="AE_START_DTM",
    component_col=None,
    severity_col=None,
    device_col="device_id",
    serial_col="matched_serial_nbr",
    severity_is_bucket=False,
    output_subdir="component_intelligence",
    scope="descriptive all-available incident history; not a future score",
):
    """Create robust device × component discovery outputs from historical incident records."""
    out = Path(category_out) / output_subdir
    out.mkdir(parents=True, exist_ok=True)
    component_col = component_col or CONFIG["ROOTCAUSE_TARGET"]
    severity_col = severity_col or CONFIG["SEVERITY_TARGET"]
    history = data.copy()
    history["_component_event_id"] = np.arange(len(history), dtype=int)
    raw_component = history[component_col] if component_col in history.columns else pd.Series(pd.NA, index=history.index, dtype="string")
    raw_component_string = raw_component.astype("string").str.strip()
    component_observed = raw_component_string.notna() & ~raw_component_string.fillna("").str.lower().isin({"", "nan", "none", "<na>", "null"})
    history["_component"] = normalise_component_label(raw_component)
    history["_device_id"] = _clean_identifier(history[device_col] if device_col in history.columns else pd.Series(pd.NA, index=history.index), UNKNOWN_DEVICE)
    history["_serial_number"] = _clean_identifier(history[serial_col] if serial_col in history.columns else pd.Series(pd.NA, index=history.index), UNKNOWN_SERIAL)
    history["_serial_status"] = np.where(history["_serial_number"].eq(UNKNOWN_SERIAL), "missing_in_source", "matched")
    history["_event_dtm"] = pd.to_datetime(history[date_col], errors="coerce") if date_col in history.columns else pd.NaT
    if "FACILITY_ID" in history.columns:
        history["_facility_id"] = _clean_identifier(history["FACILITY_ID"], "")
    if "FACILITY_NAME" in history.columns:
        history["_facility_name"] = _clean_identifier(history["FACILITY_NAME"], "")

    if severity_col in history.columns:
        raw_severity = history[severity_col].astype("string").str.strip()
        history["_severity_bucket"] = raw_severity.str.upper() if severity_is_bucket else severity_bucket(raw_severity)
        history["_known_severity"] = raw_severity.notna() & ~raw_severity.fillna("").str.upper().isin({"", "NAN", "NONE", "NULL", "UNKNOWN"})
        history["_critical"] = history["_known_severity"] & history["_severity_bucket"].eq("CRITICAL")
    else:
        history["_severity_bucket"] = "UNKNOWN"
        history["_known_severity"] = False
        history["_critical"] = False

    quality = pd.DataFrame([{
        "device_category": category,
        "source_scope": scope,
        "input_rows": int(len(history)),
        "unique_devices": int(history["_device_id"].nunique()),
        "matched_serial_coverage": float((history["_serial_number"] != UNKNOWN_SERIAL).mean()) if len(history) else np.nan,
        "component_label_coverage": float(component_observed.mean()) if len(history) else np.nan,
        "known_severity_coverage": float(history["_known_severity"].mean()) if len(history) else np.nan,
        "valid_event_time_coverage": float(history["_event_dtm"].notna().mean()) if len(history) else np.nan,
        "component_source_column": component_col,
        "severity_source_column": severity_col,
    }])
    write_csv(quality, out / "component_data_quality.csv")

    if history.empty or not component_observed.any():
        summary = {
            "status": "component_label_unavailable",
            "device_category": category,
            "source_scope": scope,
            "message": "No usable component/root-cause labels were available; the notebook did not infer or fabricate component findings.",
            "component_label_coverage": float(component_observed.mean()) if len(history) else 0.0,
        }
        write_csv(pd.DataFrame(columns=["component", "historical_incidents"]), out / "component_reliability_portfolio.csv")
        write_csv(pd.DataFrame(columns=["device_id", "component", "historical_incidents"]), out / "device_component_reliability_ranking.csv")
        write_csv(pd.DataFrame(columns=["device_id", "serial_number", "component", "historical_incidents"]), out / "device_serial_component_reliability_ranking.csv")
        write_json(summary, out / "reliability_intelligence_summary.json")
        return summary

    analysis_end = history["_event_dtm"].max()
    if pd.notna(analysis_end):
        recent_start = analysis_end - pd.Timedelta(days=CONFIG["COMPONENT_RECENT_WINDOW_DAYS"])
        prior_start = recent_start - pd.Timedelta(days=CONFIG["COMPONENT_RECENT_WINDOW_DAYS"])
        history["_recent_event"] = history["_event_dtm"].ge(recent_start)
        history["_prior_window_event"] = history["_event_dtm"].ge(prior_start) & history["_event_dtm"].lt(recent_start)
    else:
        history["_recent_event"] = False
        history["_prior_window_event"] = False
    recurrence_keys = ["_device_id", "_serial_number", "_component"]
    ordered = history.sort_values(recurrence_keys + ["_event_dtm", "_component_event_id"], kind="mergesort").copy()
    ordered["_days_since_prior"] = ordered.groupby(recurrence_keys, dropna=False)["_event_dtm"].diff().dt.total_seconds() / 86400.0
    ordered["_has_prior_interval"] = ordered["_days_since_prior"].notna() & ordered["_days_since_prior"].ge(0)
    ordered["_repeat_within_window"] = ordered["_has_prior_interval"] & ordered["_days_since_prior"].le(CONFIG["COMPONENT_RECURRENCE_WINDOW_DAYS"])
    history = history.merge(
        ordered[["_component_event_id", "_days_since_prior", "_has_prior_interval", "_repeat_within_window"]],
        on="_component_event_id", how="left", validate="1:1",
    )
    history["_has_prior_interval"] = history["_has_prior_interval"].fillna(False)
    history["_repeat_within_window"] = history["_repeat_within_window"].fillna(False)
    global_critical_rate = float(history.loc[history["_known_severity"], "_critical"].mean()) if history["_known_severity"].any() else 0.0

    portfolio = _aggregate_component_level(history, ["_component"], analysis_end, global_critical_rate)
    concentration = _component_concentration(history)
    portfolio = portfolio.merge(concentration, on="component", how="left", validate="1:1")
    portfolio = portfolio.sort_values(["reliability_priority_score", "historical_incidents"], ascending=False, kind="mergesort")
    portfolio["incident_share"] = portfolio["historical_incidents"] / max(1, portfolio["historical_incidents"].sum())
    portfolio["incident_pareto_cumulative_share"] = portfolio["incident_share"].cumsum()
    portfolio["in_80_percent_incident_pareto"] = portfolio["incident_pareto_cumulative_share"].le(0.80)
    portfolio["device_category"] = category
    portfolio["analysis_scope"] = scope
    write_csv(portfolio, out / "component_reliability_portfolio.csv")

    device_component = _aggregate_component_level(history, ["_device_id", "_component"], analysis_end, global_critical_rate)
    device_component["device_category"] = category
    device_component["analysis_scope"] = scope
    device_component = device_component.sort_values(["reliability_priority_score", "historical_incidents"], ascending=False, kind="mergesort")
    write_csv(device_component, out / "device_component_reliability_ranking.csv")

    device_serial_component = _aggregate_component_level(history, ["_device_id", "_serial_number", "_component"], analysis_end, global_critical_rate)
    device_serial_component["device_category"] = category
    device_serial_component["serial_number_status"] = np.where(device_serial_component["serial_number"].eq(UNKNOWN_SERIAL), "missing_in_source", "matched")
    device_serial_component["analysis_scope"] = scope
    device_serial_component = device_serial_component.sort_values(["reliability_priority_score", "historical_incidents"], ascending=False, kind="mergesort")
    write_csv(device_serial_component, out / "device_serial_component_reliability_ranking.csv")

    monthly = pd.DataFrame(columns=["month", "component", "incidents"])
    if history["_event_dtm"].notna().any():
        monthly = history.assign(month=history["_event_dtm"].dt.to_period("M").astype(str)).groupby(["month", "_component"], dropna=False).size().rename("incidents").reset_index().rename(columns={"_component": "component"})
        write_csv(monthly, out / "component_monthly_incidents.csv")
    else:
        write_csv(monthly, out / "component_monthly_incidents.csv")
    signals = _write_component_preincident_signatures(history, portfolio, out, category)
    drift = _write_component_temporal_stability(history, out, category)
    _write_component_visuals(portfolio, device_component, monthly, out, category)

    top_component = portfolio.iloc[0].to_dict() if not portfolio.empty else None
    top_device_component = device_component.iloc[0].to_dict() if not device_component.empty else None
    summary = {
        "status": "available",
        "device_category": category,
        "source_scope": scope,
        "analysis_end": str(analysis_end),
        "component_count": int(portfolio["component"].nunique()),
        "device_component_rows": int(len(device_component)),
        "device_serial_component_rows": int(len(device_serial_component)),
        "global_observed_critical_share": global_critical_rate if history["_known_severity"].any() else None,
        "top_component_by_priority": top_component,
        "top_device_component_by_priority": top_device_component,
        "temporal_component_mix_shift": drift,
        "preincident_signal_rows": int(len(signals)),
        "methods": {
            "criticality": "Observed critical share with empirical-Bayes smoothing and a 95% Wilson lower confidence bound.",
            "recurrence": f"Consecutive incidents on the same device + serial + component within {CONFIG['COMPONENT_RECURRENCE_WINDOW_DAYS']} days.",
            "priority": "Evidence-adjusted composite of volume, conservative criticality, recurrence, latest-window activity, and recency.",
            "precursors": "MAD-standardised pre-incident median contrasts; descriptive only, not causal attribution.",
        },
    }
    write_json(summary, out / "reliability_intelligence_summary.json")
    return summary


def write_component_prediction_reliability(incidents, category, category_out):
    """Frozen-test quality broken down by the actual component and device × serial × component."""
    out = Path(category_out) / "component_intelligence"
    out.mkdir(parents=True, exist_ok=True)
    if incidents is None or incidents.empty:
        status = {"status": "not_available", "reason": "no frozen-test prediction rows"}
        write_json(status, out / "component_prediction_reliability_summary.json")
        return status
    work = incidents.copy()
    raw_component = work[CONFIG["ROOTCAUSE_TARGET"]] if CONFIG["ROOTCAUSE_TARGET"] in work.columns else work.get("actual_root_cause", pd.Series(pd.NA, index=work.index))
    work["actual_component"] = normalise_component_label(raw_component)
    work["_root_evaluable"] = work.get("root_cause_correct", pd.Series(pd.NA, index=work.index)).notna()
    work["_root_correct_num"] = pd.to_numeric(work.get("root_cause_correct", pd.Series(np.nan, index=work.index)), errors="coerce").astype("float64").fillna(0.0)
    work["_severity_evaluable"] = work.get("severity_correct", pd.Series(pd.NA, index=work.index)).notna()
    work["_severity_correct_num"] = pd.to_numeric(work.get("severity_correct", pd.Series(np.nan, index=work.index)), errors="coerce").astype("float64").fillna(0.0)
    actual_severity = work.get("actual_severity", work.get(CONFIG["SEVERITY_TARGET"], pd.Series(pd.NA, index=work.index)))
    work["_actual_critical"] = severity_bucket(pd.Series(actual_severity, index=work.index)).eq("CRITICAL") if actual_severity is not None else False
    critical_probability = pd.to_numeric(work.get("severity_critical_probability", pd.Series(np.nan, index=work.index)), errors="coerce")
    work["_predicted_critical"] = critical_probability.ge(0.50)
    work["_critical_detected"] = work["_actual_critical"] & work["_predicted_critical"]

    def summarise(frame, group_columns):
        result = frame.groupby(group_columns, dropna=False).agg(
            frozen_test_incidents=("_row_id", "size"),
            root_cause_evaluable_predictions=("_root_evaluable", "sum"),
            root_cause_correct_predictions=("_root_correct_num", "sum"),
            severity_evaluable_predictions=("_severity_evaluable", "sum"),
            severity_correct_predictions=("_severity_correct_num", "sum"),
            actual_critical_test_incidents=("_actual_critical", "sum"),
            detected_critical_test_incidents=("_critical_detected", "sum"),
            mean_root_cause_confidence=("root_cause_confidence", "mean") if "root_cause_confidence" in frame.columns else ("_row_id", "size"),
            mean_severity_critical_probability=("severity_critical_probability", "mean") if "severity_critical_probability" in frame.columns else ("_row_id", "size"),
        ).reset_index()
        result["root_cause_accuracy_on_evaluable"] = result["root_cause_correct_predictions"] / result["root_cause_evaluable_predictions"].replace(0, np.nan)
        result["root_cause_accuracy_wilson_lower_95"] = wilson_lower_bound(result["root_cause_correct_predictions"], result["root_cause_evaluable_predictions"])
        result["severity_accuracy_on_evaluable"] = result["severity_correct_predictions"] / result["severity_evaluable_predictions"].replace(0, np.nan)
        result["severity_accuracy_wilson_lower_95"] = wilson_lower_bound(result["severity_correct_predictions"], result["severity_evaluable_predictions"])
        result["critical_recall_at_0_50"] = result["detected_critical_test_incidents"] / result["actual_critical_test_incidents"].replace(0, np.nan)
        result["prediction_evidence_tier"] = np.select(
            [result["frozen_test_incidents"].ge(20), result["frozen_test_incidents"].ge(5)],
            ["strong", "moderate"], default="limited",
        )
        return result

    component_quality = summarise(work, ["actual_component"])
    component_quality["device_category"] = category
    component_quality["prediction_scope"] = "frozen_temporal_test_holdout; performance is diagnostic, not a future score"
    component_quality = component_quality.sort_values(["root_cause_accuracy_wilson_lower_95", "frozen_test_incidents"], ascending=[True, False], na_position="last")
    write_csv(component_quality, out / "component_prediction_reliability.csv")

    device_serial_quality = summarise(work, ["device_id", "serial_number", "actual_component"])
    device_serial_quality["device_category"] = category
    device_serial_quality["serial_number_status"] = np.where(device_serial_quality["serial_number"].eq(UNKNOWN_SERIAL), "missing_in_source", "matched")
    device_serial_quality["prediction_scope"] = "frozen_temporal_test_holdout; performance is diagnostic, not a future score"
    device_serial_quality = device_serial_quality.sort_values(["root_cause_accuracy_wilson_lower_95", "frozen_test_incidents"], ascending=[True, False], na_position="last")
    write_csv(device_serial_quality, out / "test_device_serial_component_prediction_quality.csv")

    plottable = component_quality.dropna(subset=["root_cause_accuracy_on_evaluable"]).sort_values("frozen_test_incidents", ascending=False).head(15).sort_values("root_cause_accuracy_on_evaluable")
    if not plottable.empty:
        figure, axis = plt.subplots(figsize=(10, max(4.5, 0.34 * len(plottable) + 1)))
        axis.barh(plottable["actual_component"].astype(str), plottable["root_cause_accuracy_on_evaluable"], color="#377eb8")
        axis.errorbar(
            plottable["root_cause_accuracy_on_evaluable"],
            plottable["actual_component"].astype(str),
            xerr=(plottable["root_cause_accuracy_on_evaluable"] - plottable["root_cause_accuracy_wilson_lower_95"]).clip(lower=0),
            fmt="none", color="#333333", capsize=3,
        )
        axis.set_xlim(0, 1)
        axis.set_title(f"{category}: root-cause test reliability by actual component")
        axis.set_xlabel("Accuracy on evaluable frozen-test predictions (error bar to 95% lower bound)")
        plot_or_close(out / "component_root_cause_test_reliability.png")
    summary = {
        "status": "available",
        "device_category": category,
        "component_rows": int(len(component_quality)),
        "device_serial_component_rows": int(len(device_serial_quality)),
        "root_cause_evaluable_predictions": int(component_quality["root_cause_evaluable_predictions"].sum()),
        "severity_evaluable_predictions": int(component_quality["severity_evaluable_predictions"].sum()),
        "interpretation": "Component quality uses frozen temporal test predictions. The model target may contain OTHER where a class was too rare in training; use actual_component alongside the modelled target outputs.",
    }
    write_json(summary, out / "component_prediction_reliability_summary.json")
    return summary


## 10. Head training — validation-selected champion, frozen test diagnostics, and promotion gates


In [ ]:
def _head_output_dir(category_out, head_name):
    directory = Path(category_out) / head_name
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def _encode_supported_targets(data):
    train_labels = data.loc[data["_split"] == "train", "_target"].astype(str)
    encoder = LabelEncoder().fit(train_labels)
    supported = data[data["_label_supported"]].copy()
    supported["_y"] = encoder.transform(supported["_target"].astype(str))
    return supported, encoder


def _prediction_rows(frame, probabilities, encoder, head_name):
    labels = [str(x) for x in encoder.classes_]
    predicted_idx = probabilities.argmax(axis=1)
    result = pd.DataFrame({"_row_id": frame["_row_id"].to_numpy()})
    result[f"actual_{head_name}"] = frame["_target"].astype(str).to_numpy()
    result[f"pred_{head_name}"] = encoder.inverse_transform(predicted_idx)
    result[f"{head_name}_confidence"] = probabilities.max(axis=1)
    result[f"{head_name}_entropy"] = probability_entropy(probabilities)
    for index, label in enumerate(labels[:12]):
        result[f"prob_{head_name}_{safe_name(label)}"] = probabilities[:, index]
    if head_name == "severity":
        critical_idx = [idx for idx, label in enumerate(labels) if severity_bucket(pd.Series([label])).iloc[0] == "CRITICAL"]
        result["severity_critical_probability"] = probabilities[:, critical_idx].sum(axis=1) if critical_idx else 0.0
        result["pred_severity_bucket"] = severity_bucket(result[f"pred_{head_name}"])
    return result


def evaluate_candidate(factory, name, train, validation, numeric_features, categorical_features, n_classes):
    features = numeric_features + categorical_features
    model, prep = fit_model(factory, train[features], train["_y"], numeric_features, categorical_features)
    train_probability = aligned_probabilities(model, prep.transform(train[features]), n_classes)
    valid_probability = aligned_probabilities(model, prep.transform(validation[features]), n_classes)
    return {
        "name": name,
        "factory": factory,
        "model": model,
        "prep": prep,
        "train_metrics": metric_bundle(train["_y"], train_probability, np.arange(n_classes)),
        "validation_metrics": metric_bundle(validation["_y"], valid_probability, np.arange(n_classes)),
        "validation_probability": valid_probability,
    }


def promotion_gate(head_name, champion_row, baseline_row, target_meta):
    floor = CONFIG["SEVERITY_F1_FLOOR"] if head_name == "severity" else CONFIG["ROOTCAUSE_F1_FLOOR"]
    reasons = []
    if champion_row["validation_f1_macro"] < floor:
        reasons.append(f"validation macro-F1 {champion_row['validation_f1_macro']:.3f} < floor {floor:.3f}")
    delta = champion_row["validation_f1_macro"] - baseline_row["validation_f1_macro"]
    if delta < CONFIG["MIN_VALIDATION_DELTA_VS_BASELINE"]:
        reasons.append(f"validation delta vs baseline {delta:.3f} is below {CONFIG['MIN_VALIDATION_DELTA_VS_BASELINE']:.3f}")
    if champion_row["train_validation_f1_gap"] > CONFIG["MAX_TRAIN_VALIDATION_F1_GAP"]:
        reasons.append(f"train-validation F1 gap {champion_row['train_validation_f1_gap']:.3f} exceeds guardrail")
    if champion_row.get("test_validation_f1_drop", 0.0) > CONFIG["MAX_TEST_VALIDATION_F1_DROP"]:
        reasons.append(f"test-vs-validation F1 drop {champion_row['test_validation_f1_drop']:.3f} exceeds guardrail")
    if champion_row.get("test_ece", np.inf) > CONFIG["MAX_TEST_ECE"]:
        reasons.append(f"test ECE {champion_row['test_ece']:.3f} exceeds guardrail")
    unseen_test = target_meta.get("unseen_labels_by_split", {}).get("test", {}).get("unseen_label_rate", 0.0)
    if unseen_test > CONFIG["MAX_UNSEEN_TEST_LABEL_RATE"]:
        reasons.append(f"unseen test label rate {unseen_test:.1%} exceeds guardrail")
    return {"gate_pass": not reasons, "reasons": reasons, "macro_f1_floor": floor, "validation_delta_vs_baseline": float(delta)}


def fit_final_serving_bundle(factory, supported, numeric_features, categorical_features, encoder, head_name, category, target_col, output_dir):
    """Fit after the test report is frozen. This is the only model intended for future scoring."""
    trainable = supported.copy()
    features = numeric_features + categorical_features
    model, prep = fit_model(factory, trainable[features], trainable["_y"], numeric_features, categorical_features)
    try:
        import joblib
        joblib.dump(
            {
                "model": model,
                "preprocessor": prep,
                "numeric_features": numeric_features,
                "categorical_features": categorical_features,
                "features": features,
                "label_encoder": encoder,
                "head": head_name,
                "device_category": category,
                "target": target_col,
                "feature_time_contract": {feature: feature_availability(feature) for feature in features},
                "training_note": "Refit after frozen temporal test evaluation; use only for future events.",
            },
            Path(output_dir) / "serving_bundle.joblib",
        )
    except Exception as exc:
        log(f"serving bundle was not written: {type(exc).__name__}: {str(exc)[:160]}")
    return model, prep


def train_head(category_df, category, head_name, target_col, category_out):
    output_dir = _head_output_dir(category_out, head_name)
    prepared, target_meta = prepare_head_target(category_df, target_col)
    if prepared is None:
        write_json({"head": head_name, "status": "skipped", **target_meta}, output_dir / "head_status.json")
        log(f"{category} / {head_name}: skipped — {target_meta.get('reason')}")
        return None
    supported, encoder = _encode_supported_targets(prepared)
    train = supported[supported["_split"] == "train"].copy()
    validation = supported[supported["_split"] == "validation"].copy()
    test = supported[supported["_split"] == "test"].copy()
    if len(train) < 50 or len(validation) < CONFIG["MIN_EVAL_ROWS"] or len(test) < CONFIG["MIN_EVAL_ROWS"]:
        reason = f"partition too small: train={len(train)}, validation={len(validation)}, test={len(test)}"
        write_json({"head": head_name, "status": "skipped", "reason": reason, **target_meta}, output_dir / "head_status.json")
        return None
    n_classes = len(encoder.classes_)
    if n_classes < 2:
        write_json({"head": head_name, "status": "skipped", "reason": "fewer than two train classes", **target_meta}, output_dir / "head_status.json")
        return None

    numeric_features, categorical_features, manifest = build_feature_spec(prepared, target_col)
    features = numeric_features + categorical_features
    if not features:
        write_json({"head": head_name, "status": "skipped", "reason": "no approved features", **target_meta}, output_dir / "head_status.json")
        return None
    audit, suspicious = run_feature_audit(train, validation, features, manifest)
    if CONFIG["DROP_SUSPICIOUS_FEATURES"]:
        features = [feature for feature in features if feature not in suspicious]
        numeric_features = [feature for feature in numeric_features if feature in features]
        categorical_features = [feature for feature in categorical_features if feature in features]
    write_csv(audit, output_dir / "feature_contract_and_leakage_audit.csv")
    write_json(
        {
            "selected_features": features,
            "numeric_features": numeric_features,
            "categorical_features": categorical_features,
            "suspicious_single_feature_drivers": suspicious,
            "incident_time_metadata_allowed": CONFIG["ALLOW_INCIDENT_TIME_METADATA"],
        },
        output_dir / "feature_manifest.json",
    )
    if not features:
        return None

    builders = model_builders(n_classes)
    tuning = tune_xgboost(train, numeric_features, categorical_features, n_classes)
    if tuning is not None:
        params = tuning["best_params"]
        builders["XGBoost_Optuna"] = lambda p=params: xgb.XGBClassifier(
            objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss", tree_method="hist",
            random_state=RS, n_jobs=-1, verbosity=0, **p,
        )
        write_json(tuning, output_dir / "optuna_tuning.json")

    fitted = {}
    result_rows = []
    for name, factory in builders.items():
        try:
            artifact = evaluate_candidate(factory, name, train, validation, numeric_features, categorical_features, n_classes)
            fitted[name] = artifact
            trm, vam = artifact["train_metrics"], artifact["validation_metrics"]
            result_rows.append({
                "model": name,
                "train_f1_macro": trm["f1_macro"], "train_accuracy": trm["accuracy"],
                "validation_f1_macro": vam["f1_macro"], "validation_accuracy": vam["accuracy"],
                "validation_balanced_accuracy": vam["balanced_accuracy"],
                "validation_roc_auc_macro_ovr": vam["roc_auc_macro_ovr"],
                "validation_pr_auc_macro_ovr": vam["pr_auc_macro_ovr"],
                "validation_ece": vam["ece"],
                "train_validation_f1_gap": trm["f1_macro"] - vam["f1_macro"],
            })
            log(f"{category}/{head_name} {name}: train F1={trm['f1_macro']:.3f} | validation F1={vam['f1_macro']:.3f} | val acc={vam['accuracy']:.3f}")
        except Exception as exc:
            result_rows.append({"model": name, "error": f"{type(exc).__name__}: {str(exc)[:220]}"})
            log(f"{category}/{head_name} {name} failed: {type(exc).__name__}: {str(exc)[:120]}")
    leaderboard = pd.DataFrame(result_rows)
    viable = leaderboard[leaderboard["validation_f1_macro"].notna()].copy() if "validation_f1_macro" in leaderboard.columns else pd.DataFrame()
    if viable.empty:
        write_csv(leaderboard, output_dir / "model_performance.csv")
        return None
    viable = viable.sort_values(["validation_f1_macro", "validation_balanced_accuracy"], ascending=False, kind="mergesort")
    champion_name = viable.iloc[0]["model"]

    # The test benchmark is fitted after model selection is frozen. Its values are diagnostics, never a selection input.
    train_validation = pd.concat([train, validation], ignore_index=True).sort_values("_ae_date", kind="mergesort")
    test_artifacts = {}
    for name, factory in builders.items():
        if name not in fitted:
            continue
        try:
            model, prep = fit_model(factory, train_validation[features], train_validation["_y"], numeric_features, categorical_features)
            probability = aligned_probabilities(model, prep.transform(test[features]), n_classes)
            test_artifacts[name] = {"model": model, "prep": prep, "test_probability": probability}
            metrics = metric_bundle(test["_y"], probability, np.arange(n_classes))
            row_idx = leaderboard.index[leaderboard["model"] == name]
            if len(row_idx):
                for key, value in metrics.items():
                    leaderboard.loc[row_idx, f"test_{key}"] = value
        except Exception as exc:
            log(f"{category}/{head_name} test diagnostic failed for {name}: {type(exc).__name__}")
    champion_row = leaderboard.loc[leaderboard["model"] == champion_name].iloc[0].copy()
    champion_row["test_validation_f1_drop"] = champion_row["validation_f1_macro"] - champion_row.get("test_f1_macro", np.nan)
    leaderboard.loc[leaderboard["model"] == champion_name, "test_validation_f1_drop"] = champion_row["test_validation_f1_drop"]
    baseline_row = leaderboard.loc[leaderboard["model"] == "BaselineMostFrequent"].iloc[0]
    gate = promotion_gate(head_name, champion_row, baseline_row, target_meta)
    leaderboard["selected_on_validation"] = leaderboard["model"] == champion_name
    leaderboard["test_used_for_selection"] = False
    write_csv(leaderboard.sort_values("validation_f1_macro", ascending=False), output_dir / "model_performance.csv")
    plot_metric_bakeoff(leaderboard, f"{category} {head_name}: train/validation/test performance", output_dir / "model_bakeoff.png")

    selected_fit = fitted[champion_name]
    selected_test = test_artifacts.get(champion_name)
    if selected_test is None:
        return None
    test_probability = selected_test["test_probability"]
    labels = [str(item) for item in encoder.classes_]
    test_metrics = metric_bundle(test["_y"], test_probability, np.arange(n_classes))
    test_report = classification_report(
        test["_y"], test_probability.argmax(axis=1), labels=np.arange(n_classes), target_names=labels,
        zero_division=0, output_dict=True,
    )
    write_json(test_report, output_dir / "test_classification_report.json")
    plot_confusion_matrix(test["_y"], test_probability, labels, f"{category} {head_name}: temporal test confusion", output_dir / "test_confusion_matrix.png")
    plot_roc_pr(test["_y"], test_probability, labels, f"{category} {head_name}", output_dir)
    reliability = plot_reliability(test["_y"], test_probability, f"{category} {head_name}: test reliability", output_dir / "test_reliability.png")
    write_csv(reliability, output_dir / "test_reliability_bins.csv")
    importance = safe_permutation_importance(
        selected_fit["model"], selected_fit["prep"], validation, features, validation["_y"],
        numeric_features, categorical_features, output_dir, f"{category} {head_name}: validation permutation importance",
    )
    optional_shap_summary(selected_fit["model"], selected_fit["prep"], validation[features], output_dir, f"{category} {head_name}: validation SHAP")
    cv, cv_summary = run_robustness_cv(train_validation, builders[champion_name], numeric_features, categorical_features, n_classes, output_dir / "robustness_cross_validation.csv")

    # Build an explicitly post-evaluation serving bundle on all historical rows with training-supported classes.
    serving_model, serving_prep = fit_final_serving_bundle(
        builders[champion_name], supported, numeric_features, categorical_features, encoder,
        head_name, category, target_col, output_dir,
    )
    predictions = _prediction_rows(test, test_probability, encoder, head_name)
    head_summary = {
        "device_category": category,
        "head": head_name,
        "target": target_col,
        "champion_selected_on_validation": champion_name,
        "classes": labels,
        "n_features": len(features),
        "n_train": int(len(train)), "n_validation": int(len(validation)), "n_test": int(len(test)),
        "train_metrics_champion": selected_fit["train_metrics"],
        "validation_metrics_champion": selected_fit["validation_metrics"],
        "test_metrics_champion": test_metrics,
        "promotion_gate": gate,
        "target_contract": target_meta,
        "suspicious_single_feature_drivers": suspicious,
        "permutation_top_drivers": importance.head(10).to_dict(orient="records") if not importance.empty else [],
        "cross_validation_rows": int(len(cv)),
    }
    write_json(head_summary, output_dir / "champion_summary.json")
    return {
        "summary": head_summary,
        "head_name": head_name,
        "target": target_col,
        "champion_name": champion_name,
        "features": features,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "encoder": encoder,
        "test_model": selected_test["model"],
        "test_prep": selected_test["prep"],
        "serving_model": serving_model,
        "serving_prep": serving_prep,
        "test_frame": test,
        "test_probability": test_probability,
        "test_predictions": predictions,
        "gate": gate,
    }


## 11. Out-of-sample prediction feeds, action ranking, errors, and Validator OOS proxy


In [ ]:

ROOT_CAUSE_FALLBACK = "UNCLASSIFIED_NO_ROOT_CAUSE_PREDICTION"
UNKNOWN_SERIAL = "UNKNOWN_SERIAL"
UNKNOWN_DEVICE = "UNKNOWN_DEVICE"
PRIORITY_RANK = {"monitor": 1, "triage": 1, "plan": 2, "urgent": 3}
RANK_PRIORITY = {1: "monitor", 2: "plan", 3: "urgent"}


def stable_mode(values, default=ROOT_CAUSE_FALLBACK):
    """Deterministic mode that never throws when a group is entirely null.

    Root-cause predictions can legitimately be blank when the root-cause head was not promotable, a test label was
    unseen during training, or only the severity head produced a prediction.  Returning an explicit state is more
    honest than dropping the device or failing the entire category pipeline.
    """
    if values is None:
        return default
    cleaned = pd.Series(values, copy=False).astype("string").str.strip()
    invalid = cleaned.isna() | cleaned.fillna("").str.lower().isin({"", "nan", "none", "<na>", "null"})
    cleaned = cleaned.loc[~invalid]
    if cleaned.empty:
        return default
    counts = cleaned.value_counts(dropna=True)
    if counts.empty:
        return default
    winners = sorted(str(value) for value in counts[counts.eq(counts.max())].index)
    return winners[0] if winners else default


def normalise_serial_number(values):
    serial = pd.Series(values, copy=False).astype("string").str.strip()
    invalid = serial.isna() | serial.fillna("").str.lower().isin({"", "nan", "none", "<na>", "null"})
    return serial.mask(invalid, UNKNOWN_SERIAL).astype("string")


def normalise_device_identifier(values):
    device = pd.Series(values, copy=False).astype("string").str.strip()
    invalid = device.isna() | device.fillna("").str.lower().isin({"", "nan", "none", "<na>", "null"})
    return device.mask(invalid, UNKNOWN_DEVICE).astype("string")


def add_precursor_anomaly_scores(incidents, reference_history, feature_columns):
    """Score approved pre-incident features against a training-only robust baseline.

    The action feed intentionally uses only the training partition as the
    reference distribution.  Median/MAD scaling is resilient to skewed and
    outlier-heavy operational telemetry, and unavailable features simply do
    not contribute to a row's score.
    """
    output = incidents.copy()
    output["precursor_anomaly_score"] = np.nan
    output["top_precursor_anomaly_driver"] = pd.NA
    output["precursor_anomaly_feature_count"] = 0
    output["precursor_anomaly_baseline_scope"] = "training_partition_only"
    if output.empty or reference_history is None or reference_history.empty:
        return output
    reference = reference_history.copy()
    if "_split" in reference.columns and reference["_split"].eq("train").any():
        reference = reference.loc[reference["_split"].eq("train")].copy()
    numeric_features = [
        feature for feature in dict.fromkeys(feature_columns)
        if feature in output.columns and feature in reference.columns
        and pd.api.types.is_numeric_dtype(reference[feature])
    ]
    score_columns = {}
    minimum_rows = int(CONFIG.get("ANOMALY_MIN_REFERENCE_ROWS", 20))
    for feature in numeric_features:
        baseline = pd.to_numeric(reference[feature], errors="coerce").dropna()
        if len(baseline) < minimum_rows:
            continue
        median = float(baseline.median())
        mad = float(np.median(np.abs(baseline - median)))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale <= 1e-9:
            q75, q25 = np.nanpercentile(baseline, [75, 25])
            scale = float((q75 - q25) / 1.349)
        if not np.isfinite(scale) or scale <= 1e-9:
            continue
        values = pd.to_numeric(output[feature], errors="coerce")
        score_columns[feature] = ((values - median).abs() / scale).clip(lower=0, upper=5)
    if not score_columns:
        return output
    scores = pd.DataFrame(score_columns, index=output.index)
    output["precursor_anomaly_feature_count"] = scores.notna().sum(axis=1).astype(int)
    output["precursor_anomaly_score"] = scores.mean(axis=1, skipna=True).clip(lower=0, upper=5)
    driver = []
    for _, row in scores.iterrows():
        valid = row.dropna()
        driver.append(valid.idxmax() if not valid.empty else pd.NA)
    output["top_precursor_anomaly_driver"] = pd.Series(driver, index=output.index, dtype="string")
    return output

def add_prediction_evaluation_columns(incidents):
    """Add row-level evaluability/correctness fields without pretending an unavailable head made a prediction."""
    output = incidents.copy()
    for head_name in ("severity", "root_cause"):
        actual_col, predicted_col = f"actual_{head_name}", f"pred_{head_name}"
        available_col, correct_col = f"{head_name}_prediction_evaluable", f"{head_name}_correct"
        output[available_col] = False
        output[correct_col] = pd.Series(pd.NA, index=output.index, dtype="boolean")
        if actual_col in output.columns and predicted_col in output.columns:
            evaluable = output[actual_col].notna() & output[predicted_col].notna()
            output.loc[evaluable, available_col] = True
            output.loc[evaluable, correct_col] = (
                output.loc[evaluable, actual_col].astype("string") == output.loc[evaluable, predicted_col].astype("string")
            ).astype("boolean")
    root_prediction_present = output.get("pred_root_cause", pd.Series(pd.NA, index=output.index)).notna()
    output["root_cause_prediction_available"] = root_prediction_present.astype(bool)
    if "pred_root_cause" not in output.columns:
        output["pred_root_cause"] = ROOT_CAUSE_FALLBACK
    else:
        output["pred_root_cause"] = output["pred_root_cause"].astype("string").fillna(ROOT_CAUSE_FALLBACK)
    return output


def merge_group_performance(summary, incidents, group_columns):
    """Attach device/serial-level back-test quality only where an actual label and prediction both exist."""
    output = summary.copy()
    for head_name in ("severity", "root_cause"):
        correct_col = f"{head_name}_correct"
        if correct_col not in incidents.columns:
            continue
        evaluable = incidents.loc[incidents[correct_col].notna(), list(group_columns) + [correct_col]].copy()
        if evaluable.empty:
            output[f"{head_name}_evaluable_predictions"] = 0
            output[f"{head_name}_accuracy_on_evaluable_predictions"] = np.nan
            continue
        evaluable["_correct_numeric"] = evaluable[correct_col].astype(float)
        perf = evaluable.groupby(list(group_columns), dropna=False).agg(
            **{
                f"{head_name}_evaluable_predictions": ("_correct_numeric", "size"),
                f"{head_name}_accuracy_on_evaluable_predictions": ("_correct_numeric", "mean"),
            }
        ).reset_index()
        output = output.merge(perf, on=list(group_columns), how="left", validate="1:1")
        output[f"{head_name}_evaluable_predictions"] = output[f"{head_name}_evaluable_predictions"].fillna(0).astype(int)
    return output


def build_device_serial_history_profile(category_df, category, category_out):
    """Descriptive historical profile for every device_id + serial_number; it is not a future scoring feed."""
    history = category_df.copy()
    history["device_id"] = normalise_device_identifier(history.get("device_id", pd.Series(pd.NA, index=history.index)))
    history["serial_number"] = normalise_serial_number(history.get("matched_serial_nbr", pd.Series(pd.NA, index=history.index)))
    history["serial_number_status"] = np.where(history["serial_number"].eq(UNKNOWN_SERIAL), "missing_in_source", "matched")
    history["AE_START_DTM"] = pd.to_datetime(history.get("AE_START_DTM"), errors="coerce")
    if CONFIG["SEVERITY_TARGET"] in history.columns:
        history["_historical_severity_bucket"] = severity_bucket(history[CONFIG["SEVERITY_TARGET"]])
    if "AE_START_DTM" in history.columns and history["AE_START_DTM"].notna().any():
        ordered = history.sort_values(["device_id", "serial_number", "AE_START_DTM"], kind="mergesort").copy()
        ordered["_hours_since_previous_incident"] = ordered.groupby(["device_id", "serial_number"])["AE_START_DTM"].diff().dt.total_seconds() / 3600.0
        ordered["_repeat_within_30d"] = ordered["_hours_since_previous_incident"].le(24 * 30)
        history = history.merge(
            ordered[["_row_id", "_hours_since_previous_incident", "_repeat_within_30d"]], on="_row_id", how="left", validate="1:1"
        )
    aggregations = {
        "historical_incidents": ("_row_id", "size"),
        "first_historical_incident": ("AE_START_DTM", "min"),
        "last_historical_incident": ("AE_START_DTM", "max"),
        "serial_number_status": ("serial_number_status", stable_mode),
    }
    if "FACILITY_ID" in history.columns:
        aggregations["dominant_facility_id"] = ("FACILITY_ID", stable_mode)
    if "FACILITY_NAME" in history.columns:
        aggregations["dominant_facility_name"] = ("FACILITY_NAME", stable_mode)
    if "_historical_severity_bucket" in history.columns:
        aggregations["historical_critical_share"] = ("_historical_severity_bucket", lambda s: float((s == "CRITICAL").mean()))
    if "_repeat_within_30d" in history.columns:
        aggregations["historical_repeat_within_30d_rate"] = ("_repeat_within_30d", lambda s: float(s.dropna().mean()) if s.notna().any() else np.nan)
    component = CONFIG["ROOTCAUSE_TARGET"]
    if component in history.columns:
        aggregations["dominant_historical_component"] = (component, stable_mode)
    profile = history.groupby(["device_id", "serial_number"], dropna=False).agg(**aggregations).reset_index()
    profile["device_category"] = category
    profile["profile_scope"] = "descriptive_all_available_history_not_a_future_prediction"
    profile = profile.sort_values(["historical_incidents", "last_historical_incident"], ascending=[False, False])
    write_csv(profile, Path(category_out) / "device_serial_historical_profile.csv")
    return profile


def write_actionable_graphics(incidents, device_serial, category, category_out):
    out = Path(category_out) / "actionable_analysis"
    out.mkdir(parents=True, exist_ok=True)
    if incidents.empty:
        return
    priority_order = ["monitor", "plan", "urgent"]
    priority = incidents["action_priority"].fillna("monitor").value_counts().reindex(priority_order, fill_value=0)
    write_csv(priority.rename_axis("action_priority").reset_index(name="test_incidents"), out / "test_action_priority_distribution.csv")

    confidence_quality = pd.DataFrame()
    if "severity_confidence" in incidents.columns and "severity_correct" in incidents.columns:
        confidence = incidents.loc[incidents["severity_confidence"].notna() & incidents["severity_correct"].notna(), ["severity_confidence", "severity_correct"]].copy()
        if len(confidence) >= 10 and confidence["severity_confidence"].nunique() >= 2:
            try:
                confidence["confidence_band"] = pd.qcut(confidence["severity_confidence"], q=min(5, confidence["severity_confidence"].nunique()), duplicates="drop")
                confidence["_correct"] = confidence["severity_correct"].astype(float)
                confidence_quality = confidence.groupby("confidence_band", observed=True).agg(
                    rows=("_correct", "size"), mean_confidence=("severity_confidence", "mean"), accuracy=("_correct", "mean")
                ).reset_index()
                confidence_quality["confidence_band"] = confidence_quality["confidence_band"].astype(str)
                write_csv(confidence_quality, out / "severity_confidence_vs_accuracy.csv")
            except Exception:
                confidence_quality = pd.DataFrame()

    figure, axes = plt.subplots(2, 2, figsize=(14, 9))
    colours = {"monitor": "#4daf4a", "plan": "#ff7f00", "urgent": "#e41a1c"}
    axes[0, 0].bar(priority.index, priority.values, color=[colours[item] for item in priority.index])
    axes[0, 0].set_title(f"{category}: frozen-test action-priority mix")
    axes[0, 0].set_ylabel("Incident predictions")

    if "severity_critical_probability" in incidents.columns:
        x = pd.to_numeric(incidents.get("precursor_anomaly_score"), errors="coerce").fillna(0)
        y = pd.to_numeric(incidents["severity_critical_probability"], errors="coerce").fillna(0)
        axes[0, 1].scatter(x, y, c=incidents["action_priority_rank"].fillna(1), cmap="YlOrRd", alpha=0.65, s=24)
        axes[0, 1].set_title("Critical-risk probability vs precursor anomaly")
        axes[0, 1].set_xlabel("Precursor anomaly score")
        axes[0, 1].set_ylabel("Predicted critical probability")
    else:
        axes[0, 1].text(0.5, 0.5, "No severity probability available", ha="center", va="center")
        axes[0, 1].set_axis_off()

    date = pd.to_datetime(incidents.get("AE_START_DTM"), errors="coerce")
    if date.notna().any():
        trend_source = incidents.assign(_month=date.dt.to_period("M").astype(str))
        if "severity_critical_probability" in trend_source.columns:
            trend = trend_source.groupby("_month").agg(
                incident_predictions=("_row_id", "size"), mean_critical_probability=("severity_critical_probability", "mean")
            ).reset_index()
            trend.plot(x="_month", y="mean_critical_probability", marker="o", color="#377eb8", legend=False, ax=axes[1, 0])
            axes[1, 0].set_title("Mean predicted critical probability over time")
            axes[1, 0].set_ylabel("Probability")
        else:
            trend = trend_source.groupby("_month").size().rename("incident_predictions").reset_index()
            trend.plot(x="_month", y="incident_predictions", marker="o", color="#377eb8", legend=False, ax=axes[1, 0])
            axes[1, 0].set_title("Frozen-test incident predictions over time")
        axes[1, 0].tick_params(axis="x", rotation=45)
        write_csv(trend, out / "test_prediction_trend_by_month.csv")
    else:
        axes[1, 0].text(0.5, 0.5, "No usable incident date", ha="center", va="center")
        axes[1, 0].set_axis_off()

    if not confidence_quality.empty:
        axes[1, 1].plot(confidence_quality["mean_confidence"], confidence_quality["accuracy"], marker="o", color="#984ea3")
        axes[1, 1].plot([0, 1], [0, 1], "--", color="#777777", linewidth=1)
        axes[1, 1].set_xlim(0, 1); axes[1, 1].set_ylim(0, 1)
        axes[1, 1].set_title("Severity confidence vs observed test accuracy")
        axes[1, 1].set_xlabel("Mean confidence"); axes[1, 1].set_ylabel("Accuracy")
    else:
        axes[1, 1].text(0.5, 0.5, "Insufficient evaluable severity predictions", ha="center", va="center")
        axes[1, 1].set_axis_off()
    plot_or_close(out / "prediction_monitoring_dashboard.png")

    if not device_serial.empty:
        top = device_serial.sort_values("priority_score", ascending=False).head(20).sort_values("priority_score")
        labels = (top["device_id"].astype(str) + " | " + top["serial_number"].astype(str)).tolist()
        figure, axis = plt.subplots(figsize=(11, max(4.5, 0.35 * len(top) + 1)))
        bar_colours = [colours.get(value, "#777777") for value in top["highest_action_priority"]]
        axis.barh(labels, top["priority_score"], color=bar_colours)
        axis.set_title(f"{category}: highest-priority device / serial pairs (frozen test)")
        axis.set_xlabel("Priority score")
        plot_or_close(out / "top_device_serial_action_ranking.png")


def build_actionable_prediction_feeds(category_df, category, category_out, severity_head, root_head):
    heads = [item for item in (severity_head, root_head) if item is not None]
    if not heads:
        return None
    prediction_ids = pd.Index([], dtype=object)
    for item in heads:
        prediction_ids = prediction_ids.union(pd.Index(item["test_predictions"]["_row_id"]))
    base_columns = [
        "_row_id", "availability_event_id", "device_id", "mars_device_category", "AE_START_DTM",
        "FACILITY_ID", "FACILITY_NAME", "matched_serial_nbr", "component_age_days", "_split",
        CONFIG["SEVERITY_TARGET"], CONFIG["ROOTCAUSE_TARGET"],
    ]
    all_features = list(dict.fromkeys(sum((item["features"] for item in heads), [])))
    base_columns += [column for column in all_features if column not in base_columns]
    incidents = category_df.loc[category_df["_row_id"].isin(prediction_ids), [c for c in base_columns if c in category_df.columns]].copy()
    if incidents.empty:
        write_csv(incidents, Path(category_out) / "test_incident_predictions.csv")
        return {"incidents": incidents, "devices": pd.DataFrame(), "device_serial": pd.DataFrame()}
    for item in heads:
        incidents = incidents.merge(item["test_predictions"], on="_row_id", how="left", validate="1:1")
    incidents["device_id"] = normalise_device_identifier(incidents.get("device_id", pd.Series(pd.NA, index=incidents.index)))
    incidents["serial_number"] = normalise_serial_number(incidents.get("matched_serial_nbr", pd.Series(pd.NA, index=incidents.index)))
    incidents["serial_number_status"] = np.where(incidents["serial_number"].eq(UNKNOWN_SERIAL), "missing_in_source", "matched")
    incidents["prediction_scope"] = "frozen_temporal_test_holdout"
    incidents = add_precursor_anomaly_scores(incidents, category_df, all_features)
    if "precursor_anomaly_score" not in incidents.columns:
        incidents["precursor_anomaly_score"] = np.nan
        incidents["top_precursor_anomaly_driver"] = pd.NA
    incidents = add_prediction_evaluation_columns(incidents)
    if "severity_critical_probability" in incidents.columns:
        critical_probability = pd.to_numeric(incidents["severity_critical_probability"], errors="coerce").fillna(0)
        incidents["action_priority"] = pd.cut(
            critical_probability, bins=[-0.01, 0.30, 0.60, 1.01], labels=["monitor", "plan", "urgent"], include_lowest=True,
        ).astype("string").fillna("monitor")
        incidents["priority_score"] = 0.70 * critical_probability + 0.30 * (pd.to_numeric(incidents["precursor_anomaly_score"], errors="coerce").fillna(0).clip(0, 5) / 5)
    else:
        incidents["action_priority"] = "triage"
        incidents["priority_score"] = pd.to_numeric(incidents["precursor_anomaly_score"], errors="coerce").fillna(0).clip(0, 5) / 5
    incidents["action_priority_rank"] = incidents["action_priority"].map(PRIORITY_RANK).fillna(1).astype(int)
    incidents["priority_reason"] = np.where(
        incidents.get("root_cause_prediction_available", False),
        "severity probability + precursor anomaly; component prediction available",
        "severity probability + precursor anomaly; component unavailable/unclassified",
    )
    incidents = incidents.sort_values(["priority_score", "precursor_anomaly_score"], ascending=False, na_position="last")
    write_csv(incidents, Path(category_out) / "test_incident_predictions.csv")

    group_columns = ["device_id", "serial_number"]
    serial_aggs = {
        "device_category": ("mars_device_category", stable_mode),
        "serial_number_status": ("serial_number_status", stable_mode),
        "test_incidents": ("_row_id", "size"),
        "first_test_incident": ("AE_START_DTM", "min"),
        "last_test_incident": ("AE_START_DTM", "max"),
        "mean_confidence": ("severity_confidence", "mean") if "severity_confidence" in incidents.columns else ("_row_id", "size"),
        "max_precursor_anomaly": ("precursor_anomaly_score", "max"),
        "top_precursor_anomaly_driver": ("top_precursor_anomaly_driver", stable_mode),
        "dominant_predicted_component": ("pred_root_cause", stable_mode),
        "root_cause_prediction_coverage": ("root_cause_prediction_available", "mean"),
        "max_action_priority_rank": ("action_priority_rank", "max"),
        "max_incident_priority_score": ("priority_score", "max"),
    }
    if "FACILITY_ID" in incidents.columns:
        serial_aggs["dominant_facility_id"] = ("FACILITY_ID", stable_mode)
    if "FACILITY_NAME" in incidents.columns:
        serial_aggs["dominant_facility_name"] = ("FACILITY_NAME", stable_mode)
    if "severity_critical_probability" in incidents.columns:
        serial_aggs["mean_critical_probability"] = ("severity_critical_probability", "mean")
        serial_aggs["max_critical_probability"] = ("severity_critical_probability", "max")
    device_serial = incidents.groupby(group_columns, dropna=False).agg(**serial_aggs).reset_index()
    device_serial = merge_group_performance(device_serial, incidents, group_columns)
    if "max_critical_probability" in device_serial.columns:
        device_serial["priority_score"] = 0.70 * device_serial["max_critical_probability"].fillna(0) + 0.30 * (device_serial["max_precursor_anomaly"].fillna(0).clip(0, 5) / 5)
    else:
        device_serial["priority_score"] = device_serial["max_precursor_anomaly"].fillna(0).clip(0, 5) / 5
    device_serial["highest_action_priority"] = device_serial["max_action_priority_rank"].map(RANK_PRIORITY).fillna("monitor")
    device_serial["prediction_scope"] = "frozen_temporal_test_holdout"
    device_serial = device_serial.sort_values(["priority_score", "test_incidents"], ascending=False)
    write_csv(device_serial, Path(category_out) / "test_device_serial_action_ranking.csv")

    device_aggs = {
        "device_category": ("mars_device_category", stable_mode),
        "test_incidents": ("_row_id", "size"),
        "last_test_incident": ("AE_START_DTM", "max"),
        "mean_confidence": ("severity_confidence", "mean") if "severity_confidence" in incidents.columns else ("_row_id", "size"),
        "max_precursor_anomaly": ("precursor_anomaly_score", "max"),
        "dominant_predicted_component": ("pred_root_cause", stable_mode),
        "root_cause_prediction_coverage": ("root_cause_prediction_available", "mean"),
        "serials_in_test": ("serial_number", lambda s: int(s.loc[s.ne(UNKNOWN_SERIAL)].nunique())),
        "missing_serial_rows": ("serial_number_status", lambda s: int((s == "missing_in_source").sum())),
        "max_action_priority_rank": ("action_priority_rank", "max"),
    }
    if "severity_critical_probability" in incidents.columns:
        device_aggs["mean_critical_probability"] = ("severity_critical_probability", "mean")
        device_aggs["max_critical_probability"] = ("severity_critical_probability", "max")
    devices = incidents.groupby("device_id", dropna=False).agg(**device_aggs).reset_index()
    devices = merge_group_performance(devices, incidents, ["device_id"])
    if "max_critical_probability" in devices.columns:
        devices["priority_score"] = 0.70 * devices["max_critical_probability"].fillna(0) + 0.30 * (devices["max_precursor_anomaly"].fillna(0).clip(0, 5) / 5)
    else:
        devices["priority_score"] = devices["max_precursor_anomaly"].fillna(0).clip(0, 5) / 5
    devices["highest_action_priority"] = devices["max_action_priority_rank"].map(RANK_PRIORITY).fillna("monitor")
    devices["prediction_scope"] = "frozen_temporal_test_holdout"
    devices = devices.sort_values("priority_score", ascending=False)
    write_csv(devices, Path(category_out) / "test_device_action_ranking.csv")

    if "FACILITY_ID" in incidents.columns and incidents["FACILITY_ID"].notna().any():
        facility_aggs = {
            "test_incidents": ("_row_id", "size"), "devices": ("device_id", "nunique"),
            "device_serial_pairs": ("serial_number", "nunique"), "max_precursor_anomaly": ("precursor_anomaly_score", "max"),
            "urgent_predictions": ("action_priority_rank", lambda s: int((s == 3).sum())),
        }
        if "severity_critical_probability" in incidents.columns:
            facility_aggs["mean_critical_probability"] = ("severity_critical_probability", "mean")
            facility_aggs["max_critical_probability"] = ("severity_critical_probability", "max")
        facilities = incidents.groupby("FACILITY_ID", dropna=False).agg(**facility_aggs).reset_index()
        facilities["priority_score"] = (
            facilities.get("max_critical_probability", pd.Series(0, index=facilities.index)).fillna(0) * 0.70
            + facilities["max_precursor_anomaly"].fillna(0).clip(0, 5) / 5 * 0.30
        )
        facilities = facilities.sort_values("priority_score", ascending=False)
        write_csv(facilities, Path(category_out) / "test_facility_action_ranking.csv")

    build_device_serial_history_profile(category_df, category, category_out)
    write_actionable_graphics(incidents, device_serial, category, category_out)
    for head in heads:
        frame = head["test_predictions"].copy()
        actual_col, predicted_col = f"actual_{head['head_name']}", f"pred_{head['head_name']}"
        errors = (
            frame.assign(correct=frame[actual_col] == frame[predicted_col])
            .groupby([actual_col, predicted_col], dropna=False)
            .agg(rows=("_row_id", "size"), correct=("correct", "mean"))
            .reset_index()
            .sort_values("rows", ascending=False)
        )
        write_csv(errors, Path(category_out) / head["head_name"] / "test_error_and_confusion_pairs.csv")
    component_prediction_reliability = write_component_prediction_reliability(incidents, category, category_out)
    return {
        "incidents": incidents,
        "devices": devices,
        "device_serial": device_serial,
        "component_prediction_reliability": component_prediction_reliability,
    }

def validator_oos_proxy(category_out):
    """Descriptive Validator OOS intelligence, including component analysis only when labels exist."""
    out = Path(category_out) / "validator_oos_proxy"
    out.mkdir(parents=True, exist_ok=True)
    source = load_table(
        CONFIG["VALIDATOR_PROXY_TABLE"], f"{CONFIG['SILVER_PREFIX']}/device_failures", required=False,
        synthetic_key="device_failures",
    )
    if source is None or source.empty:
        stub = {
            "status": "unavailable",
            "message": "No Validator availability-event labels and no device_failures OOS proxy were available.",
            "recommended_action": "Export silver.device_failures with device, event date, severity, serial number, and component/root-cause fields where available. Do not fabricate a supervised Validator PS3 score.",
        }
        write_json(stub, out / "validator_proxy_status.json")
        return stub
    proxy, _ = canonicalise_incident_columns(source)
    if "mars_device_category" in proxy.columns:
        proxy = proxy[proxy["mars_device_category"].map(normalise_category) == "VALIDATOR"].copy()
    if proxy.empty:
        write_json({"status": "empty", "message": "device_failures has no VALIDATOR rows"}, out / "validator_proxy_status.json")
        return {"status": "empty"}
    device_col = _source_column(proxy, ["device_id", "DEVICE_ID", "ncs_device_id"])
    date_col = _source_column(proxy, ["ledger_date", "transit_day", "event_date", "failure_date", "AE_START_DTM"])
    severity_col = _source_column(proxy, ["failure_level_label", "failure_level", "FAILURE_LEVEL", "severity"])
    serial_col = _source_column(proxy, ["matched_serial_nbr", "serial_number", "device_serial_number", "serial_nbr"])
    if device_col is None:
        write_json({"status": "incomplete", "reason": "device identifier unavailable"}, out / "validator_proxy_status.json")
        return {"status": "incomplete"}
    proxy = proxy.rename(columns={device_col: "device_id"})
    proxy["event_dtm"] = pd.to_datetime(proxy[date_col], errors="coerce") if date_col is not None else pd.NaT
    proxy["severity_bucket"] = severity_bucket(proxy[severity_col]) if severity_col is not None else "UNKNOWN"
    proxy["serial_number"] = _clean_identifier(proxy[serial_col] if serial_col is not None else pd.Series(pd.NA, index=proxy.index), UNKNOWN_SERIAL)
    proxy["serial_number_status"] = np.where(proxy["serial_number"].eq(UNKNOWN_SERIAL), "missing_in_source", "matched")
    proxy["device_id"] = _clean_identifier(proxy["device_id"], UNKNOWN_DEVICE)
    proxy["_proxy_row_id"] = np.arange(len(proxy), dtype=int)
    proxy["_critical"] = proxy["severity_bucket"].eq("CRITICAL")
    proxy["_known_severity"] = proxy["severity_bucket"].ne("UNKNOWN")
    ordered = proxy.sort_values(["device_id", "serial_number", "event_dtm", "_proxy_row_id"], kind="mergesort").copy()
    ordered["_days_since_prior"] = ordered.groupby(["device_id", "serial_number"], dropna=False)["event_dtm"].diff().dt.total_seconds() / 86400.0
    ordered["_repeat_within_7d"] = ordered["_days_since_prior"].notna() & ordered["_days_since_prior"].le(7)
    proxy = proxy.merge(ordered[["_proxy_row_id", "_days_since_prior", "_repeat_within_7d"]], on="_proxy_row_id", how="left", validate="1:1")
    global_critical_rate = float(proxy.loc[proxy["_known_severity"], "_critical"].mean()) if proxy["_known_severity"].any() else 0.0
    priority = proxy.groupby(["device_id", "serial_number"], dropna=False).agg(
        oos_events=("_proxy_row_id", "size"),
        critical_events=("_critical", "sum"),
        known_severity_events=("_known_severity", "sum"),
        first_event=("event_dtm", "min"),
        last_event=("event_dtm", "max"),
        repeat_within_7d_events=("_repeat_within_7d", "sum"),
    ).reset_index()
    priority["critical_share_observed"] = priority["critical_events"] / priority["known_severity_events"].replace(0, np.nan)
    prior_strength = float(CONFIG["COMPONENT_PRIOR_STRENGTH"])
    priority["critical_rate_empirical_bayes"] = np.where(
        priority["known_severity_events"].gt(0),
        (priority["critical_events"] + global_critical_rate * prior_strength) / (priority["known_severity_events"] + prior_strength),
        np.nan,
    )
    priority["critical_rate_wilson_lower_95"] = wilson_lower_bound(priority["critical_events"], priority["known_severity_events"])
    priority["repeat_within_7d_rate"] = priority["repeat_within_7d_events"] / (priority["oos_events"] - 1).clip(lower=1)
    priority["proxy_priority_score"] = 100 * (
        0.45 * _positive_percentile_rank(priority["critical_rate_wilson_lower_95"])
        + 0.30 * _positive_percentile_rank(priority["repeat_within_7d_rate"])
        + 0.25 * _positive_percentile_rank(np.log1p(priority["oos_events"]))
    )
    priority["priority_scope"] = "descriptive Validator OOS proxy; not a supervised PS3 model or future failure probability"
    priority = priority.sort_values(["proxy_priority_score", "oos_events"], ascending=False)
    write_csv(priority, out / "validator_oos_device_serial_priority.csv")
    devices = priority.groupby("device_id", dropna=False).agg(
        serial_numbers=("serial_number", lambda s: int(s.loc[s.ne(UNKNOWN_SERIAL)].nunique())),
        oos_events=("oos_events", "sum"),
        critical_events=("critical_events", "sum"),
        known_severity_events=("known_severity_events", "sum"),
        last_event=("last_event", "max"),
        max_proxy_priority_score=("proxy_priority_score", "max"),
    ).reset_index()
    devices["critical_share_observed"] = devices["critical_events"] / devices["known_severity_events"].replace(0, np.nan)
    devices = devices.sort_values(["max_proxy_priority_score", "oos_events"], ascending=False)
    write_csv(devices, out / "validator_oos_device_priority.csv")
    severity = proxy["severity_bucket"].value_counts().rename_axis("severity_bucket").reset_index(name="oos_events")
    write_csv(severity, out / "oos_proxy_severity_distribution.csv")
    if proxy["event_dtm"].notna().any():
        proxy["month"] = proxy["event_dtm"].dt.to_period("M").astype(str)
        trend = proxy.groupby("month").size().rename("oos_events").reset_index()
        write_csv(trend, out / "monthly_oos_trend.csv")
        axis = trend.plot(x="month", y="oos_events", marker="o", legend=False, color="#984ea3", figsize=(9, 4))
        axis.set_title("Validator OOS-proxy events over time"); axis.set_xlabel("")
        plt.xticks(rotation=45, ha="right")
        plot_or_close(out / "monthly_oos_trend.png")
    if not severity.empty:
        axis = severity.plot(kind="bar", x="severity_bucket", y="oos_events", legend=False, color="#984ea3", figsize=(6, 4))
        axis.set_title("Validator OOS-proxy severity mix")
        plot_or_close(out / "oos_proxy_severity_distribution.png")
    component_col = CONFIG["ROOTCAUSE_TARGET"] if CONFIG["ROOTCAUSE_TARGET"] in proxy.columns else None
    component_summary = write_component_reliability_intelligence(
        proxy,
        "VALIDATOR",
        out,
        date_col="event_dtm",
        component_col=component_col,
        severity_col="severity_bucket",
        device_col="device_id",
        serial_col="serial_number",
        severity_is_bucket=True,
        scope="descriptive Validator device_failures OOS proxy; not a supervised PS3 root-cause model",
    )
    component_sequence_intelligence = safe_component_sequence_intelligence(
        proxy,
        "VALIDATOR",
        out,
        date_col="event_dtm",
        component_col=component_col,
        severity_col="severity_bucket",
        device_col="device_id",
        serial_col="serial_number",
        severity_is_bucket=True,
        scope="descriptive Validator device_failures OOS proxy sequence discovery; not a supervised PS3 model or causal transition estimate",
    )
    summary = {
        "status": "proxy_only_not_ps3_classifier",
        "oos_events": int(len(proxy)),
        "devices": int(proxy["device_id"].nunique()),
        "serial_number_coverage": float((proxy["serial_number"] != UNKNOWN_SERIAL).mean()),
        "first_event": str(proxy["event_dtm"].min()),
        "last_event": str(proxy["event_dtm"].max()),
        "critical_share": global_critical_rate if proxy["_known_severity"].any() else None,
        "component_intelligence": component_summary,
        "component_sequence_intelligence": component_sequence_intelligence,
        "note": "Validator severity is an OOS-proxy from device_failures, not an availability-event PS3 model. Component outputs are emitted only if a source component label exists.",
    }
    write_json(summary, out / "validator_proxy_summary.json")
    return summary


## 12. Per-device-type orchestration and run summary


## 12A. Component sequence intelligence and RDS dashboard contract

This section adds two deliberately descriptive reliability discoveries and a
production-friendly dashboard export contract:

- **Component transitions** quantify what component was observed next on the same
  device/serial, its conditional frequency, lift versus the overall next-component
  mix, time-to-next-event, and conservative evidence tiers. They are sequences in
  observed incident history — **not causal claims** and not a failure-rate model.
- **Inter-event reliability** summarizes observed time between repeat component
  incidents, including 7/30-day repeat shares, without claiming a survival rate
  when device exposure denominators are unavailable.
- **RDS dashboard bundle** emits governed, idempotent snapshot tables with a
  `run_id`, city, source scope, natural keys, record hashes, a data dictionary,
  PostgreSQL DDL, an upsert contract, and table-level load-readiness checks.

The notebook never connects to RDS by default. Set `RDS_WRITE_ENABLED=True` and
provide the configured connection-string environment variable only after reviewing
`rds_dashboard_bundle/rds_load_readiness.csv` and applying the generated DDL.


In [ ]:
# -----------------------------------------------------------------------------
# Component sequence discoveries + RDS dashboard export contract
# -----------------------------------------------------------------------------
# These helpers deliberately keep three analytical scopes separate:
#   1) historical all-incident reliability discovery,
#   2) frozen temporal-test prediction diagnostics, and
#   3) Validator device_failures OOS proxy intelligence.
# They never reinterpret descriptive counts, sequences, or proxy outputs as a
# supervised future-failure probability.

import csv
import hashlib
import os
import re


def assess_incident_data_quality(df):
    """Score the prepared incident input with transparent, non-destructive checks.

    The score is a data-readiness signal (not a model-performance metric):
    completeness 40%, validity 30%, consistency 20%, and uniqueness 10%.
    Serial-number coverage remains visible separately because it is important for
    drill-downs but its absence should not discard an otherwise valid incident.
    """
    rows = int(len(df))
    if rows == 0:
        return {
            "status": "empty",
            "quality_score": 0.0,
            "quality_gate_pass": False,
            "reason": "No valid incident rows remained after preparation.",
        }

    required = ["device_id", "mars_device_category", "AE_START_DTM", "availability_event_id"]
    required_present = [column for column in required if column in df.columns]
    completeness_parts = []
    for column in required_present:
        values = df[column]
        if pd.api.types.is_string_dtype(values) or values.dtype == object:
            valid = values.notna() & values.astype("string").str.strip().ne("")
        else:
            valid = values.notna()
        completeness_parts.append(float(valid.mean()))
    completeness = 100.0 * (float(np.mean(completeness_parts)) if completeness_parts else 0.0)

    valid_dates = float(pd.to_datetime(df.get("AE_START_DTM"), errors="coerce").notna().mean()) if "AE_START_DTM" in df else 0.0
    valid_devices = float(df.get("device_id", pd.Series(dtype="string")).astype("string").str.strip().ne("").mean()) if "device_id" in df else 0.0
    valid_categories = float(df.get("mars_device_category", pd.Series(dtype="string")).isin(CONFIG["DEVICE_SCOPE"]).mean()) if "mars_device_category" in df else 0.0
    validity = 100.0 * float(np.mean([valid_dates, valid_devices, valid_categories]))

    ordered = pd.to_datetime(df.get("AE_START_DTM"), errors="coerce")
    chronology_consistent = float(ordered.notna().mean())
    event_identifier_consistent = float(df.get("availability_event_id", pd.Series(dtype="string")).notna().mean()) if "availability_event_id" in df else 0.0
    consistency = 100.0 * float(np.mean([chronology_consistent, event_identifier_consistent]))

    duplicate_rate = float(df["availability_event_id"].duplicated().mean()) if "availability_event_id" in df else 1.0
    uniqueness = 100.0 * max(0.0, 1.0 - duplicate_rate)
    score = 0.40 * completeness + 0.30 * validity + 0.20 * consistency + 0.10 * uniqueness

    serial_source = "matched_serial_nbr" if "matched_serial_nbr" in df.columns else "serial_number" if "serial_number" in df.columns else None
    serial_coverage = None
    if serial_source:
        serial = df[serial_source].astype("string").str.strip()
        serial_coverage = float((serial.notna() & serial.ne("") & ~serial.str.upper().isin({"UNKNOWN", "UNKNOWN_SERIAL", "NULL", "NAN"})).mean())

    threshold = float(CONFIG.get("RDS_REQUIRED_QUALITY_SCORE", 97.0))
    return {
        "status": "assessed",
        "quality_score": round(float(score), 2),
        "quality_gate_threshold": threshold,
        "quality_gate_pass": bool(score >= threshold),
        "weights": {"completeness": 0.40, "validity": 0.30, "consistency": 0.20, "uniqueness": 0.10},
        "completeness_score": round(float(completeness), 2),
        "validity_score": round(float(validity), 2),
        "consistency_score": round(float(consistency), 2),
        "uniqueness_score": round(float(uniqueness), 2),
        "duplicate_event_rate_after_preparation": round(float(duplicate_rate), 6),
        "serial_number_coverage": None if serial_coverage is None else round(float(serial_coverage), 4),
        "rows_assessed": rows,
    }


def write_component_sequence_intelligence(
    df,
    category,
    category_out,
    *,
    date_col="AE_START_DTM",
    component_col=None,
    severity_col=None,
    device_col="device_id",
    serial_col="serial_number",
    severity_is_bucket=False,
    output_subdir="component_intelligence",
    scope="historical incident sequence discovery; descriptive only, not causal or a future failure probability",
):
    """Create device/serial component-transition and observed inter-event outputs.

    The risk set/exposure for all deployed devices is not known here. Therefore
    frequencies are explicitly retained as observed incident-sequence measures,
    not failure rates, hazards, or causal transitions.
    """
    out = Path(category_out) / output_subdir
    out.mkdir(parents=True, exist_ok=True)
    component_col = component_col or (CONFIG["ROOTCAUSE_TARGET"] if CONFIG["ROOTCAUSE_TARGET"] in df.columns else None)
    required = [date_col, device_col]
    missing = [column for column in required if column not in df.columns]
    if component_col is None or component_col not in df.columns:
        missing.append("component/root-cause label")
    if missing:
        summary = {
            "status": "component_label_or_sequence_fields_unavailable",
            "device_category": category,
            "missing_fields": missing,
            "scope": scope,
            "note": "No component transition or inter-event discovery was inferred without source component labels.",
        }
        write_json(summary, out / "component_sequence_intelligence_summary.json")
        return summary

    source = df.copy()
    events = pd.DataFrame(index=source.index)
    events["device_id"] = _clean_identifier(source[device_col], UNKNOWN_DEVICE)
    # Gold incident extracts commonly retain `matched_serial_nbr` until the
    # actionable-feed stage. Fall back through the known aliases so sequence
    # discovery keeps its intended device × serial grain rather than silently
    # collapsing every historical event to UNKNOWN_SERIAL.
    serial_candidates = [serial_col, "serial_number", "matched_serial_nbr", "device_serial_number", "serial_nbr"]
    serial_source = next((column for column in serial_candidates if column and column in source.columns), None)
    events["serial_number"] = _clean_identifier(
        source[serial_source] if serial_source is not None else pd.Series(pd.NA, index=source.index),
        UNKNOWN_SERIAL,
    )
    events["component"] = normalise_component_label(source[component_col])
    events["event_dtm"] = pd.to_datetime(source[date_col], errors="coerce")
    events["_source_row"] = np.arange(len(events), dtype=int)
    if severity_col is not None and severity_col in source.columns:
        if severity_is_bucket:
            events["_severity_bucket"] = source[severity_col].astype("string").str.upper().fillna("UNKNOWN")
        else:
            events["_severity_bucket"] = severity_bucket(source[severity_col])
    else:
        events["_severity_bucket"] = "UNKNOWN"
    events["_known_severity"] = events["_severity_bucket"].isin(["MAJOR", "CRITICAL"])
    events["_is_critical"] = events["_severity_bucket"].eq("CRITICAL")

    label_coverage = float(events["component"].ne(COMPONENT_UNKNOWN).mean()) if len(events) else 0.0
    dated = events[events["event_dtm"].notna() & events["component"].ne(COMPONENT_UNKNOWN)].copy()
    if dated.empty:
        summary = {
            "status": "no_dated_component_events",
            "device_category": category,
            "component_label_coverage": label_coverage,
            "scope": scope,
            "note": "Component labels may exist but no valid dated component event was available for sequence analysis.",
        }
        write_json(summary, out / "component_sequence_intelligence_summary.json")
        return summary

    # Primary sequences follow a device lifecycle. Serial identifiers stay as
    # continuity context so a replacement does not erase a device-level pattern.
    group_keys = ["device_id"]
    dated = dated.sort_values(group_keys + ["event_dtm", "_source_row"], kind="mergesort").reset_index(drop=True)
    grouped = dated.groupby(group_keys, dropna=False, sort=False)
    dated["next_component"] = grouped["component"].shift(-1)
    dated["next_event_dtm"] = grouped["event_dtm"].shift(-1)
    dated["next_serial_number"] = grouped["serial_number"].shift(-1)
    dated["next_is_critical"] = grouped["_is_critical"].shift(-1)
    dated["next_known_severity"] = grouped["_known_severity"].shift(-1)
    dated["hours_to_next_event"] = (dated["next_event_dtm"] - dated["event_dtm"]).dt.total_seconds() / 3600.0
    sequence = dated[
        dated["next_component"].notna()
        & dated["hours_to_next_event"].notna()
        & dated["hours_to_next_event"].ge(0)
    ].copy()
    sequence["serial_continuity_evaluable"] = (
        sequence["serial_number"].ne(UNKNOWN_SERIAL) & sequence["next_serial_number"].ne(UNKNOWN_SERIAL)
    )
    sequence["cross_serial_transition"] = (
        sequence["serial_continuity_evaluable"] & sequence["serial_number"].ne(sequence["next_serial_number"])
    )

    common_summary = {
        "device_category": category,
        "component_label_coverage": round(label_coverage, 4),
        "dated_component_events": int(len(dated)),
        "distinct_devices": int(dated["device_id"].nunique()),
        "distinct_known_serial_numbers": int(dated.loc[dated["serial_number"].ne(UNKNOWN_SERIAL), "serial_number"].nunique()),
        "scope": scope,
        "interpretation_guardrail": "Observed device-lifecycle incident sequences only. Serial continuity is reported separately. Conditional transition frequency is not a causal effect, failure rate, hazard, or prediction of the next failure.",
    }
    if sequence.empty:
        summary = {
            "status": "insufficient_within_device_sequence",
            **common_summary,
            "note": "At least two dated component events for the same device/serial are needed to estimate an observed transition.",
        }
        write_json(summary, out / "component_sequence_intelligence_summary.json")
        return summary

    transition = (
        sequence.groupby(["component", "next_component"], dropna=False)
        .agg(
            transition_events=("_source_row", "size"),
            affected_devices=("device_id", "nunique"),
            affected_serial_numbers=("serial_number", lambda s: int(s.loc[s.ne(UNKNOWN_SERIAL)].nunique())),
            first_transition=("event_dtm", "min"),
            last_transition=("event_dtm", "max"),
            median_hours_to_next_event=("hours_to_next_event", "median"),
            p90_hours_to_next_event=("hours_to_next_event", lambda s: float(s.quantile(0.90)) if len(s) else np.nan),
            cross_serial_transition_events=("cross_serial_transition", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            serial_continuity_evaluable_events=("serial_continuity_evaluable", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            next_critical_events=("next_is_critical", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            known_next_severity_events=("next_known_severity", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
        )
        .reset_index()
    )
    source_totals = sequence.groupby("component", dropna=False).size().rename("outgoing_component_sequences")
    global_next_mix = (sequence["next_component"].value_counts(dropna=False) / max(1, len(sequence))).rename("global_next_component_share")
    transition = transition.merge(source_totals, left_on="component", right_index=True, how="left")
    transition = transition.merge(global_next_mix, left_on="next_component", right_index=True, how="left")
    transition["conditional_next_component_probability"] = transition["transition_events"] / transition["outgoing_component_sequences"].replace(0, np.nan)
    transition["transition_lift_vs_global_next_component"] = (
        transition["conditional_next_component_probability"] / transition["global_next_component_share"].replace(0, np.nan)
    )
    transition["next_critical_share_observed"] = transition["next_critical_events"] / transition["known_next_severity_events"].replace(0, np.nan)
    transition["cross_serial_transition_share"] = transition["cross_serial_transition_events"] / transition["serial_continuity_evaluable_events"].replace(0, np.nan)
    transition["transition_evidence_tier"] = np.select(
        [
            transition["transition_events"].ge(10) & transition["affected_devices"].ge(3),
            transition["transition_events"].ge(int(CONFIG["COMPONENT_MIN_EVIDENCE_EVENTS"])),
        ],
        ["strong", "moderate"],
        default="limited",
    )
    transition["transition_priority_score"] = 100.0 * (
        0.40 * _positive_percentile_rank(np.log1p(transition["transition_events"]))
        + 0.35 * _positive_percentile_rank((transition["transition_lift_vs_global_next_component"] - 1.0).clip(lower=0))
        + 0.25 * _positive_percentile_rank(transition["next_critical_share_observed"])
    )
    transition["device_category"] = category
    transition["transition_scope"] = scope
    transition = transition.sort_values(["transition_priority_score", "transition_events"], ascending=False, kind="mergesort")
    write_csv(transition, out / "component_transition_portfolio.csv")

    matrix = transition.pivot_table(
        index="component", columns="next_component", values="conditional_next_component_probability", aggfunc="sum", fill_value=0.0
    )
    write_csv(matrix.reset_index(), out / "component_transition_matrix.csv")

    reference_dtm = dated["event_dtm"].max()
    device_followup = (
        sequence.groupby(["device_id", "component", "next_component"], dropna=False)
        .agg(
            observed_transitions=("_source_row", "size"),
            affected_serial_numbers=("serial_number", lambda s: int(s.loc[s.ne(UNKNOWN_SERIAL)].nunique())),
            first_transition=("event_dtm", "min"),
            last_transition=("event_dtm", "max"),
            median_hours_to_next_event=("hours_to_next_event", "median"),
            p90_hours_to_next_event=("hours_to_next_event", lambda s: float(s.quantile(0.90)) if len(s) else np.nan),
            cross_serial_transition_events=("cross_serial_transition", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            serial_continuity_evaluable_events=("serial_continuity_evaluable", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            next_critical_events=("next_is_critical", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            known_next_severity_events=("next_known_severity", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
        )
        .reset_index()
    )
    device_followup["observed_next_critical_share"] = device_followup["next_critical_events"] / device_followup["known_next_severity_events"].replace(0, np.nan)
    device_followup["cross_serial_transition_share"] = device_followup["cross_serial_transition_events"] / device_followup["serial_continuity_evaluable_events"].replace(0, np.nan)
    device_followup["days_since_last_transition"] = (reference_dtm - device_followup["last_transition"]).dt.total_seconds() / 86400.0
    device_followup["transition_followup_priority_score"] = 100.0 * (
        0.45 * _positive_percentile_rank(np.log1p(device_followup["observed_transitions"]))
        + 0.35 * _positive_percentile_rank(device_followup["observed_next_critical_share"])
        + 0.20 * _recency_percentile_rank(device_followup["days_since_last_transition"])
    )
    device_followup["device_category"] = category
    device_followup["transition_scope"] = scope
    device_followup = device_followup.sort_values(
        ["transition_followup_priority_score", "observed_transitions"], ascending=False, kind="mergesort"
    )
    write_csv(device_followup, out / "device_component_transition_followup.csv")

    repeat_events = dated.sort_values(["device_id", "component", "event_dtm", "_source_row"], kind="mergesort").copy()
    repeat_events["hours_since_same_component_incident"] = (
        repeat_events.groupby(["device_id", "component"], dropna=False)["event_dtm"].diff().dt.total_seconds() / 3600.0
    )
    intervals = repeat_events[
        repeat_events["hours_since_same_component_incident"].notna()
        & repeat_events["hours_since_same_component_incident"].ge(0)
    ].copy()
    component_events = (
        dated.groupby("component", dropna=False)
        .agg(
            historical_component_events=("_source_row", "size"),
            affected_devices=("device_id", "nunique"),
            affected_serial_numbers=("serial_number", lambda s: int(s.loc[s.ne(UNKNOWN_SERIAL)].nunique())),
            first_incident=("event_dtm", "min"),
            last_incident=("event_dtm", "max"),
        )
        .reset_index()
    )
    if intervals.empty:
        interevent = component_events.copy()
        interevent["comparable_interevent_intervals"] = 0
        interevent["median_days_between_same_component_incidents"] = np.nan
        interevent["p90_days_between_same_component_incidents"] = np.nan
        interevent["observed_repeat_within_7d_rate"] = np.nan
        interevent["observed_repeat_within_30d_rate"] = np.nan
    else:
        intervals["days_since_same_component_incident"] = intervals["hours_since_same_component_incident"] / 24.0
        interval_summary = (
            intervals.groupby("component", dropna=False)
            .agg(
                comparable_interevent_intervals=("_source_row", "size"),
                median_days_between_same_component_incidents=("days_since_same_component_incident", "median"),
                p90_days_between_same_component_incidents=("days_since_same_component_incident", lambda s: float(s.quantile(0.90)) if len(s) else np.nan),
                repeat_within_7d_intervals=("days_since_same_component_incident", lambda s: int(s.le(7).sum())),
                repeat_within_30d_intervals=("days_since_same_component_incident", lambda s: int(s.le(30).sum())),
            )
            .reset_index()
        )
        interevent = component_events.merge(interval_summary, on="component", how="left")
        interevent["comparable_interevent_intervals"] = interevent["comparable_interevent_intervals"].fillna(0).astype(int)
        interevent["observed_repeat_within_7d_rate"] = interevent["repeat_within_7d_intervals"] / interevent["comparable_interevent_intervals"].replace(0, np.nan)
        interevent["observed_repeat_within_30d_rate"] = interevent["repeat_within_30d_intervals"] / interevent["comparable_interevent_intervals"].replace(0, np.nan)
    interevent["interevent_evidence_tier"] = np.select(
        [
            interevent["comparable_interevent_intervals"].ge(10),
            interevent["comparable_interevent_intervals"].ge(int(CONFIG["COMPONENT_MIN_EVIDENCE_EVENTS"])),
        ],
        ["strong", "moderate"],
        default="limited",
    )
    interevent["device_category"] = category
    interevent["interevent_scope"] = (
        "Observed repeat incident intervals for the same device/serial/component; no fleet exposure denominator, so this is not a failure rate or survival estimate."
    )
    interevent = interevent.sort_values(
        ["observed_repeat_within_30d_rate", "historical_component_events"], ascending=False, na_position="last", kind="mergesort"
    )
    write_csv(interevent, out / "component_interevent_reliability.csv")

    try:
        top_components = transition.groupby("component", dropna=False)["transition_events"].sum().nlargest(12).index
        top_next = transition.groupby("next_component", dropna=False)["transition_events"].sum().nlargest(12).index
        visual = transition[transition["component"].isin(top_components) & transition["next_component"].isin(top_next)]
        visual_matrix = visual.pivot_table(
            index="component", columns="next_component", values="conditional_next_component_probability", aggfunc="sum", fill_value=0.0
        )
        if not visual_matrix.empty:
            plot_heatmap(
                visual_matrix,
                f"{category}: observed next-component conditional frequency",
                out / "component_transition_conditional_frequency.png",
            )
        top = transition.head(int(CONFIG["COMPONENT_PRIORITY_TOP_N"])).sort_values("transition_priority_score")
        if not top.empty:
            labels = (top["component"].astype(str) + " → " + top["next_component"].astype(str)).tolist()
            figure, axis = plt.subplots(figsize=(12, max(5, 0.32 * len(top) + 1)))
            axis.barh(labels, top["transition_priority_score"], color="#377eb8")
            axis.set_title(f"{category}: descriptive component transition follow-up priority")
            axis.set_xlabel("Observed-evidence priority score (not a probability)")
            plot_or_close(out / "component_transition_priority.png")
    except Exception as exc:
        log(f"{category}: component-sequence visualization skipped: {type(exc).__name__}: {str(exc)[:180]}")

    summary = {
        "status": "available",
        **common_summary,
        "observed_transition_events": int(len(sequence)),
        "transition_rows": int(len(transition)),
        "device_transition_followup_rows": int(len(device_followup)),
        "component_interevent_rows": int(len(interevent)),
        "top_transition": (
            f"{transition.iloc[0]['component']} -> {transition.iloc[0]['next_component']}" if len(transition) else None
        ),
    }
    write_json(summary, out / "component_sequence_intelligence_summary.json")
    return summary


def safe_component_sequence_intelligence(*args, **kwargs):
    """Keep a non-essential discovery export from failing the PS3 model run."""
    try:
        return write_component_sequence_intelligence(*args, **kwargs)
    except Exception as exc:
        category = kwargs.get("category") or (args[1] if len(args) > 1 else "UNKNOWN")
        category_out = kwargs.get("category_out") or (args[2] if len(args) > 2 else Path("."))
        output_subdir = kwargs.get("output_subdir", "component_intelligence")
        status = {
            "status": "failed",
            "device_category": category,
            "error": f"{type(exc).__name__}: {str(exc)[:600]}",
            "note": "Core PS3 outputs remain valid; only component sequence discovery failed.",
        }
        out = Path(category_out) / output_subdir
        out.mkdir(parents=True, exist_ok=True)
        write_json(status, out / "component_sequence_intelligence_summary.json")
        log(f"{category}: component sequence discovery skipped: {status['error']}")
        return status


# -----------------------------------------------------------------------------
# RDS dashboard-table generation
# -----------------------------------------------------------------------------

RDS_TABLE_VERSION = "1.0"


def _rds_identifier(value, fallback="ps3"):
    text = re.sub(r"[^a-zA-Z0-9_]+", "_", str(value or "").strip().lower())
    text = re.sub(r"_+", "_", text).strip("_")
    if not text:
        text = fallback
    if text[0].isdigit():
        text = f"x_{text}"
    return text[:63]


def _rds_quote_identifier(value):
    identifier = _rds_identifier(value)
    if identifier != str(value):
        raise ValueError(f"Unsafe or non-normalized SQL identifier: {value!r}")
    return f'"{identifier}"'


def _rds_safe_table_name(suffix):
    return _rds_identifier(f"{CONFIG.get('RDS_TABLE_PREFIX', 'ps3')}_{suffix}")


def _rds_sanitise_columns(frame):
    """Return a copy with lowercase SQL-safe, unique column names."""
    out = frame.copy()
    used, renamed = set(), {}
    for original in out.columns:
        base = _rds_identifier(original, fallback="field")
        candidate = base
        counter = 2
        while candidate in used:
            suffix = f"_{counter}"
            candidate = f"{base[:63 - len(suffix)]}{suffix}"
            counter += 1
        used.add(candidate)
        renamed[original] = candidate
    return out.rename(columns=renamed)


def _rds_read_csv(path):
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception as exc:
        log(f"RDS bundle: unable to read {path}: {type(exc).__name__}: {str(exc)[:180]}")
        return pd.DataFrame()


def _rds_source_series(frame, candidates, default=np.nan):
    for candidate in candidates:
        if candidate in frame.columns:
            return frame[candidate]
    return pd.Series(default, index=frame.index)


def _rds_collect_category_artifacts(output_root, relative_paths, scope_by_category, selector=None):
    """Read category outputs once and retain source lineage on every row."""
    root = Path(output_root)
    frames = []
    for category, relative in relative_paths.items():
        if not relative:
            continue
        path = root / relative
        if not path.exists():
            continue
        frame = _rds_read_csv(path)
        if frame.empty:
            continue
        if selector is not None:
            frame = selector(frame, category)
        if frame is None or frame.empty:
            continue
        frame = frame.copy()
        if "device_category" not in frame.columns:
            frame["device_category"] = category
        else:
            frame["device_category"] = frame["device_category"].fillna(category)
        frame["_rds_source_artifact"] = str(path.relative_to(root))
        frame["_rds_source_scope"] = scope_by_category.get(category, "dashboard source artifact")
        frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


def _rds_curate_action_queue(frame, category):
    """Normalize TVM/Gate frozen-test queues and Validator proxy queues to one grain."""
    out = pd.DataFrame(index=frame.index)
    out["device_id"] = _rds_source_series(frame, ["device_id"])
    out["serial_number"] = _rds_source_series(frame, ["serial_number"], UNKNOWN_SERIAL)
    out["event_count"] = _rds_source_series(frame, ["test_incidents", "oos_events", "incidents"], 0)
    out["first_event"] = _rds_source_series(frame, ["first_test_incident", "first_event", "first_incident"])
    out["last_event"] = _rds_source_series(frame, ["last_test_incident", "last_event", "last_incident"])
    out["priority_score"] = _rds_source_series(frame, ["priority_score", "proxy_priority_score", "max_proxy_priority_score"], np.nan)
    out["highest_action_priority"] = _rds_source_series(frame, ["highest_action_priority"], pd.NA).astype("string")
    validator_priority = pd.to_numeric(out["priority_score"], errors="coerce")
    derived_priority = pd.Series(np.select([validator_priority.ge(66), validator_priority.ge(33)], ["urgent", "plan"], default="monitor"), index=out.index)
    out["highest_action_priority"] = out["highest_action_priority"].fillna(derived_priority)
    out["max_critical_probability"] = _rds_source_series(frame, ["max_critical_probability", "critical_rate_empirical_bayes"], np.nan)
    out["root_cause_prediction_coverage"] = _rds_source_series(frame, ["root_cause_prediction_coverage"], np.nan)
    out["dominant_predicted_component"] = _rds_source_series(frame, ["dominant_predicted_component"])
    out["top_precursor_anomaly_driver"] = _rds_source_series(frame, ["top_precursor_anomaly_driver"])
    out["max_precursor_anomaly"] = _rds_source_series(frame, ["max_precursor_anomaly"])
    out["action_scope"] = _rds_source_series(frame, ["prediction_scope", "priority_scope"], pd.NA).astype("string")
    default_scope = (
        "descriptive Validator device_failures OOS proxy; not a supervised PS3 model"
        if category == "VALIDATOR"
        else "frozen temporal-test action ranking; backtest diagnostic, not a live future score"
    )
    out["action_scope"] = out["action_scope"].fillna(default_scope)
    out["queue_grain"] = "device_serial"
    return out


def _rds_curate_incident_backtest(frame, category):
    """Keep only dashboard drill-down columns, not the full feature matrix."""
    out = pd.DataFrame(index=frame.index)
    out["incident_id"] = _rds_source_series(frame, ["availability_event_id", "incident_id", "_row_id"])
    out["device_id"] = _rds_source_series(frame, ["device_id"])
    out["serial_number"] = _rds_source_series(frame, ["serial_number", "matched_serial_nbr"], UNKNOWN_SERIAL)
    out["event_dtm"] = _rds_source_series(frame, ["AE_START_DTM", "event_dtm"])
    out["facility_id"] = _rds_source_series(frame, ["FACILITY_ID", "facility_id"])
    out["facility_name"] = _rds_source_series(frame, ["FACILITY_NAME", "facility_name"])
    out["actual_severity"] = _rds_source_series(frame, ["actual_severity", "failure_level_label"])
    out["pred_severity"] = _rds_source_series(frame, ["pred_severity"])
    out["severity_confidence"] = _rds_source_series(frame, ["severity_confidence"])
    out["severity_critical_probability"] = _rds_source_series(frame, ["severity_critical_probability"])
    out["actual_root_cause"] = _rds_source_series(frame, ["actual_root_cause", "derived_component_type"])
    out["pred_root_cause"] = _rds_source_series(frame, ["pred_root_cause"], CONFIG["UNCLASSIFIED_ROOT_CAUSE_LABEL"])
    out["root_cause_confidence"] = _rds_source_series(frame, ["root_cause_confidence"])
    out["action_priority"] = _rds_source_series(frame, ["action_priority"])
    out["priority_score"] = _rds_source_series(frame, ["priority_score"])
    out["priority_reason"] = _rds_source_series(frame, ["priority_reason"])
    out["precursor_anomaly_score"] = _rds_source_series(frame, ["precursor_anomaly_score"])
    out["top_precursor_anomaly_driver"] = _rds_source_series(frame, ["top_precursor_anomaly_driver"])
    out["prediction_scope"] = _rds_source_series(frame, ["prediction_scope"], "frozen_temporal_test_holdout")
    return out


def _rds_normalize_key_series(values):
    series = pd.Series(values, copy=False).astype("string")
    return series.fillna("UNKNOWN").str.strip().replace({"": "UNKNOWN", "<NA>": "UNKNOWN", "nan": "UNKNOWN", "None": "UNKNOWN"})


def _rds_scalar_text(value):
    if value is None or value is pd.NA:
        return "<NULL>"
    try:
        if pd.isna(value):
            return "<NULL>"
    except (TypeError, ValueError):
        pass
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return "<NULL>"
    return str(value)


def _rds_record_hashes(frame):
    payload_columns = [column for column in frame.columns if column != "record_hash"]
    if frame.empty:
        return pd.Series(dtype="string")

    def encode_row(row):
        payload = "\x1f".join(f"{column}={_rds_scalar_text(row[column])}" for column in payload_columns)
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

    return frame.apply(encode_row, axis=1).astype("string")


def _rds_prepare_table(frame, *, table_name, natural_keys, run_context, default_scope, optional=False):
    """Add governed metadata and make natural keys deterministic for one RDS table."""
    if frame is None:
        frame = pd.DataFrame()
    frame = frame.copy()
    source_artifact = frame.pop("_rds_source_artifact") if "_rds_source_artifact" in frame.columns else pd.Series("generated_in_notebook", index=frame.index)
    source_scope = frame.pop("_rds_source_scope") if "_rds_source_scope" in frame.columns else pd.Series(default_scope, index=frame.index)
    frame = _rds_sanitise_columns(frame)
    if "mars_device_category" in frame.columns:
        if "device_category" not in frame.columns:
            frame = frame.rename(columns={"mars_device_category": "device_category"})
        else:
            frame["device_category"] = frame["device_category"].fillna(frame["mars_device_category"])
            frame = frame.drop(columns=["mars_device_category"])
    if "device_category" in frame.columns:
        frame["device_category"] = _rds_normalize_key_series(frame["device_category"]).str.upper()

    normal_keys = [_rds_identifier(key, fallback="key") for key in natural_keys]
    for key in normal_keys:
        if key not in frame.columns:
            frame[key] = "UNKNOWN"
        frame[key] = _rds_normalize_key_series(frame[key])

    frame["run_id"] = run_context["run_id"]
    frame["city_id"] = run_context["city_id"]
    frame["generated_at_utc"] = run_context["generated_at_utc"]
    frame["dashboard_table_version"] = RDS_TABLE_VERSION
    frame["source_artifact"] = pd.Series(source_artifact, index=frame.index).fillna("generated_in_notebook").astype("string")
    frame["source_scope"] = pd.Series(source_scope, index=frame.index).fillna(default_scope).astype("string")
    entity_columns = [key for key in normal_keys if key != "run_id"]
    frame["entity_key"] = (
        frame[entity_columns].astype("string").fillna("UNKNOWN").agg("|".join, axis=1)
        if entity_columns and not frame.empty
        else pd.Series("", index=frame.index, dtype="string")
    )
    # Run ID is always part of the database natural key, even when the source did
    # not expose it. This permits immutable run snapshots and idempotent retries.
    keys = ["run_id"] + [key for key in normal_keys if key != "run_id"]
    for key in keys:
        frame[key] = _rds_normalize_key_series(frame[key])
    frame["record_hash"] = _rds_record_hashes(frame)

    common = ["run_id", "city_id", "generated_at_utc", "dashboard_table_version", "entity_key", "record_hash", "source_artifact", "source_scope"]
    ordered = common + [column for column in frame.columns if column not in common]
    frame = frame.loc[:, ordered]
    return {
        "table_name": _rds_identifier(table_name),
        "frame": frame,
        "natural_keys": keys,
        "optional": bool(optional),
        "scope": default_scope,
    }


def _rds_quality_row(table_spec):
    frame, keys = table_spec["frame"], table_spec["natural_keys"]
    rows = int(len(frame))
    if rows == 0:
        return {
            "rds_table": table_spec["table_name"],
            "row_count": 0,
            "natural_key": " | ".join(keys),
            "missing_natural_key_rows": 0,
            "duplicate_natural_key_rows": 0,
            "invalid_numeric_rows": 0,
            "completeness_score": np.nan,
            "validity_score": np.nan,
            "consistency_score": np.nan,
            "uniqueness_score": np.nan,
            "data_quality_score": np.nan,
            "optional_table": table_spec["optional"],
            "ready_for_rds": False,
            "load_status": "empty_optional" if table_spec["optional"] else "empty_required",
            "reason": "No rows were emitted for this optional analytical scope." if table_spec["optional"] else "Required RDS table has no rows.",
        }

    key_missing = pd.Series(False, index=frame.index)
    for key in keys:
        values = _rds_normalize_key_series(frame[key])
        key_missing |= values.eq("UNKNOWN")
    duplicate_rows = int(frame.duplicated(keys, keep=False).sum())
    invalid_numeric = 0
    for column in frame.columns:
        if pd.api.types.is_numeric_dtype(frame[column]):
            invalid_numeric += int(np.isinf(pd.to_numeric(frame[column], errors="coerce")).sum())
    completeness = 100.0 * (1.0 - float(key_missing.mean()))
    validity = 100.0 * (1.0 - min(1.0, invalid_numeric / max(1, rows)))
    consistency = 100.0 * float(frame["record_hash"].notna().mean() * frame["entity_key"].notna().mean())
    uniqueness = 100.0 * (1.0 - duplicate_rows / max(1, rows))
    quality_score = 0.40 * completeness + 0.30 * validity + 0.20 * consistency + 0.10 * uniqueness
    threshold = float(CONFIG.get("RDS_REQUIRED_QUALITY_SCORE", 97.0))
    ready = bool(key_missing.sum() == 0 and duplicate_rows == 0 and quality_score >= threshold)
    return {
        "rds_table": table_spec["table_name"],
        "row_count": rows,
        "natural_key": " | ".join(keys),
        "missing_natural_key_rows": int(key_missing.sum()),
        "duplicate_natural_key_rows": duplicate_rows,
        "invalid_numeric_rows": invalid_numeric,
        "completeness_score": round(float(completeness), 2),
        "validity_score": round(float(validity), 2),
        "consistency_score": round(float(consistency), 2),
        "uniqueness_score": round(float(uniqueness), 2),
        "data_quality_score": round(float(quality_score), 2),
        "optional_table": table_spec["optional"],
        "ready_for_rds": ready,
        "load_status": "ready" if ready else "not_ready",
        "reason": "Natural keys, record hashes, and output DQ checks passed." if ready else "Review missing keys, duplicate keys, invalid numeric values, or quality threshold.",
    }


def _rds_postgres_type(series, column):
    lower = str(column).lower()
    timestamp_names = {
        "generated_at_utc", "run_timestamp_utc", "event_dtm", "ae_start_dtm", "first_event", "last_event",
        "first_incident", "last_incident", "first_transition", "last_transition", "first_test_incident", "last_test_incident",
    }
    if lower in timestamp_names or lower.endswith("_dtm") or lower.endswith("_timestamp"):
        return "TIMESTAMPTZ"
    if pd.api.types.is_bool_dtype(series):
        return "BOOLEAN"
    if pd.api.types.is_integer_dtype(series):
        return "BIGINT"
    if pd.api.types.is_numeric_dtype(series):
        return "DOUBLE PRECISION"
    return "TEXT"


def _rds_column_description(column):
    descriptions = {
        "run_id": "Immutable PS3 run snapshot identifier; all dashboard tables are keyed by it.",
        "city_id": "Configured city or operating-region identifier.",
        "generated_at_utc": "UTC timestamp at which the dashboard record was exported.",
        "dashboard_table_version": "Version of the dashboard-table contract.",
        "entity_key": "Human-readable deterministic business key excluding the run ID.",
        "record_hash": "SHA-256 fingerprint of the exported record for lineage and idempotence checks.",
        "source_artifact": "Relative notebook output artifact used to generate the record.",
        "source_scope": "Analytical scope and interpretation guardrail for this record.",
        "device_category": "PS3 device category: TVM, GATE, or VALIDATOR.",
        "component": "Observed or derived component label; not silently imputed when source labels are unavailable.",
        "component_label_coverage": "Share of source rows with a usable component label.",
        "reliability_priority_score": "Transparent historical triage score; not a future failure probability.",
        "transition_priority_score": "Observed-sequence follow-up score; not a causal or predictive transition probability.",
        "conditional_next_component_probability": "Observed conditional next-component frequency within available incident sequences.",
        "transition_lift_vs_global_next_component": "Observed conditional next-component frequency divided by the global next-component mix.",
        "observed_repeat_within_30d_rate": "Observed share of comparable repeat intervals within 30 days; not a fleet failure rate.",
        "priority_score": "Backtest or proxy triage score. Consult source_scope before operational interpretation.",
    }
    return descriptions.get(column, "PS3 dashboard field exported from the governed notebook artifact.")


def _rds_build_ddl(schema, table_specs):
    schema = _rds_identifier(schema, fallback="mars_dashboard")
    statements = [f"CREATE SCHEMA IF NOT EXISTS {_rds_quote_identifier(schema)};"]
    for spec in table_specs:
        table = _rds_quote_identifier(spec["table_name"])
        columns = []
        for column in spec["frame"].columns:
            dtype = _rds_postgres_type(spec["frame"][column], column)
            required = " NOT NULL" if column in spec["natural_keys"] else ""
            columns.append(f"    {_rds_quote_identifier(column)} {dtype}{required}")
        primary_key = ", ".join(_rds_quote_identifier(column) for column in spec["natural_keys"])
        statements.append(
            f"CREATE TABLE IF NOT EXISTS {_rds_quote_identifier(schema)}.{table} (\n"
            + ",\n".join(columns)
            + f",\n    PRIMARY KEY ({primary_key})\n);"
        )
        # Metrics evolve more often than dashboard dimensions. ADD COLUMN IF NOT
        # EXISTS makes a later notebook version additive rather than destructive.
        for column in spec["frame"].columns:
            if column in spec["natural_keys"]:
                continue
            dtype = _rds_postgres_type(spec["frame"][column], column)
            statements.append(
                f"ALTER TABLE {_rds_quote_identifier(schema)}.{table} "
                f"ADD COLUMN IF NOT EXISTS {_rds_quote_identifier(column)} {dtype};"
            )
    return "\n\n".join(statements) + "\n"


def _rds_build_upsert_template(schema, table_specs):
    schema = _rds_identifier(schema, fallback="mars_dashboard")
    blocks = [
        "-- The notebook loader creates a per-table staging table, then applies this pattern.",
        "-- For a retry of the same run_id it first deletes only that run snapshot, so other runs remain immutable.",
    ]
    for spec in table_specs:
        columns = list(spec["frame"].columns)
        keys = list(spec["natural_keys"])
        update_columns = [column for column in columns if column not in keys]
        q_columns = ", ".join(_rds_quote_identifier(column) for column in columns)
        conflict = ", ".join(_rds_quote_identifier(column) for column in keys)
        if update_columns:
            updates = ", ".join(f"{_rds_quote_identifier(column)} = EXCLUDED.{_rds_quote_identifier(column)}" for column in update_columns)
            conflict_clause = f"DO UPDATE SET {updates}"
        else:
            conflict_clause = "DO NOTHING"
        blocks.append(
            f"-- {spec['table_name']}\n"
            f"INSERT INTO {_rds_quote_identifier(schema)}.{_rds_quote_identifier(spec['table_name'])} ({q_columns})\n"
            f"SELECT {q_columns} FROM {_rds_quote_identifier(schema)}.{_rds_quote_identifier('staging_' + spec['table_name'])}\n"
            f"ON CONFLICT ({conflict}) {conflict_clause};"
        )
    return "\n\n".join(blocks) + "\n"


def _rds_data_dictionary(table_specs):
    rows = []
    for spec in table_specs:
        for ordinal, column in enumerate(spec["frame"].columns, start=1):
            rows.append(
                {
                    "rds_table": spec["table_name"],
                    "ordinal_position": ordinal,
                    "column_name": column,
                    "postgres_type": _rds_postgres_type(spec["frame"][column], column),
                    "nullable": column not in spec["natural_keys"],
                    "natural_key_member": column in spec["natural_keys"],
                    "description": _rds_column_description(column),
                    "analytical_scope": spec["scope"],
                }
            )
    return pd.DataFrame(rows)


def _rds_file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _rds_run_context(output_root, run_summary):
    timestamp = str(run_summary.get("run_timestamp_utc") or datetime.now(timezone.utc).isoformat())
    parsed = pd.to_datetime(timestamp, errors="coerce", utc=True)
    if pd.isna(parsed):
        parsed = pd.Timestamp.now(tz="UTC")
    generated = parsed.isoformat()
    configured = CONFIG.get("RDS_RUN_ID")
    default_id = f"ps3_{CONFIG.get('CITY_ID', 'city')}_{parsed.strftime('%Y%m%dT%H%M%SZ')}"
    return {
        "run_id": _rds_identifier(configured or default_id, fallback="ps3_run"),
        "city_id": str(CONFIG.get("CITY_ID", "UNKNOWN")),
        "generated_at_utc": generated,
        "output_root": str(output_root),
    }


def write_rds_dashboard_bundle(output_root, run_summary):
    """Emit governed dashboard tables and PostgreSQL load assets; do not connect to RDS."""
    root = Path(output_root)
    bundle = root / "rds_dashboard_bundle"
    table_dir = bundle / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)
    context = _rds_run_context(root, run_summary)
    schema = _rds_identifier(CONFIG.get("RDS_SCHEMA", "mars_dashboard"), fallback="mars_dashboard")
    category_scope = {
        "TVM": "historical TVM incident reliability discovery and frozen-test diagnostics; scores are not live future probabilities",
        "GATE": "historical Gate incident reliability discovery and frozen-test diagnostics; scores are not live future probabilities",
        "VALIDATOR": "descriptive Validator device_failures OOS proxy when Gold labels are absent; not a supervised PS3 classifier",
    }

    result_rows = run_summary.get("results", []) if isinstance(run_summary, dict) else []
    run_row = {
        "ps_id": "PS3",
        "run_timestamp_utc": run_summary.get("run_timestamp_utc"),
        "output_root": str(root),
        "device_scope": "|".join(CONFIG.get("DEVICE_SCOPE", [])),
        "category_modes_json": json.dumps(
            {str(row.get("category")): row.get("mode") for row in result_rows if isinstance(row, dict)}, sort_keys=True
        ),
        "input_rows": run_summary.get("data_quality", {}).get("input_rows"),
        "prepared_rows": run_summary.get("data_quality", {}).get("output_rows"),
        "input_quality_score": run_summary.get("data_quality", {}).get("data_quality_assessment", {}).get("quality_score"),
        "input_quality_gate_pass": run_summary.get("data_quality", {}).get("data_quality_assessment", {}).get("quality_gate_pass"),
        "rds_schema": schema,
        "rds_table_contract_version": RDS_TABLE_VERSION,
    }

    paths = {
        "component_portfolio": {
            "TVM": "tvm/component_intelligence/component_reliability_portfolio.csv",
            "GATE": "gates/component_intelligence/component_reliability_portfolio.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/component_reliability_portfolio.csv",
        },
        "device_component": {
            "TVM": "tvm/component_intelligence/device_component_reliability_ranking.csv",
            "GATE": "gates/component_intelligence/device_component_reliability_ranking.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/device_component_reliability_ranking.csv",
        },
        "device_serial_component": {
            "TVM": "tvm/component_intelligence/device_serial_component_reliability_ranking.csv",
            "GATE": "gates/component_intelligence/device_serial_component_reliability_ranking.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/device_serial_component_reliability_ranking.csv",
        },
        "component_prediction_quality": {
            "TVM": "tvm/component_intelligence/component_prediction_reliability.csv",
            "GATE": "gates/component_intelligence/component_prediction_reliability.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/component_prediction_reliability.csv",
        },
        "component_monthly": {
            "TVM": "tvm/component_intelligence/component_monthly_incidents.csv",
            "GATE": "gates/component_intelligence/component_monthly_incidents.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/component_monthly_incidents.csv",
        },
        "component_transition": {
            "TVM": "tvm/component_intelligence/component_transition_portfolio.csv",
            "GATE": "gates/component_intelligence/component_transition_portfolio.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/component_transition_portfolio.csv",
        },
        "device_component_transition": {
            "TVM": "tvm/component_intelligence/device_component_transition_followup.csv",
            "GATE": "gates/component_intelligence/device_component_transition_followup.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/device_component_transition_followup.csv",
        },
        "component_interevent": {
            "TVM": "tvm/component_intelligence/component_interevent_reliability.csv",
            "GATE": "gates/component_intelligence/component_interevent_reliability.csv",
            "VALIDATOR": "validators/validator_oos_proxy/component_intelligence/component_interevent_reliability.csv",
        },
        "action_queue": {
            "TVM": "tvm/test_device_serial_action_ranking.csv",
            "GATE": "gates/test_device_serial_action_ranking.csv",
            "VALIDATOR": "validators/validator_oos_proxy/validator_oos_device_serial_priority.csv",
        },
        "incident_backtest": {
            "TVM": "tvm/test_incident_predictions.csv",
            "GATE": "gates/test_incident_predictions.csv",
        },
    }

    table_specs = []
    table_specs.append(
        _rds_prepare_table(
            pd.DataFrame([run_row]),
            table_name=_rds_safe_table_name("dashboard_runs"),
            natural_keys=["run_id"],
            run_context=context,
            default_scope="PS3 run-level governance metadata and data-quality gate.",
            optional=False,
        )
    )
    source_specs = [
        ("dashboard_component_portfolio", "component_portfolio", ["device_category", "component"], "All-history component reliability discovery; not a future failure rate or probability."),
        ("dashboard_device_component", "device_component", ["device_category", "device_id", "component"], "All-history device × component reliability discovery; historical and descriptive."),
        ("dashboard_device_serial_component", "device_serial_component", ["device_category", "device_id", "serial_number", "component"], "All-history device × serial × component reliability discovery; historical and descriptive."),
        ("dashboard_component_prediction_quality", "component_prediction_quality", ["device_category", "actual_component"], "Frozen temporal-test component diagnostics with evidence bounds; not live performance."),
        ("dashboard_component_monthly", "component_monthly", ["device_category", "month", "component"], "Historical component incident trend; counts lack fleet exposure denominators."),
        ("dashboard_component_transitions", "component_transition", ["device_category", "component", "next_component"], "Observed device-lifecycle component sequences with serial-continuity context; descriptive, not causal or predictive."),
        ("dashboard_device_component_transition_followup", "device_component_transition", ["device_category", "device_id", "component", "next_component"], "Observed device-level component-transition follow-up with serial-continuity context; descriptive, not causal or predictive."),
        ("dashboard_component_interevent", "component_interevent", ["device_category", "component"], "Observed repeat component intervals; not a fleet failure rate or survival estimate."),
        ("dashboard_action_queue", "action_queue", ["device_category", "device_id", "serial_number", "queue_grain"], "Frozen-test action ranking for TVM/Gate or descriptive Validator OOS proxy; inspect source_scope."),
        ("dashboard_incident_backtest", "incident_backtest", ["device_category", "incident_id"], "Curated frozen temporal-test incident drill-down only; never a live scoring feed."),
    ]
    for suffix, path_key, natural_keys, scope in source_specs:
        selector = _rds_curate_action_queue if path_key == "action_queue" else _rds_curate_incident_backtest if path_key == "incident_backtest" else None
        frame = _rds_collect_category_artifacts(root, paths[path_key], category_scope, selector=selector)
        table_specs.append(
            _rds_prepare_table(
                frame,
                table_name=_rds_safe_table_name(suffix),
                natural_keys=natural_keys,
                run_context=context,
                default_scope=scope,
                optional=True,
            )
        )

    model_registry_path = root / "model_registry_rows.csv"
    model_registry = _rds_read_csv(model_registry_path) if model_registry_path.exists() else pd.DataFrame()
    if not model_registry.empty:
        model_registry["_rds_source_artifact"] = str(model_registry_path.relative_to(root))
        model_registry["_rds_source_scope"] = "Validation-selected PS3 model registry with frozen temporal-test diagnostics and promotion gates."
    table_specs.append(
        _rds_prepare_table(
            model_registry,
            table_name=_rds_safe_table_name("dashboard_model_registry"),
            natural_keys=["device_category", "head"],
            run_context=context,
            default_scope="Validation-selected PS3 model registry with frozen temporal-test diagnostics and promotion gates.",
            optional=True,
        )
    )

    readiness_rows = []
    for spec in table_specs:
        output_path = table_dir / f"{spec['table_name']}.csv"
        write_csv(spec["frame"], output_path)
        readiness = _rds_quality_row(spec)
        readiness["csv_path"] = str(output_path.relative_to(bundle))
        readiness["csv_sha256"] = _rds_file_sha256(output_path)
        readiness_rows.append(readiness)

    # Input quality and output readiness are themselves dashboard metrics. Their
    # source is explicit so dashboard consumers can distinguish source health from
    # model quality.
    readiness_frame = pd.DataFrame(readiness_rows)
    input_assessment = run_summary.get("data_quality", {}).get("data_quality_assessment", {})
    dq_frame = readiness_frame.copy()
    dq_frame["input_quality_score"] = input_assessment.get("quality_score")
    dq_frame["input_quality_gate_pass"] = input_assessment.get("quality_gate_pass")
    dq_frame["input_quality_threshold"] = input_assessment.get("quality_gate_threshold", CONFIG.get("RDS_REQUIRED_QUALITY_SCORE", 97.0))
    dq_frame["dq_scope"] = "RDS table-level load-readiness plus prepared incident input quality gate."
    dq_frame["_rds_source_artifact"] = "rds_dashboard_bundle/rds_load_readiness.csv"
    dq_frame["_rds_source_scope"] = "Operational data-quality metadata; do not interpret as model performance."
    dq_spec = _rds_prepare_table(
        dq_frame,
        table_name=_rds_safe_table_name("dashboard_data_quality"),
        natural_keys=["rds_table"],
        run_context=context,
        default_scope="Operational data-quality metadata; do not interpret as model performance.",
        optional=False,
    )
    dq_path = table_dir / f"{dq_spec['table_name']}.csv"
    write_csv(dq_spec["frame"], dq_path)
    dq_readiness = _rds_quality_row(dq_spec)
    dq_readiness["csv_path"] = str(dq_path.relative_to(bundle))
    dq_readiness["csv_sha256"] = _rds_file_sha256(dq_path)
    table_specs.append(dq_spec)
    readiness_rows.append(dq_readiness)
    readiness_frame = pd.DataFrame(readiness_rows)
    write_csv(readiness_frame, bundle / "rds_load_readiness.csv")

    dictionary = _rds_data_dictionary(table_specs)
    write_csv(dictionary, bundle / "rds_data_dictionary.csv")
    ddl = _rds_build_ddl(schema, table_specs)
    (bundle / "ps3_dashboard_postgres_ddl.sql").write_text(ddl, encoding="utf-8")
    upsert_sql = _rds_build_upsert_template(schema, table_specs)
    (bundle / "ps3_dashboard_postgres_upsert_template.sql").write_text(upsert_sql, encoding="utf-8")

    input_gate = bool(input_assessment.get("quality_gate_pass", True))
    block_on_input_quality = bool(CONFIG.get("RDS_BLOCK_ON_INPUT_QUALITY", True))
    load_allowed = bool((input_gate or not block_on_input_quality))
    manifest_tables = []
    for readiness, spec in zip(readiness_rows, table_specs):
        manifest_tables.append(
            {
                "rds_table": spec["table_name"],
                "csv_path": readiness["csv_path"],
                "csv_sha256": readiness["csv_sha256"],
                "row_count": readiness["row_count"],
                "natural_key": spec["natural_keys"],
                "load_mode": "delete_same_run_then_upsert",
                "optional": spec["optional"],
                "ready_for_rds": readiness["ready_for_rds"],
                "load_status": readiness["load_status"],
                "analytical_scope": spec["scope"],
            }
        )
    manifest = {
        "contract_version": RDS_TABLE_VERSION,
        "run_id": context["run_id"],
        "city_id": context["city_id"],
        "generated_at_utc": context["generated_at_utc"],
        "rds_schema": schema,
        "input_quality_gate_pass": input_gate,
        "input_quality_score": input_assessment.get("quality_score"),
        "load_allowed": load_allowed,
        "load_block_reason": None if load_allowed else "Prepared incident quality is below the configured RDS threshold.",
        "tables": manifest_tables,
        "guardrails": [
            "TVM/Gate action and incident views are frozen temporal-test backtests, not live scoring outputs.",
            "Validator rows are descriptive device_failures OOS proxy outputs unless true Gold PS3 labels are present.",
            "Component transitions and inter-event metrics are descriptive observed incident sequences, not causal effects, hazards, or fleet failure rates.",
            "The optional RDS writer is disabled by default and requires an explicit connection string plus a successful readiness check.",
        ],
    }
    write_json(manifest, bundle / "rds_load_manifest.json")

    readme = f"""# PS3 RDS dashboard bundle\n\nRun ID: `{context['run_id']}`  \nSchema: `{schema}`  \n\n1. Review `rds_load_readiness.csv`. Load only rows with `ready_for_rds=true`.\n2. Apply `ps3_dashboard_postgres_ddl.sql` to the target PostgreSQL RDS database. The DDL is additive for evolving metric columns.\n3. Either use your governed ETL/COPY process with `rds_load_manifest.json`, or set `RDS_WRITE_ENABLED=True`, set `{CONFIG.get('RDS_CONNECTION_ENV', 'PS3_RDS_DATABASE_URL')}`, and rerun the notebook. The optional loader deletes only the matching `run_id` before upserting it.\n4. Use `rds_data_dictionary.csv` to label dashboard metrics and preserve the scope cautions.\n\nImportant: TVM/Gate action queues are frozen temporal-test backtests. Validator queues are OOS-proxy intelligence unless true Validator Gold labels exist. Component transition/inter-event outputs are observed history, not causal or survival-model claims.\n"""
    (bundle / "README.md").write_text(readme, encoding="utf-8")
    summary = {
        "status": "available",
        "bundle_path": str(bundle),
        "run_id": context["run_id"],
        "rds_schema": schema,
        "tables": int(len(table_specs)),
        "ready_tables": int(sum(bool(row["ready_for_rds"]) for row in readiness_rows)),
        "empty_optional_tables": int(sum(row["load_status"] == "empty_optional" for row in readiness_rows)),
        "input_quality_gate_pass": input_gate,
        "load_allowed": load_allowed,
        "rds_write_enabled": bool(CONFIG.get("RDS_WRITE_ENABLED", False)),
    }
    write_json(summary, bundle / "rds_dashboard_bundle_summary.json")

    output_manifest_path = root / "output_manifest.json"
    if output_manifest_path.exists():
        try:
            output_manifest = json.loads(output_manifest_path.read_text())
        except Exception:
            output_manifest = {}
        output_manifest["rds_dashboard_bundle"] = {
            "directory": "rds_dashboard_bundle",
            "manifest": "rds_dashboard_bundle/rds_load_manifest.json",
            "readiness": "rds_dashboard_bundle/rds_load_readiness.csv",
            "ddl": "rds_dashboard_bundle/ps3_dashboard_postgres_ddl.sql",
            "data_dictionary": "rds_dashboard_bundle/rds_data_dictionary.csv",
        }
        write_json(output_manifest, output_manifest_path)
    return summary


def _rds_execute_sql_script(connection, sql_text):
    """Execute generated semicolon-delimited DDL; all identifiers are notebook-normalized."""
    for statement in sql_text.split(";"):
        statement = statement.strip()
        if statement:
            connection.exec_driver_sql(statement)


def load_dashboard_bundle_to_rds(bundle_dir, connection_url=None):
    """Optionally apply the generated RDS bundle with transactional same-run upserts.

    This function is intentionally never called unless `RDS_WRITE_ENABLED=True`.
    It expects PostgreSQL-compatible RDS and an SQLAlchemy-supported driver.
    """
    if not CONFIG.get("RDS_WRITE_ENABLED", False):
        return {
            "status": "disabled",
            "reason": "RDS_WRITE_ENABLED is False. The CSV/DDL bundle is available for a governed external load.",
        }
    bundle = Path(bundle_dir)
    manifest_path = bundle / "rds_load_manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"RDS bundle manifest was not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text())
    if not manifest.get("load_allowed", False):
        raise RuntimeError(f"RDS load blocked by readiness policy: {manifest.get('load_block_reason')}")
    url = connection_url or os.getenv(CONFIG.get("RDS_CONNECTION_ENV", "PS3_RDS_DATABASE_URL"))
    if not url:
        raise RuntimeError(
            f"Set {CONFIG.get('RDS_CONNECTION_ENV', 'PS3_RDS_DATABASE_URL')} or pass connection_url after approval to load RDS."
        )
    try:
        from sqlalchemy import create_engine
    except ImportError as exc:
        raise ImportError("Install sqlalchemy plus the PostgreSQL driver in the SageMaker image before enabling RDS writes.") from exc

    schema = _rds_identifier(manifest["rds_schema"], fallback="mars_dashboard")
    engine = create_engine(url, pool_pre_ping=True)
    ddl = (bundle / "ps3_dashboard_postgres_ddl.sql").read_text(encoding="utf-8")
    loaded, skipped = [], []
    try:
        with engine.begin() as connection:
            _rds_execute_sql_script(connection, ddl)
            for table in manifest["tables"]:
                if not table.get("ready_for_rds", False):
                    skipped.append({"rds_table": table["rds_table"], "reason": table.get("load_status")})
                    continue
                path = bundle / table["csv_path"]
                frame = pd.read_csv(path)
                if frame.empty:
                    skipped.append({"rds_table": table["rds_table"], "reason": "empty_after_read"})
                    continue
                table_name = _rds_identifier(table["rds_table"])
                stage_name = _rds_identifier(f"staging_{table_name}_{manifest['run_id'][-10:]}")
                keys = [_rds_identifier(key) for key in table["natural_key"]]
                columns = [_rds_identifier(column) for column in frame.columns]
                if list(frame.columns) != columns:
                    frame.columns = columns
                connection.exec_driver_sql(
                    f"DELETE FROM {_rds_quote_identifier(schema)}.{_rds_quote_identifier(table_name)} "
                    f"WHERE {_rds_quote_identifier('run_id')} = :run_id",
                    {"run_id": manifest["run_id"]},
                )
                frame.to_sql(
                    stage_name,
                    connection,
                    schema=schema,
                    if_exists="replace",
                    index=False,
                    method="multi",
                    chunksize=int(CONFIG.get("RDS_WRITE_CHUNK_SIZE", 5000)),
                )
                q_columns = ", ".join(_rds_quote_identifier(column) for column in columns)
                conflict = ", ".join(_rds_quote_identifier(key) for key in keys)
                updates = [column for column in columns if column not in keys]
                if updates:
                    update_sql = ", ".join(
                        f"{_rds_quote_identifier(column)} = EXCLUDED.{_rds_quote_identifier(column)}" for column in updates
                    )
                    conflict_sql = f"DO UPDATE SET {update_sql}"
                else:
                    conflict_sql = "DO NOTHING"
                connection.exec_driver_sql(
                    f"INSERT INTO {_rds_quote_identifier(schema)}.{_rds_quote_identifier(table_name)} ({q_columns}) "
                    f"SELECT {q_columns} FROM {_rds_quote_identifier(schema)}.{_rds_quote_identifier(stage_name)} "
                    f"ON CONFLICT ({conflict}) {conflict_sql}"
                )
                connection.exec_driver_sql(f"DROP TABLE IF EXISTS {_rds_quote_identifier(schema)}.{_rds_quote_identifier(stage_name)}")
                loaded.append({"rds_table": table_name, "rows": int(len(frame))})
    finally:
        engine.dispose()
    result = {"status": "loaded", "run_id": manifest["run_id"], "loaded_tables": loaded, "skipped_tables": skipped}
    write_json(result, bundle / "rds_load_execution_summary.json")
    return result


In [ ]:
def category_minimum_rows(category):
    value = CONFIG["MIN_ROWS_TO_MODEL"]
    return int(value.get(category, max(value.values())) if isinstance(value, dict) else value)


def run_category(all_incidents, category, output_root):
    category_out = Path(output_root) / {"TVM": "tvm", "GATE": "gates", "VALIDATOR": "validators"}.get(category, safe_name(category).lower())
    category_out.mkdir(parents=True, exist_ok=True)
    raw = all_incidents[all_incidents["mars_device_category"] == category].copy()
    log(f"{'=' * 18} {category}: {len(raw):,} Gold incidents {'=' * 18}")
    if raw.empty:
        if category == "VALIDATOR":
            proxy = validator_oos_proxy(category_out)
            return {"category": category, "mode": "validator_proxy", "n_incidents": 0, "proxy": proxy}
        status = {"category": category, "mode": "no_incidents", "n_incidents": 0}
        write_json(status, category_out / "category_status.json")
        return status

    data = add_temporal_features(raw)
    data = add_causal_recurrence_features(data)
    data = enrich_all(data)
    write_csv(pd.DataFrame(ENRICHMENT_AUDIT), category_out / "enrichment_audit.csv")
    data, split_meta = make_temporal_splits(data)
    write_json(split_meta, category_out / "temporal_split.json")
    discovery = category_discovery_pack(data, category, category_out)
    component_intelligence = write_component_reliability_intelligence(data, category, category_out)
    component_sequence_intelligence = safe_component_sequence_intelligence(
        data,
        category,
        category_out,
        date_col="AE_START_DTM",
        component_col=CONFIG["ROOTCAUSE_TARGET"],
        severity_col=CONFIG["SEVERITY_TARGET"],
        device_col="device_id",
        serial_col="serial_number",
        scope="historical all-incident component sequence discovery; descriptive only, not causal or a future failure probability",
    )
    if len(data) < category_minimum_rows(category):
        status = {
            "category": category,
            "mode": "report_only_small_sample",
            "n_incidents": int(len(data)),
            "minimum_required": category_minimum_rows(category),
            "discovery": discovery,
            "component_intelligence": component_intelligence,
        "component_sequence_intelligence": component_sequence_intelligence,
            "component_sequence_intelligence": component_sequence_intelligence,
        }
        write_json(status, category_out / "category_status.json")
        return status

    severity = train_head(data, category, "severity", CONFIG["SEVERITY_TARGET"], category_out)
    root_cause = train_head(data, category, "root_cause", CONFIG["ROOTCAUSE_TARGET"], category_out)
    feeds = build_actionable_prediction_feeds(data, category, category_out, severity, root_cause)
    result = {
        "category": category,
        "mode": "modelled" if severity or root_cause else "report_only_no_viable_head",
        "n_incidents": int(len(data)),
        "n_devices": int(data["device_id"].nunique()),
        "split": split_meta,
        "discovery": discovery,
        "component_intelligence": component_intelligence,
        "component_sequence_intelligence": component_sequence_intelligence,
        "severity": severity["summary"] if severity else None,
        "root_cause": root_cause["summary"] if root_cause else None,
        "action_feed_rows": int(len(feeds["incidents"])) if feeds else 0,
        "component_prediction_reliability": feeds.get("component_prediction_reliability") if feeds else None,
    }
    write_json(result, category_out / "category_summary.json")
    return result


def finalise_run(results, quality_report, probe, output_root, cross_device_operational=None):
    root = Path(output_root)
    registry = []
    for category in results:
        for head_name in ("severity", "root_cause"):
            head = category.get(head_name) if isinstance(category, dict) else None
            if head:
                registry.append({
                    "ps_id": "PS3",
                    "device_category": category["category"],
                    "head": head_name,
                    "champion": head["champion_selected_on_validation"],
                    "validation_f1_macro": head["validation_metrics_champion"]["f1_macro"],
                    "validation_accuracy": head["validation_metrics_champion"]["accuracy"],
                    "test_f1_macro": head["test_metrics_champion"]["f1_macro"],
                    "test_accuracy": head["test_metrics_champion"]["accuracy"],
                    "test_balanced_accuracy": head["test_metrics_champion"]["balanced_accuracy"],
                    "test_roc_auc_macro_ovr": head["test_metrics_champion"]["roc_auc_macro_ovr"],
                    "test_pr_auc_macro_ovr": head["test_metrics_champion"]["pr_auc_macro_ovr"],
                    "test_ece": head["test_metrics_champion"]["ece"],
                    "gate_pass": head["promotion_gate"]["gate_pass"],
                    "gate_reasons": " | ".join(head["promotion_gate"]["reasons"]),
                })
    if registry:
        write_csv(pd.DataFrame(registry), root / "model_registry_rows.csv")
    run_summary = {
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "configuration": CONFIG,
        "data_quality": quality_report,
        "servicenow_conformance_probe": probe,
        "results": results,
        "cross_device_operational": cross_device_operational,
        "interpretation": {
            "model_selection": "Validation macro-F1 only; the temporal test was frozen until after selection.",
            "validator": "The Validator proxy is descriptive OOS intelligence, not a supervised PS3 classifier when Gold has no Validator incidents.",
            "root_cause": "Derived component classification remains distinct from a future true ServiceNow root-cause model.",
        },
    }
    write_json(run_summary, root / "ps3_run_summary.json")
    manifest = {
        "out_of_sample_feeds": [
            "{category}/test_incident_predictions.csv",
            "{category}/test_device_action_ranking.csv",
            "{category}/test_device_serial_action_ranking.csv",
            "{category}/device_serial_historical_profile.csv",
            "cross_device/cross_device_device_serial_action_ranking.csv",
            "{category}/test_facility_action_ranking.csv",
            "{category}/component_intelligence/component_reliability_portfolio.csv",
            "{category}/component_intelligence/device_component_reliability_ranking.csv",
            "{category}/component_intelligence/device_serial_component_reliability_ranking.csv",
            "{category}/component_intelligence/component_prediction_reliability.csv",
            "{category}/component_intelligence/component_preincident_signal_lift.csv",
            "cross_device/cross_device_component_reliability_portfolio.csv",
            "cross_device/cross_device_device_component_reliability_ranking.csv",
        ],
        "serving_bundles": ["{category}/{head}/serving_bundle.joblib"],
        "important_note": "Test feeds are historical, frozen temporal-holdout backtests. Use serving bundles only to score future feature rows.",
    }
    write_json(manifest, root / "output_manifest.json")
    return run_summary


def run_ps3(synthetic=False):
    output_root = make_output_dir()
    log(f"PS3 SageMaker run starting. Outputs: {output_root}")
    if synthetic:
        incidents = make_synthetic_incidents()
        log("Running synthetic smoke test; no S3 data are being used.")
    else:
        incidents = load_table(CONFIG["GOLD_TABLE"], CONFIG["GOLD_PARQUET"], required=True)
    prepared, quality = validate_and_prepare_incidents(incidents)
    write_json(quality, output_root / "data_quality_report.json")
    write_cross_device_overview(prepared, output_root)
    probe = probe_servicenow_conformance(output_root)
    results = []
    for category in CONFIG["DEVICE_SCOPE"]:
        try:
            results.append(run_category(prepared, category, output_root))
        except Exception as exc:
            import traceback
            traceback.print_exc()
            failure = {"category": category, "mode": "failed", "error": f"{type(exc).__name__}: {str(exc)[:600]}"}
            write_json(failure, output_root / safe_name(category).lower() / "category_failure.json")
            results.append(failure)
    cross_device_operational = write_cross_device_operational_portfolio(output_root)
    summary = finalise_run(results, quality, probe, output_root, cross_device_operational)
    if CONFIG.get("RDS_DASHBOARD_BUNDLE_ENABLED", True):
        try:
            rds_bundle = write_rds_dashboard_bundle(output_root, summary)
            summary["rds_dashboard_bundle"] = rds_bundle
            if CONFIG.get("RDS_WRITE_ENABLED", False):
                summary["rds_load"] = load_dashboard_bundle_to_rds(Path(output_root) / "rds_dashboard_bundle")
            write_json(summary, Path(output_root) / "ps3_run_summary.json")
        except Exception as exc:
            rds_failure = {
                "status": "failed",
                "error": f"{type(exc).__name__}: {str(exc)[:600]}",
                "note": "Core PS3 modeling outputs completed; inspect this failure before enabling any RDS load.",
            }
            summary["rds_dashboard_bundle"] = rds_failure
            write_json(rds_failure, Path(output_root) / "rds_dashboard_bundle_failure.json")
            write_json(summary, Path(output_root) / "ps3_run_summary.json")
            log(f"RDS dashboard bundle failed without invalidating core outputs: {rds_failure['error']}")
    log("PS3 run complete. Start with ps3_run_summary.json, rds_dashboard_bundle/, and model_registry_rows.csv.")
    return summary


## 13. Synthetic smoke-test generator — optional, safe, and deliberately imperfect


In [ ]:
def make_synthetic_incidents(n_tvm=800, n_gate=280, seed=7):
    """Creates a compact offline test set and all optional source tables. Never use its metrics as a business result."""
    global SYNTHETIC_SOURCES
    rng = np.random.default_rng(seed)
    component_types = ["CSC_READER", "PRINTER", "BHU", "CHU", "GATE_MECH", "COMMS"]
    severity_codes = {2: "PURCHASE_CARD", 3: "PURCHASE_PRODUCT", 4: "ALL_PURCHASE", 5: "ALL_FUNCTIONS", 16: "BUS_READER_ASSEMBLY"}

    def generate(category, rows, prefix, probabilities):
        component = rng.choice(component_types, size=rows, p=probabilities)
        base = {name: rng.poisson(0.7, rows).astype(float) for name in ["bhu", "chu", "printer", "gate_mech", "csc_reader", "comms"]}
        component_to_signal = {"CSC_READER": "csc_reader", "PRINTER": "printer", "BHU": "bhu", "CHU": "chu", "GATE_MECH": "gate_mech", "COMMS": "comms"}
        for index, comp in enumerate(component):
            base[component_to_signal[comp]][index] += rng.poisson(3.0)
        age = rng.integers(30, 3000, rows).astype(float)
        load = sum(base.values())
        score = 0.45 * (component == "GATE_MECH") + 0.35 * (component == "COMMS") + 0.00045 * age + 0.04 * load + rng.normal(0, 0.85, rows)
        level = np.where(score > 1.7, 16, np.where(score > 1.2, 5, np.where(score > 0.75, 4, np.where(score > 0.25, 3, 2))))
        timestamps = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 720, rows), unit="D") + pd.to_timedelta(rng.integers(0, 86400, rows), unit="s")
        device_ids = [f"{prefix}{rng.integers(1, max(3, rows // 6)):05d}" for _ in range(rows)]
        return pd.DataFrame({
            "availability_event_id": [f"AE-{category}-{idx:07d}" for idx in range(rows)],
            "device_id": device_ids,
            "mars_device_category": category,
            "AE_START_DTM": timestamps,
            "transit_day": timestamps.normalize(),
            "failure_level_label": [severity_codes[int(item)] for item in level],
            "derived_component_type": component,
            "events_24h_prior": load + rng.poisson(1, rows),
            "oos_onsets_24h": rng.poisson(0.6, rows),
            "bhu_events_24h": base["bhu"], "chu_events_24h": base["chu"], "printer_events_24h": base["printer"],
            "gate_mech_events_24h": base["gate_mech"], "csc_reader_events_24h": base["csc_reader"], "comms_events_24h": base["comms"],
            "oos_onsets_7d_prior": rng.poisson(2.0, rows), "events_7d_prior": load * 3 + rng.poisson(4, rows),
            "component_age_days": age,
            "matched_serial_nbr": [f"SN{prefix}{rng.integers(1, max(3, rows // 4)):06d}" for _ in range(rows)],
            "FACILITY_ID": [f"FAC-{rng.integers(1, 35):03d}" for _ in range(rows)],
            "FACILITY_NAME": ["Synthetic Station"] * rows,
            # Deliberately blocked features prove that the contract excludes them.
            "sn_event_code_id": level * 100 + rng.integers(0, 40, rows),
            "AE_FAULT_DESCRIPTION": [f"{value} fault" for value in component],
            "affected_component": component,
        })

    incidents = pd.concat([
        generate("TVM", n_tvm, "TVM", [0.50, 0.17, 0.12, 0.08, 0.00, 0.13]),
        generate("GATE", n_gate, "GATE", [0.12, 0.00, 0.00, 0.00, 0.62, 0.26]),
    ], ignore_index=True)
    devices = incidents[["device_id", "mars_device_category", "FACILITY_ID"]].drop_duplicates()
    days = pd.date_range(incidents["transit_day"].min() - pd.Timedelta(days=30), incidents["transit_day"].max(), freq="D")
    grid = devices.assign(_join=1).merge(pd.DataFrame({"transit_day": days, "_join": 1}), on="_join").drop(columns="_join")
    n = len(grid)
    base = grid.rename(columns={"device_id": "DEVICE_ID"})
    usage = base[["DEVICE_ID", "transit_day"]].copy()
    usage["days_since_last_failure"] = rng.integers(0, 90, n)
    usage["days_since_last_maintenance"] = rng.integers(0, 180, n)
    usage["failure_count_30d"] = rng.poisson(1.2, n)
    usage["cumulative_failure_count"] = rng.integers(0, 60, n)
    usage["cumulative_outage_min"] = rng.integers(0, 5000, n)
    usage["days_in_service"] = rng.integers(30, 3000, n)
    metric = base[["DEVICE_ID", "transit_day"]].copy()
    metric["m401_avg_txn_time_ms"] = rng.normal(1800, 300, n).clip(250, None)
    metric["m401_p95_txn_time_ms"] = metric["m401_avg_txn_time_ms"] * rng.uniform(1.2, 1.8, n)
    metric["m401_p99_txn_time_ms"] = metric["m401_p95_txn_time_ms"] * rng.uniform(1.05, 1.35, n)
    metric["m401_slow_tap_pct"] = rng.uniform(0, 0.12, n)
    metric["m401_rolling_7d_avg_ms"] = metric["m401_avg_txn_time_ms"] * rng.uniform(0.9, 1.1, n)
    metric["m401_z_score_vs_28d"] = rng.normal(0, 1, n)
    metric["volume_drop_flag"] = rng.integers(0, 2, n)
    metric["comms_total_count"] = rng.poisson(0.8, n)
    metric["comms_event_flag"] = (metric["comms_total_count"] > 2).astype(int)
    uptime = base[["DEVICE_ID", "transit_day"]].copy()
    uptime["uptime_pct"] = rng.uniform(0.85, 1.0, n)
    uptime["downtime_hours"] = (1 - uptime["uptime_pct"]) * 24
    uptime["hours_since_last_hb"] = rng.exponential(2.0, n)
    uptime["is_silent_device"] = (uptime["hours_since_last_hb"] > 12).astype(int)
    uptime["distinct_message_types"] = rng.integers(1, 15, n)
    uptime["total_msg_count"] = rng.poisson(200, n)
    facilities = pd.DataFrame({"FACILITY_ID": sorted(incidents["FACILITY_ID"].unique())}).assign(_join=1).merge(pd.DataFrame({"transit_day": days, "_join": 1}), on="_join").drop(columns="_join")
    station = []
    for category in ("TVM", "GATE"):
        piece = facilities.copy()
        piece["device_category"] = category
        piece["devices_failed"] = rng.poisson(0.6, len(piece))
        piece["total_failure_events"] = piece["devices_failed"] * rng.uniform(1.0, 2.5, len(piece))
        piece["avg_downtime_minutes"] = rng.uniform(5, 100, len(piece))
        piece["is_coordinated_failure"] = (piece["devices_failed"] >= 3).astype(int)
        piece["is_major_station_event"] = (piece["devices_failed"] >= 5).astype(int)
        station.append(piece)
    validator_proxy = pd.DataFrame({
        "DEVICE_ID": [f"VAL{rng.integers(1, 150):05d}" for _ in range(1200)],
        "mars_device_category": "VALIDATOR",
        "ledger_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 720, 1200), unit="D"),
        "failure_level": rng.choice([2, 4, 5, 16], 1200, p=[0.50, 0.30, 0.12, 0.08]),
    })
    SYNTHETIC_SOURCES = {
        "usage_lifecycle_daily": usage,
        "metric_daily": metric,
        "device_uptime_intervals": uptime,
        "station_network_daily": pd.concat(station, ignore_index=True),
        "device_failures": validator_proxy,
        "incident_root_cause": pd.DataFrame(),
    }
    return incidents


## 14. Run the pipeline


In [ ]:
results = run_ps3(synthetic=CONFIG["SMOKE_TEST_SYNTHETIC"])
results


## 15. How to read the results

All outputs are written beneath `PS3_outputs_pavan_enhanced/`:

- `tvm/`, `gates/`, and `validators/` hold device-type-specific results; `cross_device/` holds comparable portfolio views.
- `component_intelligence/component_reliability_portfolio.csv` is the dashboard-ready component portfolio. It includes incident burden, affected devices/serials/facilities, conservative criticality, recurrence, concentration, trend signals, evidence tier, and the transparent triage score.
- `component_intelligence/device_component_reliability_ranking.csv` and `component_intelligence/device_serial_component_reliability_ranking.csv` are the all-history, individual `device_id` and `serial_number` views. They are descriptive historical discovery tables, not future scores.
- `component_intelligence/component_prediction_reliability.csv` and `test_device_serial_component_prediction_quality.csv` break frozen-test root-cause and severity quality down by actual component, including Wilson confidence bounds. They distinguish weak evidence from poor performance.
- `component_intelligence/component_preincident_signal_lift.csv` contains MAD-standardised contrasts for only approved pre-incident features. It identifies useful signatures for investigation; it does not claim causality.
- `component_intelligence/component_temporal_mix_shift.json` reports a categorical PSI-style train/test component-mix monitor. A material shift means model transferability needs review.
- PNGs in `component_intelligence/` are dashboard-ready: priority ranking, recurrence × conservative criticality matrix, device × component heatmap, monthly component heatmap, root-cause quality chart, and a compact reliability dashboard.
- `cross_device/cross_device_component_reliability_portfolio.csv`, `cross_device_component_incident_matrix.csv`, and `cross_device_device_component_reliability_ranking.csv` compare TVs, Gates, and any Validator component data using the same explicit fields.

### Method and guardrails

The historical triage score combines incident volume, a 95% Wilson lower bound of observed critical share, recurrence within the configured window, recent-window activity, and recency. Empirical-Bayes smoothing and evidence weighting prevent tiny samples from dominating the ranking. It is deliberately **not** a predicted future failure probability; use the frozen-test action feeds and serving bundles only as documented.

The Validator path remains explicitly separate: it is a descriptive `device_failures` OOS proxy unless Gold PS3 labels are available. If its source lacks a component/root-cause field, the notebook writes a component-label-coverage status instead of inventing component findings.

`UNCLASSIFIED_NO_ROOT_CAUSE_PREDICTION`, `UNCLASSIFIED_COMPONENT`, and `UNKNOWN_SERIAL` are data-quality states. They stay in the files so coverage gaps can be monitored rather than silently omitted.


### RDS dashboard bundle

`rds_dashboard_bundle/` is created at the end of every successful run and is safe
to hand to a governed RDS ingestion process:

- `tables/` contains canonical, run-versioned CSV snapshots for component portfolios,
  device × component and device × serial × component detail, action queues,
  frozen-test incident drill-downs, monthly component trends, component transitions,
  inter-event reliability, model registry, and data quality.
- `rds_load_manifest.json` names every table, its natural key, record count,
  SHA-256 file hash, optionality, and its approved `delete_same_run_then_upsert`
  load mode.
- `rds_load_readiness.csv` uses the declared 40/30/20/10 completeness/validity/
  consistency/uniqueness weights and blocks live loading when the prepared input
  quality gate is below the configured 97 score.
- `ps3_dashboard_postgres_ddl.sql`, `ps3_dashboard_postgres_upsert_template.sql`,
  and `rds_data_dictionary.csv` provide the RDS contract. The optional loader is
  disabled unless `RDS_WRITE_ENABLED=True` and the configured connection-string
  environment variable is present.

`component_transition_portfolio.csv` and
`component_interevent_reliability.csv` add sequence insight at component level;
`device_component_transition_followup.csv` identifies device-level paths worth
review and reports serial-continuity context. These are explicitly observed
incident-history measures. They are not
causal root-cause chains, failure rates, survival estimates, or predictions of
what will happen next.
